# Stream live events
Part of the retail-demo setup utility. A Spark Structured Streaming generator
that continuously emits synthetic retail events as JSON `EventEnvelope`s and
writes each one **directly to the Fabric Eventhouse** with the Spark Kusto
connector, routed by `event_type` to its KQL event table (`receipt_created`,
`inventory_updated`, …) → Silver → Gold. Everything stays inside Fabric: no
Eventstream, no Event Hubs.

The live stream continues the history the setup notebooks wrote, with the same
models (embedded below):

- **Volume and hours.** The default rate comes from the store count and the
  historical `transactions_per_store_day`. Each store's visits follow the
  profile's hourly, daily and monthly weights in the store's **local** time,
  its traffic multiplier and its operating hours (no sales while closed).
  Stores open and close at their local operating-hour boundaries.
- **Customers.** As in history, about 45% of shoppers are anonymous; the rest
  are mostly the store's own home customers, weighted by visit frequency and
  kept only while signed up and not churned; about 22% of visits buy. Visits follow an ordered
  path through the store type's zones (entrance → departments → checkout →
  exit) with realistic dwell times, and their BLE IDs resolve to
  `dim_customers.BLEId`.
- **Purchases and marketing.** Baskets (with complementary products and
  repeated SKUs merged), shelf prices of the catalog date, one basket-level
  promotion, the tender mix, declines and campaigns use the historical
  catalogs; volume follows history's year-over-year trend. Marketing journey
  touches carry their touch time in `impression_time` and arrive with the
  purchase, so notebook 03 never skips them.
- **Store inventory.** Each store keeps a running balance per product,
  seeded from the Silver ledger. Receipt lines and store-fulfilled online
  lines take stock; a line the store cannot fill is a lost sale and is left
  off the receipt. Stockouts and reorders come from the balance and history's
  replenishment policy, with the store's DC on every event.
- **Trucks.** Reorders wait at the store's DC for a consolidated load, then
  ride the next free truck of the DC's fleet: planned and actual arrival,
  a load-size driven unload, and a departure once the unload is done. The
  unload restocks the store.
- **Online orders.** Created with one event per line, then paid, picked,
  shipped and delivered with realistic delays (BOPIS: ready, then collected),
  with history's fulfillment mix, cancellations, backorders and declines.

State that spans micro-batches (store open/closed, visits and online orders
in progress, store inventory, shipments and trucks) lives in Lakehouse Delta
tables `<silver>._stream_state_*`, versioned by micro-batch, so a replayed
batch regenerates exactly the same events.

The connector write pattern follows the Real-Time Intelligence tutorial
"Use a notebook with Apache Spark to query a KQL database"
(https://learn.microsoft.com/fabric/real-time-intelligence/spark-connector):
`df.write.format("com.microsoft.kusto.spark.synapse.datasource")`, inside
`foreachBatch`, one write per event table.

Run this AFTER the setup notebooks (the KQL tables must already exist). It is the
optional **live driver**, not part of the ordered batch setup. Stop the streaming
query to stop generating.

Latency and cadence: a live run triggers a micro-batch every `trigger_seconds`
(60 by default) and writes each event table once per batch, as one coalesced
partition. Each write sets `flushImmediately`, so events are queryable seconds
after the batch instead of waiting for queued-ingestion batching. The per-table
writes run concurrently so a micro-batch keeps up with the trigger.
If a table looks empty or rows seem dropped, run `.show ingestion failures` in the
KQL database — rows that fail schema/type validation are reported there, not lost
silently.

In [ ]:
# Fabric parameters — override per run via the pipeline/parameterization.
source_rows_per_second = 0     # 0 = derive from the store count and history volume;
                               # a larger value only adds resolution, not volume
volume_scale = 1.0             # multiply every calibrated live volume (1.0 = history)
sink = "eventhouse"            # "eventhouse" | "delta"
run_seconds = 0                # 0 = run forever; >0 = stop after N seconds (test/smoke)
trigger_seconds = 60           # live micro-batch interval; bounded runs use at most 10
max_lag_seconds = 600          # after a stop, skip source rows older than this
max_restarts = 5               # supervised restarts of a failed live query
allow_history_gap = False      # start a new stream even if history ends >1 day ago or overlaps now
event_source = "retail-datagen"  # envelope `source`; kept compatible with downstream
stream_id = ""                 # blank = persist a UUID beside the checkpoint

# Eventhouse (Kusto) sink — the default. Writes each event straight to its KQL
# event table with the Fabric Spark connector. Leave `kusto_uri` blank and it is
# resolved automatically at runtime from the `kql_database` below (its Query URI
# in this workspace); set it explicitly only to target a different cluster. The
# operator identity running the notebook needs ingestor/admin rights on the
# database. The tables must already exist (created by the KQL setup), so no table
# is auto-created here.
kusto_uri = ""                 # blank = auto-resolve from kql_database; or a Query URI like "https://<host>.kusto.fabric.microsoft.com"
kql_database = "retail_eventhouse"  # KQL database name (the Eventhouse's database)

# Delta sink (used when sink == "delta") — a local landing table for debugging the
# generator without a KQL database. Not part of the live demo path.
delta_landing_table = ""       # default derived from LAKEHOUSE_NAME below if blank

checkpoint_path = "Files/setup/stream/checkpoint"

In [ ]:
"""Pydantic models for dictionary JSON files.

Entry models mirror the field names in
datagen/src/retail_datagen/shared/models.py (PascalCase) so the supercenter
conversion is verifiable 1:1. StoreTypeProfile is new: the behavioral knobs
that differentiate store types.
"""

import re
from decimal import Decimal
from typing import Literal

from pydantic import BaseModel, Field, field_validator, model_validator


class TaxJurisdictionEntry(BaseModel):
    StateCode: str = Field(..., min_length=2, max_length=2)
    County: str = Field(..., min_length=1)
    City: str = Field(..., min_length=1)
    CombinedRate: Decimal = Field(..., ge=0, le=Decimal("0.20"))

    @field_validator("StateCode")
    @classmethod
    def _state_upper(cls, v: str) -> str:
        if not v.isalpha():
            raise ValueError("StateCode must be alphabetic")
        return v.upper()

    @field_validator("CombinedRate", mode="before")
    @classmethod
    def _rate_decimal(cls, v) -> Decimal:
        try:
            return Decimal(str(v))
        except Exception:
            raise ValueError("CombinedRate must be a valid decimal number") from None


class GeographyEntry(BaseModel):
    City: str = Field(..., min_length=1)
    State: str = Field(..., min_length=2, max_length=2)
    Zip: str
    District: str = Field(..., min_length=1)
    Region: str = Field(..., min_length=1)

    @field_validator("State")
    @classmethod
    def _state_upper(cls, v: str) -> str:
        if not v.isalpha():
            raise ValueError("State must be alphabetic")
        return v.upper()

    @field_validator("Zip")
    @classmethod
    def _zip_format(cls, v: str) -> str:
        if not re.match(r"^\d{5}(-\d{4})?$", v):
            raise ValueError("Zip must be 12345 or 12345-6789")
        return v


class NameEntry(BaseModel):
    """One first or last name (shared across store types).

    The ``Name`` field intentionally unifies the separate ``FirstNameDict`` and
    ``LastNameDict`` models from datagen — both source models carry a single
    name string, so one model suffices here.
    """

    Name: str = Field(..., min_length=1)


class ProductBrandEntry(BaseModel):
    Brand: str = Field(..., min_length=1)
    Company: str = Field(..., min_length=1)
    Category: str = Field(..., min_length=1)


class ProductEntry(BaseModel):
    ProductName: str = Field(..., min_length=1)
    BasePrice: Decimal = Field(..., gt=0)
    Department: str = Field(..., min_length=1)
    Category: str = Field(..., min_length=1)
    Subcategory: str = Field(..., min_length=1)
    Tags: str | None = None
    # Optional explicit attributes. Preset catalogs omit them and rely on the
    # department/category name rules in generation.dims; custom catalogs set them
    # because their department names are not known in advance.
    Taxability: Literal["TAXABLE", "NON_TAXABLE", "REDUCED_RATE"] | None = None
    Refrigerated: bool | None = None

    @field_validator("BasePrice", mode="before")
    @classmethod
    def _price_decimal(cls, v) -> Decimal:
        try:
            return Decimal(str(v))
        except Exception:
            raise ValueError("BasePrice must be a valid decimal number") from None


class ProductTagEntry(BaseModel):
    ProductName: str = Field(..., min_length=1)
    Tags: str = Field(..., min_length=1)


# Store formats, largest first. Volume classes pick among the formats their size
# suits (generation.dims.VOLUME_FORMATS); a profile may narrow the set.
STORE_FORMATS = ["hypermarket", "superstore", "standard", "neighborhood", "express", "boutique"]
_HOURS_PATTERN = re.compile(r"^(\d{1,2})-(\d{1,2})$")
# Derived defaults for profiles that leave the store-format knobs unset (custom
# layouts): traffic falls with ticket size around the supercenter reference.
_REFERENCE_TICKET = 85.0
_REFERENCE_TRANSACTIONS = 400
_TICKET_TRAFFIC_EXPONENT = 0.6
_MIN_DERIVED_TRANSACTIONS = 20
_MAX_DERIVED_TRANSACTIONS = 1000
# A custom profile's hours are the hours whose traffic weight reaches this
# share of its peak hour; a profile busy around the clock runs "24h".
_OPEN_HOUR_PEAK_SHARE = 0.10
# Above this average ticket a derived profile runs boutique stores.
_BOUTIQUE_TICKET = 500.0
# Derived trip frequency of an active customer around the supercenter reference.
_REFERENCE_VISITS = 8.0
_TICKET_VISITS_EXPONENT = 0.4
_MIN_DERIVED_VISITS = 1.0
_MAX_DERIVED_VISITS = 52.0


def parse_operating_hours(value: str) -> tuple[int, int]:
    """``[open_hour, close_hour)`` of an ``operating_hours`` value ("24h" or "H-H")."""
    text = value.strip()
    if text == "24h":
        return 0, 24
    match = _HOURS_PATTERN.fullmatch(text)
    if match is None:
        raise ValueError(f"operating hours {value!r} must be '24h' or 'open-close' hours")
    open_hour, close_hour = int(match.group(1)), int(match.group(2))
    if not 0 <= open_hour < close_hour <= 24:
        raise ValueError(f"operating hours {value!r} must open before closing, within 0-24")
    return open_hour, close_hour


class StoreTypeProfile(BaseModel):
    """Behavioral knobs that make a store type act differently.

    Weight lists are relative (normalized at use time, not here). The store
    knobs (transactions, hours, formats) are optional: presets set them, and
    custom layouts derive them from the ticket size and hourly weights.
    """

    store_type: str = Field(..., min_length=1)
    display_name: str = Field(..., min_length=1)
    basket_lambda: float = Field(..., gt=0, description="Nominal mean basket-line ceiling")
    avg_ticket_target: float = Field(..., gt=0, description="Basket calibration target, USD")
    hourly_weights: list[float] = Field(..., description="24 relative traffic weights")
    daily_weights: list[float] = Field(..., description="7 weights, Monday first")
    monthly_weights: list[float] = Field(..., description="12 seasonality weights, Jan first")
    department_weights: dict[str, float] = Field(..., min_length=1)
    promo_rate: float = Field(..., ge=0, le=1, description="Share of lines with a promotion")
    online_order_share: float = Field(..., ge=0, le=1)
    zones: list[str] = Field(..., min_length=1, description="Store footprint zones for BLE")
    transactions_per_store_day: int | None = Field(
        default=None, gt=0, le=5000, description="In-store receipts per store-day at multiplier 1"
    )
    operating_hours: list[str] | None = Field(
        default=None, min_length=1, description="Opening-hour patterns stores pick from"
    )
    store_formats: list[str] | None = Field(
        default=None, min_length=1, description="Store formats this store type runs"
    )
    customer_visits_per_year: float | None = Field(
        default=None, gt=0, le=156, description="Mean yearly trips of an active loyalty customer"
    )

    @field_validator("operating_hours")
    @classmethod
    def _hours_parse(cls, v: list[str] | None) -> list[str] | None:
        if v is None:
            return v
        for value in v:
            parse_operating_hours(value)
        return [value.strip() for value in v]

    @field_validator("store_formats")
    @classmethod
    def _known_formats(cls, v: list[str] | None) -> list[str] | None:
        if v is None:
            return v
        unknown = sorted(set(v) - set(STORE_FORMATS))
        if unknown:
            raise ValueError(f"unknown store formats {unknown}; known: {STORE_FORMATS}")
        return list(dict.fromkeys(v))

    def resolved_transactions_per_store_day(self) -> int:
        """Receipts per store-day: the profile's value, else derived from the ticket."""
        if self.transactions_per_store_day is not None:
            return self.transactions_per_store_day
        derived = (
            _REFERENCE_TRANSACTIONS
            * (_REFERENCE_TICKET / self.avg_ticket_target) ** _TICKET_TRAFFIC_EXPONENT
        )
        return int(min(_MAX_DERIVED_TRANSACTIONS, max(_MIN_DERIVED_TRANSACTIONS, round(derived))))

    def resolved_operating_hours(self) -> list[str]:
        """Opening-hour patterns: the profile's, else the hours its traffic weights use."""
        if self.operating_hours is not None:
            return list(self.operating_hours)
        floor = max(self.hourly_weights) * _OPEN_HOUR_PEAK_SHARE
        busy = [hour for hour, weight in enumerate(self.hourly_weights) if weight >= floor]
        open_hour, close_hour = busy[0], busy[-1] + 1
        if open_hour <= 1 and close_hour >= 23:
            return ["24h"]
        return [f"{open_hour}-{close_hour}"]

    def resolved_store_formats(self) -> list[str]:
        """Store formats: the profile's, else boutiques for high tickets, else every format."""
        if self.store_formats is not None:
            return list(self.store_formats)
        if self.avg_ticket_target >= _BOUTIQUE_TICKET:
            return ["boutique"]
        return [name for name in STORE_FORMATS if name != "boutique"]

    def resolved_customer_visits_per_year(self) -> float:
        """Mean yearly trips of an active customer: the profile's, else from the ticket.

        Shoppers return less often to stores with bigger tickets.
        """
        if self.customer_visits_per_year is not None:
            return self.customer_visits_per_year
        derived = _REFERENCE_VISITS * (_REFERENCE_TICKET / self.avg_ticket_target) ** (
            _TICKET_VISITS_EXPONENT
        )
        return float(min(_MAX_DERIVED_VISITS, max(_MIN_DERIVED_VISITS, round(derived, 1))))

    @model_validator(mode="after")
    def _hours_have_traffic(self) -> "StoreTypeProfile":
        # A store pattern whose open hours all carry zero weight would never sell.
        for value in self.resolved_operating_hours():
            open_hour, close_hour = parse_operating_hours(value)
            if sum(self.hourly_weights[open_hour:close_hour]) <= 0:
                raise ValueError(f"operating hours {value!r} have no hourly traffic weight")
        return self

    @field_validator("hourly_weights", "daily_weights", "monthly_weights")
    @classmethod
    def _weights_nonneg(cls, v: list[float]) -> list[float]:
        if any(w < 0 for w in v):
            raise ValueError("weight values must be non-negative")
        return v

    @field_validator("hourly_weights", "daily_weights", "monthly_weights")
    @classmethod
    def _weights_have_effect(cls, v: list[float]) -> list[float]:
        # An all-zero weight vector is a declared-but-unused control: it can't be
        # normalized and would silently suppress all traffic. Reject it so profile
        # controls provably drive generation (IMP-010 validated profile controls).
        if sum(v) <= 0:
            raise ValueError("weight values must not all be zero")
        return v

    @field_validator("hourly_weights")
    @classmethod
    def _24(cls, v: list[float]) -> list[float]:
        if len(v) != 24:
            raise ValueError("hourly_weights must have 24 entries")
        return v

    @field_validator("daily_weights")
    @classmethod
    def _7(cls, v: list[float]) -> list[float]:
        if len(v) != 7:
            raise ValueError("daily_weights must have 7 entries")
        return v

    @field_validator("monthly_weights")
    @classmethod
    def _12(cls, v: list[float]) -> list[float]:
        if len(v) != 12:
            raise ValueError("monthly_weights must have 12 entries")
        return v

    @field_validator("department_weights")
    @classmethod
    def _departments_have_effect(cls, v: dict[str, float]) -> dict[str, float]:
        # Every declared department must carry positive weight: a zero/negative
        # weight is an unused control that silently excludes a department from
        # every basket (IMP-010 validated profile controls).
        bad = sorted(k for k, w in v.items() if w <= 0)
        if bad:
            raise ValueError(f"department_weights must be positive; got <=0 for {bad}")
        return v


"""Load and validate dictionary JSON sets (shared + one store type)."""

import json
from dataclasses import dataclass, field
from importlib.resources import files
from pathlib import Path

from pydantic import BaseModel, ValidationError



@dataclass
class DictionarySet:
    store_type: str
    profile: StoreTypeProfile
    first_names: list[NameEntry] = field(default_factory=list)
    last_names: list[NameEntry] = field(default_factory=list)
    geographies: list[GeographyEntry] = field(default_factory=list)
    tax_rates: list[TaxJurisdictionEntry] = field(default_factory=list)
    products: list[ProductEntry] = field(default_factory=list)
    brands: list[ProductBrandEntry] = field(default_factory=list)
    tags: list[ProductTagEntry] = field(default_factory=list)


def default_dictionary_root() -> Path:
    """Return the packaged dictionary root, with a source-tree fallback."""
    packaged_root = files("retail_setup").joinpath("resources", "dictionaries")
    if packaged_root.is_dir():
        if not isinstance(packaged_root, Path):
            raise RuntimeError("Packaged dictionaries are not available as filesystem paths.")
        return packaged_root

    source_root = Path(__file__).resolve().parents[3] / "data" / "dictionaries"
    if source_root.is_dir():
        return source_root
    raise RuntimeError("Default dictionaries are missing from the installed package.")


def available_store_types(root: Path) -> list[str]:
    if not root.is_dir():
        return []
    return sorted(
        p.name
        for p in root.iterdir()
        if p.is_dir() and not p.name.startswith("_") and (p / "profile.json").exists()
    )


def load_list(path: Path, model: type[BaseModel]) -> list:
    """Load and validate a JSON array file, returning a list of model instances.

    Raises ValueError with filename and row index on validation failure.
    """
    try:
        raw = json.loads(path.read_text())
    except FileNotFoundError:
        raise ValueError(f"missing dictionary file: {path}") from None
    if not isinstance(raw, list):
        raise ValueError(f"{path.name}: expected a JSON array")
    result = []
    for i, row in enumerate(raw):
        try:
            result.append(model.model_validate(row))
        except ValidationError as exc:
            raise ValueError(f"{path.name}[{i}]: {exc}") from exc
    return result


# Keep the private alias for any callers that haven't migrated yet.
_load_list = load_list


def load_dictionaries(root: Path, store_type: str) -> DictionarySet:
    type_dir = root / store_type
    if not (type_dir / "profile.json").exists():
        raise ValueError(
            f"unknown store type {store_type!r}; available: {available_store_types(root)}"
        )
    shared = root / "_shared"

    try:
        profile = StoreTypeProfile.model_validate(
            json.loads((type_dir / "profile.json").read_text())
        )
    except (json.JSONDecodeError, ValidationError) as exc:
        raise ValueError(f"profile.json: {exc}") from exc
    if profile.store_type != store_type:
        raise ValueError(
            f"profile.json store_type {profile.store_type!r} does not match folder {store_type!r}"
        )

    tags_path = type_dir / "tags.json"
    return DictionarySet(
        store_type=store_type,
        profile=profile,
        first_names=load_list(shared / "first_names.json", NameEntry),
        last_names=load_list(shared / "last_names.json", NameEntry),
        geographies=load_list(shared / "geographies.json", GeographyEntry),
        tax_rates=load_list(shared / "tax_rates.json", TaxJurisdictionEntry),
        products=load_list(type_dir / "products.json", ProductEntry),
        brands=load_list(type_dir / "brands.json", ProductBrandEntry),
        tags=load_list(tags_path, ProductTagEntry) if tags_path.exists() else [],
    )


"""Generation settings (utility/config.yaml). Environment settings live in deploy/config/."""

import calendar
import json
from datetime import date, timedelta
from pathlib import Path
from typing import Any

import yaml
from pydantic import BaseModel, Field, ValidationError, model_validator


# Minimum number of synthetic customers, regardless of store_count. The churn
# model (fabric/lakehouse/09-ml-churn-prediction.ipynb) needs at least two
# customers in each class (active / churned) to build a train/test split; very
# small customer pools can leave fewer than two churned customers and abort
# training. Flooring the count keeps small-store demos trainable.
MIN_CUSTOMER_COUNT = 5000
# Fallbacks when the store type's profile cannot be read (the store type is
# then reported as unknown by validation).
DEFAULT_TRANSACTIONS_PER_STORE_DAY = 400
DEFAULT_ONLINE_ORDER_SHARE = 0.12
DEFAULT_CUSTOMER_VISITS_PER_YEAR = 8.0
# Expected store daily_traffic_multiplier: the mean of generation.dims
# VOLUME_CLASSES (a test keeps the two in step).
MEAN_TRAFFIC_MULTIPLIER = 1.0525
# Share of all customers active on a typical day of an 18-month history at the
# default churn rate (generation.customers turns the base over by churn and
# acquisition); sizes the default customer count.
NOMINAL_ACTIVE_SHARE = 0.75


def default_online_orders_per_day(
    online_order_share: float, store_count: int, transactions_per_store_day: float
) -> int:
    """Network online orders per day so online is ``online_order_share`` of all orders.

    In-store volume is ``store_count * transactions_per_store_day`` at the mean
    traffic multiplier; online orders make up the given share of the total.
    """
    share = min(max(online_order_share, 0.0), 0.95)
    in_store = store_count * transactions_per_store_day * MEAN_TRAFFIC_MULTIPLIER
    return max(1, round(in_store * share / (1.0 - share)))


def default_customer_count(
    store_count: int,
    transactions_per_store_day: float,
    anonymous_share: float,
    visits_per_year: float,
) -> int:
    """Customers so the identified receipts give active customers ``visits_per_year`` trips.

    Identified receipts per year are ``store_count * transactions_per_store_day
    * 365 * (1 - anonymous_share)`` at the mean traffic multiplier, spread over
    the active customers, who are ``NOMINAL_ACTIVE_SHARE`` of all customers.
    """
    identified = (store_count * transactions_per_store_day * 365 * MEAN_TRAFFIC_MULTIPLIER) * (
        1.0 - anonymous_share
    )
    return max(1, round(identified / (NOMINAL_ACTIVE_SHARE * visits_per_year)))


def _profile_for(data: dict[str, Any]) -> StoreTypeProfile | None:
    """The configured store type's profile, or None when it cannot be read."""
    root = data.get("dictionary_root")
    base = Path(root) if root else default_dictionary_root()
    store_type = data.get("store_type", GenerationConfig.model_fields["store_type"].default)
    if not isinstance(store_type, str):
        return None
    try:
        raw = json.loads((base / store_type / "profile.json").read_text(encoding="utf-8"))
        return StoreTypeProfile.model_validate(raw)
    except (OSError, ValueError, ValidationError):
        return None


def _subtract_months(anchor: date, months: int) -> date:
    """Return the date ``months`` calendar months before ``anchor`` (day clamped)."""

    month_index = anchor.month - 1 - months
    year = anchor.year + month_index // 12
    month = month_index % 12 + 1
    day = min(anchor.day, calendar.monthrange(year, month)[1])
    return date(year, month, day)


class GenerationConfig(BaseModel):
    store_type: str = "supercenter"
    # Preferred input: number of months of history to generate. The window ends
    # *yesterday* so real-time streaming continues seamlessly from today.
    # ``start_date``/``end_date`` may be provided explicitly instead (back-compat);
    # they are derived from ``months`` only when not both already supplied.
    months: int | None = Field(default=None, ge=1, le=120)
    start_date: date
    end_date: date
    # Anchor for dimension dates (product launch dates). Dimensions depend on
    # the seed and this anchor only, never on the run date, so re-rendering a
    # ``months`` config on a later day regenerates identical dimensions.
    # `retail-setup configure` pins it; it defaults to ``start_date``.
    anchor_date: date
    store_count: int = Field(default=50, gt=0, le=2000)
    seed: int = 42
    silver_db: str = "ag"
    gold_db: str = "au"
    # Optional override for dictionary root; when None, default_dictionary_root() is used.
    # Pass an absolute path string to point at a custom dictionary tree (e.g. a Fabric
    # lakehouse Files mount) without mutating the package data directory.
    dictionary_root: str | None = None

    # scale knobs; omitted or None -> derived from store_count before validation
    dc_count: int = Field(gt=0, le=999)
    customer_count: int = Field(gt=0)
    # base in-store transactions per store-day at multiplier 1.0; profiles'
    # hourly/daily/monthly weights shape it, store daily_traffic_multiplier scales it.
    # Omitted -> the store type's profile value (luxury boutiques far below supercenters).
    transactions_per_store_day: int = Field(gt=0)
    # fraction of SALE receipts returned per day (Dec 26 spikes 6x, capped 10%)
    # nominal daily return share; Dec 26 applies a 6x spike capped at 0.10, so
    # values near the ceiling flatten the spike - keep this small (~0.01)
    return_rate: float = Field(default=0.01, ge=0.0, le=0.10)
    # network-wide online orders per day at multiplier 1.0; omitted -> derived so
    # online orders are the profile's online_order_share of all orders
    online_orders_per_day: int = Field(gt=0)
    # number of category-matched branded SKUs generated per base catalog product
    # (datagen combinatorial SKUs); 1 = one SKU per dictionary row
    brands_per_product: int = Field(default=3, ge=1, le=10)
    # truck load capacity in units; a store-day shipment exceeding this is split
    # across multiple truck legs (datagen multi-truck shipments)
    truck_capacity: int = Field(default=15000, gt=0)
    # share of in-store receipts with no loyalty customer (cash and anonymous
    # shoppers); the rest carry a customer_id (generation.customers)
    anonymous_share: float = Field(default=0.45, ge=0.0, lt=1.0)
    # share of the customers active at a time who churn within a year; churn is
    # driven by segment, trip frequency and spend, with a tapering lead-in
    annual_churn_rate: float = Field(default=0.20, gt=0.0, lt=1.0)
    # year-over-year growth of store traffic and online orders, compounding from
    # anchor_date (1.0 there); prices carry their own inflation (pricing.py)
    annual_volume_growth: float = Field(default=0.03, ge=-0.5, le=1.0)

    @model_validator(mode="before")
    @classmethod
    def _derive_defaults(cls, data: Any) -> Any:
        """Fill the date window and scale knobs that derive from other settings."""

        if not isinstance(data, dict):
            return data
        data = dict(data)
        months = data.get("months")
        if months is not None and not (data.get("start_date") and data.get("end_date")):
            # Invalid months are left for field validation to report.
            if isinstance(months, int) and not isinstance(months, bool) and months >= 1:
                end = date.today() - timedelta(days=1)
                data["end_date"] = end
                data["start_date"] = _subtract_months(end, months)
        elif data.get("start_date") is None or data.get("end_date") is None:
            raise ValueError("provide `months`, or both `start_date` and `end_date`")
        if data.get("anchor_date") is None:
            data["anchor_date"] = data["start_date"]
        profile = _profile_for(data)
        if data.get("transactions_per_store_day") is None:
            data["transactions_per_store_day"] = (
                profile.resolved_transactions_per_store_day()
                if profile is not None
                else DEFAULT_TRANSACTIONS_PER_STORE_DAY
            )
        store_count = data.get("store_count", cls.model_fields["store_count"].default)
        transactions = data["transactions_per_store_day"]
        numeric = isinstance(transactions, (int, float)) and not isinstance(transactions, bool)
        if isinstance(store_count, int) and not isinstance(store_count, bool):
            if data.get("dc_count") is None:
                data["dc_count"] = max(1, store_count // 10)
            anonymous = data.get("anonymous_share", cls.model_fields["anonymous_share"].default)
            share_ok = isinstance(anonymous, (int, float)) and not isinstance(anonymous, bool)
            if data.get("customer_count") is None and numeric and share_ok:
                data["customer_count"] = default_customer_count(
                    store_count,
                    transactions,
                    anonymous,
                    profile.resolved_customer_visits_per_year()
                    if profile is not None
                    else DEFAULT_CUSTOMER_VISITS_PER_YEAR,
                )
            if data.get("online_orders_per_day") is None and numeric:
                data["online_orders_per_day"] = default_online_orders_per_day(
                    profile.online_order_share
                    if profile is not None
                    else DEFAULT_ONLINE_ORDER_SHARE,
                    store_count,
                    transactions,
                )
        return data

    @model_validator(mode="after")
    def _known_store_type(self) -> "GenerationConfig":
        root = Path(self.dictionary_root) if self.dictionary_root else default_dictionary_root()
        known = available_store_types(root)
        if self.store_type not in known:
            raise ValueError(f"store_type {self.store_type!r} not found; available: {known}")
        return self

    @property
    def resolved_dictionary_root(self) -> Path:
        """Resolved dictionary root path (explicit override or package default)."""
        return Path(self.dictionary_root) if self.dictionary_root else default_dictionary_root()

    @model_validator(mode="after")
    def _date_order(self) -> "GenerationConfig":
        if self.end_date < self.start_date:
            raise ValueError("end_date must be on or after start_date")
        return self

    @model_validator(mode="after")
    def _customer_floor(self) -> "GenerationConfig":
        # Guarantee enough customers for the churn model's two-class train/test
        # split; see MIN_CUSTOMER_COUNT above. Applied to the final value so an
        # explicitly small customer_count is lifted too.
        self.customer_count = max(self.customer_count, MIN_CUSTOMER_COUNT)
        return self


def generation_config_path(repo_root: Path, env: str) -> Path:
    """Return the environment-scoped generation configuration path."""

    return repo_root / "utility" / "config" / "environments" / f"{env}.yaml"


def resolve_generation_config_path(repo_root: Path, env: str) -> Path:
    """Prefer environment-scoped generation config, with a legacy fallback."""

    scoped = generation_config_path(repo_root, env)
    if scoped.is_file():
        return scoped
    return repo_root / "utility" / "config.yaml"


def load_generation_config(path: Path, *, dictionary_root: Path | None = None) -> GenerationConfig:
    """Load a generation YAML, optionally pointing it at a custom dictionary root."""

    data = yaml.safe_load(path.read_text())
    if dictionary_root is not None and isinstance(data, dict):
        data = {**data, "dictionary_root": str(dictionary_root)}
    return GenerationConfig.model_validate(data)


"""Authoritative output schemas for generated tables (Plan 2a + 2b scope).

Spark simple type strings. Dimension columns keep the legacy PascalCase names
because the semantic model TMDL binds sourceColumn to them (e.g. StoreNumber,
Cost, MSRP). Most fact tables are snake_case, but several 2b tables have
TMDL-bound PascalCase or mixed-case columns documented per table below.

The TMDL contract test (tests/generation/test_schema_contract.py) is the
arbiter: every sourceColumn in a table's TMDL must be present here with a
compatible type. Extra columns not in TMDL are allowed.

Columns added vs plan (from TMDL audit 2026-06-12):
  fact_receipts: added ("Subtotal", "string") — legacy trace/aggregate string
    column bound as sourceColumn: Subtotal in the semantic model. Not a
    snake_case column; appears to be a pre-existing legacy column the model
    still references.

  --- Plan 2b TMDL deltas (per-table, added to reconcile TMDL bindings) ---

  fact_ble_pings:
    - ("CustomerId", "double") — TMDL sourceColumn is CustomerId (PascalCase),
      not customer_id. Plan listed customer_id (kept as extra); TMDL column added.
    - ("__index_level_0__", "long") — TMDL-bound legacy pandas-index column.

  fact_customer_zone_changes:
    - TMDL uses PascalCase for all data columns: StoreID, CustomerBLEId,
      FromZone, ToZone (plan listed snake_case equivalents, kept as extras).
    - ("StoreID", "long"), ("CustomerBLEId", "string"), ("FromZone", "string"),
      ("ToZone", "string") — TMDL-bound columns added.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_marketing:
    - ("CustomerId", "double") — TMDL sourceColumn is CustomerId; plan's
      customer_id kept as extra.
    - ("CostCents", "long") — TMDL sourceColumn is CostCents; plan's cost_cents
      kept as extra.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_promo_lines:
    - TMDL uses PascalCase: ReceiptId, PromoCode, LineNumber, ProductID, Qty,
      DiscountAmount, DiscountCents (plan listed snake_case equivalents, kept
      as extras).
    - ("ReceiptId", "string"), ("PromoCode", "string"), ("LineNumber", "long"),
      ("ProductID", "long"), ("Qty", "long"), ("DiscountAmount", "string"),
      ("DiscountCents", "long") — TMDL-bound columns added.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_online_order_lines:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_reorders:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_truck_moves:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_truck_inventory:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_dc_inventory_txn:
    - ("Source", "string") — TMDL sourceColumn is PascalCase Source. The
      lowercase ``source`` that appears during construction is renamed to
      ``Source`` in the final select (inventory.py); keeping both would cause
      a case-insensitive collision that Delta on Fabric rejects.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_store_inventory_txn:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_stockouts:
    - ("__index_level_0__", "long") — TMDL-bound.

Columns in plan but NOT in TMDL (allowed — Direct Lake ignores extra columns):
  fact_receipts: trace_id, tender_type
  fact_payments: tender_type (not in payments TMDL at all)
  fact_ble_pings: customer_id (TMDL uses CustomerId), trace_id, event_ts,
    event_date
  fact_customer_zone_changes: store_id, customer_ble_id, from_zone, to_zone,
    event_ts, trace_id, event_date
  fact_marketing: customer_id, cost_cents, event_ts, trace_id, event_date,
    channel (extra beyond TMDL-bound channel — already in TMDL, fine)
  fact_promo_lines: receipt_id_ext, promo_code, line_number, product_id,
    quantity, discount_amount, discount_cents, event_ts, trace_id, event_date
  fact_dc_inventory_txn: trace_id, event_ts, event_date
    (lowercase source is renamed to Source in the final select; not an extra)
"""

# table -> list of (column, spark_type)
TABLES: dict[str, list[tuple[str, str]]] = {
    "dim_geographies": [
        ("ID", "long"),
        ("City", "string"),
        ("State", "string"),
        ("ZipCode", "string"),
        ("District", "string"),
        ("Region", "string"),
    ],
    "dim_stores": [
        ("ID", "long"),
        ("StoreNumber", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
        ("tax_rate", "double"),
        ("volume_class", "string"),
        ("store_format", "string"),
        ("operating_hours", "string"),
        ("daily_traffic_multiplier", "double"),
        # IANA time zone from the store geography's state. Generation works in
        # this local time; Silver timestamps are UTC (see timezones.py).
        ("TimeZone", "string"),
    ],
    "dim_distribution_centers": [
        ("ID", "long"),
        ("DCNumber", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
    ],
    "dim_trucks": [
        ("ID", "long"),
        ("LicensePlate", "string"),
        ("Refrigeration", "boolean"),
        # double, not long: NULL for pool trucks + Direct Lake nullability
        ("DCID", "double"),
    ],
    "dim_customers": [
        ("ID", "long"),
        ("FirstName", "string"),
        ("LastName", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
        ("LoyaltyCard", "string"),
        ("Phone", "string"),
        ("BLEId", "string"),
        ("AdId", "string"),
        # The store where the customer makes most visits (generation.customers).
        ("HomeStoreID", "long"),
    ],
    "dim_products": [
        ("ID", "long"),
        ("ProductName", "string"),
        ("Brand", "string"),
        ("Company", "string"),
        ("Department", "string"),
        ("Category", "string"),
        ("Subcategory", "string"),
        ("Cost", "double"),
        ("MSRP", "double"),
        ("SalePrice", "double"),
        ("RequiresRefrigeration", "boolean"),
        ("LaunchDate", "timestamp"),
        ("taxability", "string"),
        ("Tags", "string"),
    ],
    "dim_date": [
        ("date_key", "long"),
        ("date", "date"),
        ("year", "long"),
        ("quarter", "long"),
        ("month", "long"),
        ("month_name", "string"),
        ("day", "long"),
        ("day_of_week", "long"),
        ("day_name", "string"),
        ("week_of_year", "long"),
        ("is_weekend", "long"),
        ("fiscal_year", "long"),
        ("fiscal_quarter", "long"),
    ],
    "fact_receipts": [
        ("receipt_id_ext", "string"),
        ("trace_id", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("store_id", "long"),
        ("customer_id", "long"),
        ("receipt_type", "string"),
        ("tender_type", "string"),
        ("subtotal_cents", "long"),
        ("discount_amount", "string"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("subtotal_amount", "string"),
        ("tax_amount", "string"),
        ("total_amount", "string"),
        ("payment_method", "string"),
        # Legacy column bound by the semantic model (sourceColumn: Subtotal).
        # Present in fact_receipts.tmdl as a string column; added here to
        # satisfy the TMDL contract test (TMDL arbiter rule).
        ("Subtotal", "string"),
        # --- IMP-007 marketing attribution (additive, snake_case) ---
        # gross_subtotal_cents = subtotal_cents (net, unchanged) + discount_cents;
        # discount before tax is preserved (subtotal_cents stays the net amount).
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        # NULL unless this SALE receipt was selected for marketing attribution.
        ("attribution_journey_id", "string"),
        ("campaign_id", "string"),
        ("impression_id_ext", "string"),
        # Store-local hour (0-23) of the UTC event_ts, for hour-of-day reporting.
        ("local_hour", "int"),
    ],
    "fact_receipt_lines": [
        ("receipt_id_ext", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("line_num", "int"),
        ("product_id", "long"),
        ("quantity", "int"),
        ("unit_price", "string"),
        ("unit_cents", "long"),
        ("ext_price", "string"),
        ("ext_cents", "long"),
        ("promo_code", "string"),
    ],
    "fact_payments": [
        ("receipt_id_ext", "string"),
        ("order_id_ext", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("payment_method", "string"),
        ("amount_cents", "long"),
        ("amount", "string"),
        ("transaction_id", "string"),
        ("status", "string"),
        ("decline_reason", "string"),
        ("processing_time_ms", "long"),
        ("store_id", "long"),
        ("customer_id", "long"),
        # IMP-007: NULL unless this payment's receipt/order was attributed.
        ("attribution_journey_id", "string"),
    ],
    # -----------------------------------------------------------------------
    # Plan 2b: 15 remaining fact tables
    # -----------------------------------------------------------------------
    "fact_store_ops": [
        # TMDL-bound: trace_id, store_id (int64→long), operation_type, event_date
        # Extra (not in TMDL, allowed): event_ts
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("operation_type", "string"),
        ("event_date", "date"),
    ],
    "fact_foot_traffic": [
        # TMDL-bound: count (int64→long), zone, dwell_seconds (int64→long),
        #   store_id (int64→long), sensor_id, event_date
        # Extra (not in TMDL, allowed): event_ts, trace_id
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("sensor_id", "string"),
        ("zone", "string"),
        ("dwell_seconds", "long"),
        ("count", "long"),
        ("event_date", "date"),
    ],
    "fact_ble_pings": [
        # TMDL-bound: beacon_id, rssi (int64→long), customer_ble_id, zone,
        #   store_id (int64→long), CustomerId (double), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   customer_id (plan name; TMDL uses CustomerId)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("beacon_id", "string"),
        ("customer_ble_id", "string"),
        ("customer_id", "double"),
        # TMDL-bound PascalCase column (sourceColumn: CustomerId)
        ("CustomerId", "double"),
        ("rssi", "long"),
        ("zone", "string"),
        ("event_date", "date"),
        # TMDL-bound legacy pandas-index column
        ("__index_level_0__", "long"),
    ],
    "fact_customer_zone_changes": [
        # TMDL-bound PascalCase: StoreID (int64→long), CustomerBLEId (string),
        #   FromZone (string), ToZone (string), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   store_id, customer_ble_id, from_zone, to_zone (snake_case plan names)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        # snake_case plan columns (extras)
        ("store_id", "long"),
        ("customer_ble_id", "string"),
        ("from_zone", "string"),
        ("to_zone", "string"),
        # TMDL-bound PascalCase columns
        ("StoreID", "long"),
        ("CustomerBLEId", "string"),
        ("FromZone", "string"),
        ("ToZone", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_marketing": [
        # TMDL-bound: device, cost, customer_ad_id, impression_id_ext, campaign_id,
        #   creative_id, channel, CustomerId (double), CostCents (int64→long),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   customer_id (plan name; TMDL uses CustomerId),
        #   cost_cents (plan name; TMDL uses CostCents)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("channel", "string"),
        ("campaign_id", "string"),
        ("creative_id", "string"),
        ("customer_ad_id", "string"),
        # snake_case plan columns (extras)
        ("customer_id", "double"),
        ("cost_cents", "long"),
        # TMDL-bound PascalCase columns (sourceColumn: CustomerId, CostCents)
        ("CustomerId", "double"),
        ("CostCents", "long"),
        ("impression_id_ext", "string"),
        ("cost", "string"),
        ("device", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
        # IMP-007: NULL for organic/background impressions; set on the two
        # synthetic touches created for a selected purchase's journey.
        ("attribution_journey_id", "string"),
    ],
    "fact_promotions": [
        # TMDL-bound: receipt_id_ext, promo_code, discount_amount, discount_cents
        #   (int64→long), discount_type, product_count (int64→long), product_ids,
        #   store_id (int64→long), customer_id (int64→long), event_date
        # Extra (not in TMDL, allowed): event_ts, trace_id
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("receipt_id_ext", "string"),
        ("promo_code", "string"),
        ("discount_amount", "string"),
        ("discount_cents", "long"),
        ("discount_type", "string"),
        ("product_count", "long"),
        ("product_ids", "string"),
        ("store_id", "long"),
        ("customer_id", "long"),
        ("event_date", "date"),
        # IMP-007: NULL unless the underlying receipt was attributed.
        ("attribution_journey_id", "string"),
    ],
    "fact_promo_lines": [
        # TMDL-bound PascalCase: ReceiptId, PromoCode, LineNumber (int64→long),
        #   ProductID (int64→long), Qty (int64→long), DiscountAmount (string),
        #   DiscountCents (int64→long), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   receipt_id_ext, promo_code, line_number, product_id, quantity,
        #   discount_amount, discount_cents (snake_case plan names)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        # snake_case plan columns (extras)
        ("receipt_id_ext", "string"),
        ("promo_code", "string"),
        ("line_number", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("discount_amount", "string"),
        ("discount_cents", "long"),
        # TMDL-bound PascalCase columns
        ("ReceiptId", "string"),
        ("PromoCode", "string"),
        ("LineNumber", "long"),
        ("ProductID", "long"),
        ("Qty", "long"),
        ("DiscountAmount", "string"),
        ("DiscountCents", "long"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_online_order_headers": [
        # TMDL-bound: customer_id (int64→long), subtotal_cents (int64→long),
        #   order_id_ext, payment_method, total_amount, total_cents (int64→long),
        #   tax_amount, subtotal_amount, tax_cents (int64→long), event_date
        # Extra (not in TMDL, allowed): event_ts
        ("order_id_ext", "string"),
        ("customer_id", "long"),
        ("subtotal_cents", "long"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("subtotal_amount", "string"),
        ("tax_amount", "string"),
        ("total_amount", "string"),
        ("payment_method", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        # --- IMP-007 marketing attribution (additive, snake_case) ---
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        ("attribution_journey_id", "string"),
        ("campaign_id", "string"),
        ("impression_id_ext", "string"),
    ],
    # -----------------------------------------------------------------------
    # IMP-007: dedicated marketing-attribution fact. One row per in-store SALE
    # receipt or online order, including attributed, unattributed, declined,
    # and reconciliation-failed outcomes (see attribution.py).
    # -----------------------------------------------------------------------
    "fact_marketing_attribution": [
        ("attribution_id", "string"),
        ("attribution_journey_id", "string"),
        ("attribution_status", "string"),
        ("attribution_model", "string"),
        ("attribution_window_days", "int"),
        ("impression_id_ext", "string"),
        ("campaign_id", "string"),
        ("creative_id", "string"),
        ("channel", "string"),
        ("customer_ad_id", "string"),
        ("customer_id", "long"),
        ("touch_ts", "timestamp"),
        ("purchase_ts", "timestamp"),
        ("lag_seconds", "long"),
        ("purchase_type", "string"),
        ("receipt_id_ext", "string"),
        ("order_id_ext", "string"),
        ("store_id", "long"),
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        ("net_subtotal_cents", "long"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("payment_cents", "long"),
        ("attributed_revenue_cents", "long"),
        ("event_date", "date"),
    ],
    "fact_online_order_lines": [
        # TMDL-bound: order_id, unit_cents (int64→long), fulfillment_mode,
        #   promo_code, node_type, product_id (int64→long), ext_price, ext_cents
        #   (int64→long), line_num (int64→long), fulfillment_status, unit_price,
        #   quantity (int64→long), node_id (int64→long), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): picked_ts, shipped_ts, delivered_ts,
        #   event_ts, event_date
        ("order_id", "string"),
        ("product_id", "long"),
        ("line_num", "long"),
        ("quantity", "long"),
        ("unit_price", "string"),
        ("unit_cents", "long"),
        ("ext_price", "string"),
        ("ext_cents", "long"),
        ("promo_code", "string"),
        ("fulfillment_mode", "string"),
        ("fulfillment_status", "string"),
        ("node_type", "string"),
        ("node_id", "long"),
        ("picked_ts", "timestamp"),
        ("shipped_ts", "timestamp"),
        ("delivered_ts", "timestamp"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_reorders": [
        # TMDL-bound: store_id (int64→long), dc_id (int64→long), product_id
        #   (int64→long), current_quantity (int64→long), reorder_quantity
        #   (int64→long), reorder_point (int64→long), priority,
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("dc_id", "long"),
        ("product_id", "long"),
        ("current_quantity", "long"),
        ("reorder_quantity", "long"),
        ("reorder_point", "long"),
        ("priority", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_truck_moves": [
        # TMDL-bound: status, shipment_id, dc_id (int64→long), truck_id
        #   (int64→long), store_id (int64→long), actual_unload_duration (double),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, eta, etd,
        #   departure_time, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("truck_id", "long"),
        ("dc_id", "long"),
        ("store_id", "long"),
        ("shipment_id", "string"),
        ("status", "string"),
        ("eta", "timestamp"),
        ("etd", "timestamp"),
        ("departure_time", "timestamp"),
        ("actual_unload_duration", "double"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
        # Planned vs actual arrival. Historical moves arrive as planned (delay
        # 0, no reason) until the logistics incident model populates delays.
        ("planned_arrival_time", "timestamp"),
        ("delay_minutes", "double"),
        ("delay_reason", "string"),
    ],
    "fact_truck_inventory": [
        # TMDL-bound: truck_id (int64→long), shipment_id, product_id (int64→long),
        #   quantity (int64→long), action, location_id (int64→long), location_type,
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("truck_id", "long"),
        ("shipment_id", "string"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("action", "string"),
        ("location_id", "long"),
        ("location_type", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_dc_inventory_txn": [
        # TMDL-bound: txn_type, quantity (int64→long), dc_id (int64→long),
        #   balance (int64→long), product_id (int64→long), Source (PascalCase!),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   source (lowercase snake_case plan name)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("dc_id", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("balance", "long"),
        ("txn_type", "string"),
        # TMDL-bound PascalCase column (sourceColumn: Source). The lowercase
        # ``source`` used during construction is renamed AS "Source" in
        # inventory.py before the final select; keeping both would produce a
        # case-insensitive duplicate that Delta on Fabric rejects.
        ("Source", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_store_inventory_txn": [
        # TMDL-bound: txn_type, quantity (int64→long), balance (int64→long),
        #   product_id (int64→long), store_id (int64→long), source (lowercase OK),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("balance", "long"),
        ("txn_type", "string"),
        ("source", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
        # Attempted demand is retained even when a whole line cannot be filled.
        # LOST_SALE changes no stock (quantity=0); all non-demand rows use zero.
        ("requested_quantity", "long"),
        ("unfulfilled_quantity", "long"),
    ],
    "fact_stockouts": [
        # PascalCase IS the contract: TMDL binds StoreID (double), DCID (double),
        #   ProductID (int64→long), LastKnownQuantity (int64→long),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("StoreID", "double"),
        ("DCID", "double"),
        ("ProductID", "long"),
        ("LastKnownQuantity", "long"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    # Logistics disruptions (DC outages, lane delays, weather, holidays,
    # supplier short-ships): one row per incident, resolved_at NULL while open.
    # store_id is NULL for network- or DC-wide incidents. Historical
    # generation publishes an empty table until the incident model lands; the
    # live stream fills it from logistics_incident_started/_resolved.
    "fact_logistics_incidents": [
        ("incident_id", "string"),
        ("incident_type", "string"),
        ("dc_id", "long"),
        ("store_id", "long"),
        ("started_at", "timestamp"),
        ("resolved_at", "timestamp"),
        ("severity", "string"),
        ("affected_store_count", "long"),
        ("description", "string"),
        ("trace_id", "string"),
        ("event_date", "date"),
    ],
    # -----------------------------------------------------------------------
    # Plan 2c: 10 Gold (au) aggregate tables
    # TMDL audit 2026-06-12: bindings match exactly; `computed_at` and `as_of`
    # are produced by the legacy transforms but unbound in TMDL (extras OK).
    # TMDL `day`/`ts` are dateTime — TYPE_COMPAT accepts timestamp|date.
    # -----------------------------------------------------------------------
    "sales_minute_store": [
        ("store_id", "long"),
        ("ts", "timestamp"),
        ("total_sales", "double"),
        ("receipts", "long"),
        ("avg_basket", "double"),
    ],
    "top_products_15m": [
        ("product_id", "long"),
        ("revenue", "double"),
        ("units", "long"),
        ("computed_at", "timestamp"),  # window end; bound as 'Window End' in TMDL
    ],
    "inventory_position_current": [
        ("store_id", "long"),
        ("product_id", "long"),
        ("on_hand", "long"),
        ("as_of", "timestamp"),
    ],
    "dc_inventory_position_current": [
        ("dc_id", "long"),
        ("product_id", "long"),
        ("on_hand", "long"),
        ("as_of", "timestamp"),
    ],
    "truck_dwell_daily": [
        ("site", "string"),
        ("day", "date"),
        ("avg_dwell_min", "double"),
        ("trucks", "long"),
    ],
    "online_sales_daily": [
        ("day", "date"),
        ("orders", "long"),
        ("subtotal", "double"),
        ("tax", "double"),
        ("total", "double"),
        ("avg_order_value", "double"),
    ],
    "zone_dwell_minute": [
        ("store_id", "long"),
        ("zone", "string"),
        ("ts", "timestamp"),
        ("avg_dwell", "double"),
        ("customers", "long"),
    ],
    "marketing_cost_daily": [
        ("campaign_id", "string"),
        ("day", "date"),
        ("impressions", "long"),
        ("cost", "double"),
    ],
    "campaign_performance_daily": [
        ("campaign_id", "string"),
        ("channel", "string"),
        ("day", "date"),
        ("impressions", "long"),
        ("spend_cents", "long"),
        ("conversions", "long"),
        ("attributed_revenue_cents", "long"),
        ("discount_cents", "long"),
        ("tax_cents", "long"),
        ("payment_cents", "long"),
        ("conversion_rate", "double"),
        ("roas", "double"),
    ],
    "tender_mix_daily": [
        ("day", "date"),
        ("payment_method", "string"),
        ("transactions", "long"),
        ("total_amount", "double"),
    ],
}


_SPARK_TYPE_MAP = None


def _type_map():
    """Lazy-import PySpark type map (avoids import cost when pyspark not installed)."""
    global _SPARK_TYPE_MAP
    if _SPARK_TYPE_MAP is None:
        from pyspark.sql.types import (
            BooleanType,
            DateType,
            DoubleType,
            IntegerType,
            LongType,
            StringType,
            TimestampType,
        )

        _SPARK_TYPE_MAP = {
            "long": LongType(),
            "int": IntegerType(),
            "string": StringType(),
            "double": DoubleType(),
            "boolean": BooleanType(),
            "timestamp": TimestampType(),
            "date": DateType(),
        }
    return _SPARK_TYPE_MAP


def spark_schema(table: str):
    """Build a StructType for createDataFrame with explicit types."""
    from pyspark.sql.types import StructField, StructType

    tmap = _type_map()
    fields = [StructField(name, tmap[typ], nullable=True) for name, typ in TABLES[table]]
    return StructType(fields)


def column_names(table: str) -> list[str]:
    return [name for name, _ in TABLES[table]]


"""Deterministic seeding + partition grids for the generation engine."""

import hashlib
import math
from collections.abc import Callable, Iterable
from datetime import date

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F


# Spark's default of 200 shuffle partitions leaves billion-row full-demo shuffles
# (receipt lines, inventory events, BLE pings) at millions of rows per task: long
# spilling sorts and too few tasks to occupy a large pool. Target roughly one
# million rows of the dominant stream per partition; adaptive query execution
# still coalesces the many small shuffles back to its advisory size.
SHUFFLE_ROWS_PER_PARTITION = 1_000_000
MIN_SHUFFLE_PARTITIONS = 200
MAX_SHUFFLE_PARTITIONS = 8192


def shuffle_partitions_for(cfg: GenerationConfig, basket_lambda: float = 12.0) -> int:
    """Shuffle partition count sized to the configured receipt-line volume.

    Rounded up to a multiple of 64 so tasks spread evenly across executors.
    Generation output is independent of the partition count.
    """
    days = (cfg.end_date - cfg.start_date).days + 1
    lines = cfg.store_count * days * cfg.transactions_per_store_day * basket_lambda
    wanted = math.ceil(lines / SHUFFLE_ROWS_PER_PARTITION / 64) * 64
    return max(MIN_SHUFFLE_PARTITIONS, min(MAX_SHUFFLE_PARTITIONS, wanted))


class seeded_draws:
    """Deterministic draw expressions bound to a seed.

    u(cols, salt)        -> uniform [0,1) double
    gauss(cols, salt)    -> N(0,1) (Box-Muller)
    negative_binomial(cols, salt, mean, shape) -> count, variance mean + mean²/shape
    h64(cols, salt)      -> non-negative long hash
    pick_by_weights(cols, salt, [(value, weight), ...]) -> weighted categorical
    """

    _U_MOD = 10**12

    def __init__(self, seed: int):
        self.seed = seed

    def h64(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        return F.pmod(F.xxhash64(*cols, F.lit(f"{salt}|{self.seed}")), F.lit(2**62))

    def u(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        return (self.h64(cols, salt) % F.lit(self._U_MOD)) / F.lit(float(self._U_MOD))

    def gauss(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        radius = F.sqrt(-2.0 * F.log(F.lit(1.0) - self.u(cols, f"{salt}|g1")))
        return radius * F.cos(F.lit(6.283185307179586) * self.u(cols, f"{salt}|g2"))

    def negative_binomial(self, cols: list, salt: str, mean: Column, shape: int = 12) -> Column:
        """Exact sum of geometric draws; integer shape controls overdispersion."""
        if shape < 1:
            raise ValueError("negative-binomial shape must be positive")
        valid_mean = F.when((mean >= 0) & (mean < float("inf")), mean).otherwise(
            F.raise_error("negative-binomial mean must be finite and non-negative")
        )
        probability = valid_mean / (valid_mean + F.lit(float(shape)))
        draws = [
            F.when(valid_mean == 0, F.lit(0))
            .otherwise(
                F.floor(F.log(F.lit(1.0) - self.u(cols, f"{salt}|nb{i}")) / F.log(probability))
            )
            .cast("long")
            for i in range(shape)
        ]
        return sum(draws, F.lit(0).cast("long"))

    def pick_by_weights(self, cols: list, salt: str, weighted: list[tuple[str, float]]):
        from pyspark.sql import functions as F

        total = sum(w for _, w in weighted)
        uu = self.u(cols, salt)
        expr, acc = None, 0.0
        for value, w in weighted[:-1]:
            acc += w / total
            expr = expr.when(uu < acc, value) if expr is not None else F.when(uu < acc, value)
        return expr.otherwise(weighted[-1][0]) if expr is not None else F.lit(weighted[0][0])


def derive_seed(global_seed: int, section: str, key: int, day: date) -> int:
    """Stable 31-bit seed from (global_seed, section, key, day).

    Independent of Spark partitioning/execution order — safe for per-row
    or per-group RNG seeding.
    """
    payload = f"{global_seed}|{section}|{key}|{day.isoformat()}".encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "big") % (2**31)


def legacy_index(*key_cols):
    """Deterministic long for the legacy __index_level_0__ pandas column.

    The semantic model binds the column by name only; nothing consumes its
    values, so a hash beats a dense row_number — which forced a
    single-partition global sort at full volume.
    """
    from pyspark.sql import functions as F

    return F.pmod(F.xxhash64(*key_cols, F.lit("__legacy_index__")), F.lit(2**62))


def bind(value: Column, body: Callable[[Column], Column]) -> Column:
    """``body(value)`` with ``value`` evaluated once.

    Binds ``value`` to a lambda variable (a one-element ``transform``), so a
    sub-expression used several times in ``body`` is not copied into the
    plan; large copied expressions make query planning slow.
    """
    return F.element_at(F.transform(F.array(value), body), 1)


def case_when(branches: Iterable[tuple[Column, object]]) -> Column:
    """Chain `F.when` over `(condition, value)` branches (at least one)."""
    expr: Column | None = None
    for condition, value in branches:
        expr = F.when(condition, value) if expr is None else expr.when(condition, value)
    if expr is None:
        raise ValueError("case_when needs at least one branch")
    return expr


def fixed_width_decimal(value: Column, width: int, label: str) -> Column:
    """Zero-pad a non-negative decimal value and fail if it exceeds ``width``."""
    if width < 1:
        raise ValueError("width must be positive")
    text = value.cast("string")
    invalid = (value < F.lit(0)) | (F.length(text) > F.lit(width))
    error = F.raise_error(F.lit(f"{label} exceeds {width} digits"))
    return (
        F.when(value.isNull(), F.lit(None).cast("string"))
        .when(invalid, error)
        .otherwise(F.lpad(text, width, "0"))
    )


def minimum_width_decimal(value: Column, width: int) -> Column:
    """Zero-pad to at least ``width`` digits without truncating larger values."""
    return minimum_width_text(value.cast("string"), width)


def minimum_width_text(value: Column, width: int) -> Column:
    """Zero-pad text to at least ``width`` characters without truncation."""
    if width < 1:
        raise ValueError("width must be positive")
    text = value.cast("string")
    return F.when(F.length(text) >= width, text).otherwise(F.lpad(text, width, "0"))


def date_grid(
    spark: SparkSession,
    start: date,
    end: date,
    column: str = "day",
) -> DataFrame:
    """Inclusive Spark-native date range without a driver-materialized row list."""
    if end < start:
        raise ValueError("end must be on or after start")
    days = (end - start).days + 1
    return spark.range(days).select(F.date_add(F.lit(start), F.col("id").cast("int")).alias(column))


def store_day_grid(
    spark: SparkSession,
    store_ids: list[int],
    start: date,
    end: date,
    global_seed: int,
    section: str,
) -> DataFrame:
    """store_id x day cross grid.

    Only the small store-ID set is materialized on the driver. Dates and the
    Cartesian expansion remain Spark-native, so long histories do not create a
    large Python row list. Random draws derive from stable row keys rather than
    partition order.
    """
    del global_seed, section
    store_values = F.array(*[F.lit(int(store_id)).cast("long") for store_id in store_ids])
    stores = spark.range(len(store_ids)).select(
        F.element_at(store_values, F.col("id").cast("int") + 1).alias("store_id")
    )
    return stores.crossJoin(date_grid(spark, start, end))


"""Shared retail vocabularies and identifier formats.

Historical generation and the live stream must agree on these names and
formats. Batch modules import them from here, and ``build_notebooks.py``
inlines this module into the stream notebook, so both producers read one
definition.

The module holds:

- payment tenders, their mix, and the decline model
- the named promotion catalog
- identifier formats: BLE IDs, ad IDs, beacon and sensor IDs, campaign IDs
- zone vocabularies and the visit path through a store type's zones
- marketing channels, their cost-per-impression bands, and device types
- online fulfillment modes

The live stream does not use every definition yet. The remaining differences
are listed, with their review IDs, in the known-gap allowlist of
``tests/generation/test_vocabulary.py``.

Column helpers here (tender, decline and BLE draws) are the single
implementation both producers call, keyed on each producer's own IDs.
"""

from dataclasses import dataclass
from datetime import date

from pyspark.sql import Column
from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Payments
# ---------------------------------------------------------------------------

# In-store tenders: (method, mix weight, decline multiplier, processing_ms lo,
# processing_ms hi). Weights are relative.
TENDERS: list[tuple[str, float, float, int, int]] = [
    ("CREDIT_CARD", 0.4, 1.0, 1500, 4000),
    ("DEBIT_CARD", 0.3, 0.8, 1200, 3500),
    ("CASH", 0.2, 0.0, 500, 2000),
    ("MOBILE_PAY", 0.1, 1.2, 800, 2500),
]

# Online tenders (no cash): 60% credit / 25% debit / 10% PayPal / 5% other.
ONLINE_TENDERS: list[tuple[str, float, float, int, int]] = [
    ("CREDIT_CARD", 0.60, 1.0, 1500, 4000),
    ("DEBIT_CARD", 0.25, 0.8, 1200, 3500),
    ("PAYPAL", 0.10, 1.1, 2000, 5000),
    ("OTHER", 0.05, 0.5, 1500, 4000),
]

# Decline model: P(decline) = BASE_DECLINE x the tender's decline multiplier.
BASE_DECLINE = 0.025
DECLINE_REASONS = [
    "INSUFFICIENT_FUNDS",
    "CARD_EXPIRED",
    "INVALID_CVV",
    "NETWORK_ERROR",
    "FRAUD_SUSPECTED",
    "CARD_BLOCKED",
    "LIMIT_EXCEEDED",
]


def tender_names(tenders: list[tuple[str, float, float, int, int]] = TENDERS) -> list[str]:
    """Tender method names in table order."""
    return [name for name, *_ in tenders]


def decline_probability(
    method: str, tenders: list[tuple[str, float, float, int, int]] = TENDERS
) -> float:
    """Probability that a payment with ``method`` is declined."""
    for name, _, multiplier, _, _ in tenders:
        if name == method:
            return BASE_DECLINE * multiplier
    raise ValueError(f"unknown tender {method!r}")


def tender_column(
    draws: seeded_draws,
    keys: list,
    tenders: list[tuple[str, float, float, int, int]] = TENDERS,
    salt: str = "tender",
) -> Column:
    """Tender drawn by the mix weights of ``tenders``."""
    return draws.pick_by_weights(keys, salt, [(name, weight) for name, weight, *_ in tenders])


def decline_probability_column(
    method: Column, tenders: list[tuple[str, float, float, int, int]] = TENDERS
) -> Column:
    """Column form of :func:`decline_probability` (0 for an unknown method)."""
    return case_when(
        (method == name, F.lit(BASE_DECLINE * multiplier)) for name, _, multiplier, _, _ in tenders
    ).otherwise(F.lit(0.0))


def payment_status_column(
    draws: seeded_draws,
    keys: list,
    method: Column,
    tenders: list[tuple[str, float, float, int, int]] = TENDERS,
    salt: str = "decline",
) -> Column:
    """``DECLINED`` with the tender's decline probability, else ``APPROVED``."""
    declined = draws.u(keys, salt) < decline_probability_column(method, tenders)
    return F.when(declined, F.lit("DECLINED")).otherwise(F.lit("APPROVED"))


def decline_reason_column(
    draws: seeded_draws, keys: list, status: Column, salt: str = "reason"
) -> Column:
    """A decline reason for declined payments, NULL otherwise."""
    index = (draws.h64(keys, salt) % len(DECLINE_REASONS)).cast("int")
    return F.when(
        status == "DECLINED",
        F.element_at(F.array(*[F.lit(reason) for reason in DECLINE_REASONS]), index + 1),
    )


def processing_ms_column(
    draws: seeded_draws,
    keys: list,
    method: Column,
    tenders: list[tuple[str, float, float, int, int]] = TENDERS,
    salt: str = "proc",
) -> Column:
    """Payment processing time in ms, uniform in the tender's range."""
    low: Column = F.lit(tenders[-1][3])
    high: Column = F.lit(tenders[-1][4])
    for name, _, _, lo, hi in tenders[:-1]:
        low = F.when(method == name, F.lit(lo)).otherwise(low)
        high = F.when(method == name, F.lit(hi)).otherwise(high)
    return (low + draws.u(keys, salt) * (high - low)).cast("long")


# ---------------------------------------------------------------------------
# Promotions
# ---------------------------------------------------------------------------

# Named promo catalog. A purchase with an offer gets at most one code for the
# whole basket (``baskets.basket_promotion``): seasonal codes apply only in
# their eligible months, minimums apply to the basket gross subtotal, and each
# code discounts only the lines in its department scope (``baskets.PROMO_SCOPES``).
# BOGO discounts every second unit and needs a line with at least two units.
# (code, discount_pct, eligible_months | None, min_cents, kind)
PROMO_CATALOG: list[tuple[str, int, list[int] | None, int, str]] = [
    ("SAVE10", 10, None, 0, "PCT"),
    ("SAVE15", 15, None, 2500, "PCT"),  # min $25
    ("SAVE20", 20, None, 5000, "PCT"),  # min $50
    ("SAVE25", 25, None, 7500, "PCT"),  # min $75
    ("CLEARANCE30", 30, None, 0, "PCT"),
    ("BOGO50", 50, None, 0, "BOGO"),  # buy one get one 50% off (qty >= 2)
    ("NEWYEAR15", 15, [1, 2], 0, "PCT"),
    ("SPRINGSALE20", 20, [3, 4, 5], 0, "PCT"),
    ("SUMMER25", 25, [6, 7, 8], 0, "PCT"),
    ("BACKTOSCHOOL20", 20, [8, 9], 0, "PCT"),
    ("BFRIDAY30", 30, [11], 0, "PCT"),
    ("BFRIDAY40", 40, [11], 10000, "PCT"),  # min $100
    ("HOLIDAY20", 20, [12], 0, "PCT"),
]


def promo_eligible(index: int, month: int, basket_cents: int, max_quantity: int) -> bool:
    """Whether ``PROMO_CATALOG[index]`` may apply to a basket.

    The code must be in season, the basket gross subtotal must meet any
    minimum, and BOGO needs a line with at least two units. Department scope
    is checked per line by ``baskets.basket_promotion``.
    """
    _, _, months, min_cents, kind = PROMO_CATALOG[index]
    if months is not None and month not in months:
        return False
    if basket_cents < min_cents:
        return False
    return kind != "BOGO" or max_quantity >= 2


# ---------------------------------------------------------------------------
# Identifier formats
# ---------------------------------------------------------------------------

BLE_ID_PREFIX = "BLE"
BLE_ID_BASE = 36
BLE_ID_WIDTH = 6
AD_ID_PREFIX = "AD"
AD_ID_WIDTH = 8
BEACON_ID_FORMAT = "BEACON_%03d_%s"
SENSOR_ID_FORMAT = "SENSOR_%03d_%s"
CAMPAIGN_ID_PREFIX = "CAMP"
CAMPAIGN_INDEX_WIDTH = 2

_BASE36 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"


def ble_id(customer_id: int) -> str:
    """BLE device ID of a customer: ``BLE`` + upper-case base-36 ID, padded to 6."""
    if customer_id < 0:
        raise ValueError("customer_id must be non-negative")
    digits = ""
    value = customer_id
    while True:
        value, remainder = divmod(value, BLE_ID_BASE)
        digits = _BASE36[remainder] + digits
        if value == 0:
            break
    return BLE_ID_PREFIX + digits.rjust(BLE_ID_WIDTH, "0")


def ble_id_column(customer_id: Column) -> Column:
    """Column form of :func:`ble_id`."""
    digits = F.upper(F.conv(customer_id.cast("string"), 10, BLE_ID_BASE))
    return F.concat(F.lit(BLE_ID_PREFIX), minimum_width_text(digits, BLE_ID_WIDTH))


def ad_id(customer_id: int) -> str:
    """Advertising ID of a customer: ``AD`` + decimal ID, padded to 8 digits."""
    return AD_ID_PREFIX + str(customer_id).rjust(AD_ID_WIDTH, "0")


def ad_id_column(customer_id: Column) -> Column:
    """Column form of :func:`ad_id`."""
    return F.concat(F.lit(AD_ID_PREFIX), minimum_width_decimal(customer_id, AD_ID_WIDTH))


def beacon_id(store_id: int, zone: str) -> str:
    return BEACON_ID_FORMAT % (store_id, zone)


def beacon_id_column(store_id: Column, zone: Column) -> Column:
    return F.format_string(BEACON_ID_FORMAT, store_id, zone)


def sensor_id(store_id: int, zone: str) -> str:
    return SENSOR_ID_FORMAT % (store_id, zone)


def sensor_id_column(store_id: Column, zone: Column) -> Column:
    return F.format_string(SENSOR_ID_FORMAT, store_id, zone)


# Share of in-store visits whose phone is identified (its BLE ID is in
# dim_customers); the rest are anonymous.
BLE_KNOWN_SHARE = 0.3
ANONYMOUS_BLE_PREFIX = "ANON-"


def anonymous_ble_id_column(store_id: Column, draw: Column) -> Column:
    """Anonymous BLE device ID: ``ANON-<store>-<6 digits>`` from a non-negative hash."""
    return F.concat(
        F.lit(ANONYMOUS_BLE_PREFIX),
        store_id.cast("string"),
        F.lit("-"),
        (draw % F.lit(900000) + F.lit(100000)).cast("string"),
    )


def campaign_id(campaign_start: date, archetype_index: int) -> str:
    """Campaign ID: ``CAMP`` + window start (yyyyMMdd) + 2-digit archetype index."""
    if not 0 <= archetype_index < 10**CAMPAIGN_INDEX_WIDTH:
        raise ValueError("archetype_index exceeds 2 digits")
    return f"{CAMPAIGN_ID_PREFIX}{campaign_start:%Y%m%d}{archetype_index:02d}"


def campaign_id_column(campaign_start: Column, archetype_index: Column) -> Column:
    """Column form of :func:`campaign_id`."""
    return F.concat(
        F.lit(CAMPAIGN_ID_PREFIX),
        F.date_format(campaign_start, "yyyyMMdd"),
        fixed_width_decimal(archetype_index, CAMPAIGN_INDEX_WIDTH, "campaign archetype index"),
    )


# ---------------------------------------------------------------------------
# Zones
# ---------------------------------------------------------------------------

# BLE beacon zones a shopper's phone is seen in.
BLE_ZONES = ["ENTRANCE", "ELECTRONICS", "GROCERY", "CLOTHING", "CHECKOUT"]

# Generic zone dwell ranges (seconds), which the store types' zones borrow.
FOOT_TRAFFIC_ZONE_DWELL: dict[str, tuple[int, int]] = {
    "ENTRANCE_MAIN": (45, 90),
    "ENTRANCE_SIDE": (30, 75),
    "AISLES_A": (180, 420),
    "AISLES_B": (120, 300),
    "CHECKOUT": (90, 240),
}


# A shopper's path starts at the entrance, visits some department zones in a
# random order, and passes checkout when the visit buys (visit_zone_path). Each
# zone borrows the dwell range of the generic zone it overlaps.
BLE_ENTRY_ZONE = "ENTRANCE"
BLE_CHECKOUT_ZONE = "CHECKOUT"
BLE_ZONE_DWELL: dict[str, tuple[int, int]] = {
    "ENTRANCE": FOOT_TRAFFIC_ZONE_DWELL["ENTRANCE_MAIN"],
    "ELECTRONICS": FOOT_TRAFFIC_ZONE_DWELL["AISLES_B"],
    "GROCERY": FOOT_TRAFFIC_ZONE_DWELL["AISLES_A"],
    "CLOTHING": FOOT_TRAFFIC_ZONE_DWELL["AISLES_B"],
    "CHECKOUT": FOOT_TRAFFIC_ZONE_DWELL["CHECKOUT"],
}


def ble_zones_for_profile(profile: StoreTypeProfile | None) -> list[str]:
    """BLE zones for a store type: the profile's ``zones`` when it declares any.

    Profile zone names are normalized to upper case so they match the rest of
    the zone vocabulary. Without a profile the default ``BLE_ZONES`` apply.
    History (``sensors.py``) and the live stream both use these zones.
    """
    if profile is None or not profile.zones:
        return list(BLE_ZONES)
    return list(dict.fromkeys(zone.strip().upper() for zone in profile.zones if zone.strip()))


BLE_EXIT_ZONE = "EXIT"
# Dwell of zones outside BLE_ZONE_DWELL: exits pass like entrances, every other
# store-type zone (produce, tools, jewelry, ...) dwells like a secondary aisle.
BLE_EXIT_DWELL = FOOT_TRAFFIC_ZONE_DWELL["ENTRANCE_MAIN"]
BLE_DEPARTMENT_DWELL = FOOT_TRAFFIC_ZONE_DWELL["AISLES_B"]


@dataclass(frozen=True)
class BleLayout:
    """A store type's BLE zones by their role in a shopper's path."""

    entry: str
    middle: list[str]
    checkout: str | None
    exit: str | None
    dwell: dict[str, tuple[int, int]]

    @property
    def zones(self) -> list[str]:
        ordered = [self.entry, *self.middle, self.checkout, self.exit]
        return list(dict.fromkeys(zone for zone in ordered if zone is not None))


def ble_layout(profile: StoreTypeProfile | None) -> BleLayout:
    """Entry, department, checkout and exit zones with their dwell ranges.

    The entry is ``ENTRANCE`` (else the first zone), checkout and exit are the
    ``CHECKOUT`` and ``EXIT`` zones when the store type has them, and every
    other zone is a department zone a shopper may browse.
    """
    zones = ble_zones_for_profile(profile)
    entry = BLE_ENTRY_ZONE if BLE_ENTRY_ZONE in zones else zones[0]
    checkout = BLE_CHECKOUT_ZONE if BLE_CHECKOUT_ZONE in zones else None
    exit_zone = BLE_EXIT_ZONE if BLE_EXIT_ZONE in zones and BLE_EXIT_ZONE != entry else None
    middle = [zone for zone in zones if zone not in (entry, checkout, exit_zone)]
    if not middle:
        middle = [zone for zone in zones if zone != entry] or [entry]
    dwell = {
        zone: BLE_ZONE_DWELL.get(
            zone, BLE_EXIT_DWELL if zone == BLE_EXIT_ZONE else BLE_DEPARTMENT_DWELL
        )
        for zone in zones
    }
    return BleLayout(entry, middle, checkout, exit_zone, dwell)


# A shopper's visit (history and the live stream): the entrance, 1-3 of the
# store type's department zones in a random order, checkout when the visit
# buys, and the exit. Foot-traffic sensors count every zone a visitor passes;
# beacons detect the phones of BLE_DETECTED_SHARE of visits, 2-5 pings a zone.
BLE_MAX_MIDDLE_ZONES = 3
BLE_PINGS_PER_ZONE = (2, 5)
BLE_DETECTED_SHARE = 0.25
# Pings carry an RSSI uniform in [-80, -29] dBm.
RSSI_FLOOR = -80
RSSI_SPAN = 52


def zone_bound(zone: Column, dwell: dict[str, tuple[int, int]], index: int) -> Column:
    """Lower (``index`` 0) or upper (1) dwell seconds of ``zone``."""
    return case_when((zone == name, F.lit(bounds[index])) for name, bounds in dwell.items())


def visit_zone_path(draws: seeded_draws, key: list, converts: Column, layout: BleLayout) -> Column:
    """Ordered zones of a visit: entrance, browsed departments, checkout (buyers), exit.

    ``key`` holds the visit's key columns; every draw is keyed on it, so history
    and the stream lay out a visit the same way.
    """
    shuffled = F.array_sort(
        F.array(
            *[
                F.struct(
                    draws.u([*key, F.lit(zone)], "zone-order").alias("draw"),
                    F.lit(zone).alias("zone"),
                )
                for zone in layout.middle
            ]
        )
    )
    browse = min(BLE_MAX_MIDDLE_ZONES, len(layout.middle))
    middle_count = (F.floor(draws.u(key, "zone-count") * browse) + 1).cast("int")
    middle = F.transform(F.slice(shuffled, 1, middle_count), lambda entry: entry["zone"])
    no_zone = F.array().cast("array<string>")
    checkout = (
        F.when(converts, F.array(F.lit(layout.checkout))).otherwise(no_zone)
        if layout.checkout is not None
        else no_zone
    )
    exit_zone = F.array(F.lit(layout.exit)) if layout.exit is not None else no_zone
    return F.concat(F.array(F.lit(layout.entry)), middle, checkout, exit_zone)


def visit_zone_dwell(draws: seeded_draws, key: list, zones: Column, layout: BleLayout) -> Column:
    """Dwell seconds in each zone of a visit's path, uniform in the zone's range."""
    return F.transform(
        zones,
        lambda zone, i: (
            zone_bound(zone, layout.dwell, 0)
            + F.floor(
                draws.u([*key, i], "zone-dwell")
                * (zone_bound(zone, layout.dwell, 1) - zone_bound(zone, layout.dwell, 0) + 1)
            )
        ).cast("long"),
    )


def visit_zone_starts(dwell: Column) -> Column:
    """Seconds from the visit start at which each zone of the path is entered."""
    return F.transform(
        dwell,
        lambda _dwell, i: F.aggregate(
            F.slice(dwell, 1, i), F.lit(0).cast("long"), lambda acc, x: acc + x
        ),
    )


def visit_ping_counts(draws: seeded_draws, key: list, zones: Column) -> Column:
    """Beacon pings in each zone of a detected visit (2-5)."""
    low, high = BLE_PINGS_PER_ZONE
    return F.transform(
        zones,
        lambda _zone, i: (F.floor(draws.u([*key, i], "pings") * (high - low + 1)) + low).cast(
            "int"
        ),
    )


def checkout_index(zones: Column, layout: BleLayout) -> Column:
    """1-based position of the zone a buyer pays in: the last before any exit."""
    return F.size(zones) - (1 if layout.exit is not None else 0)


def zone_passage_shares(layout: BleLayout) -> dict[str, float]:
    """Share of visitors that pass each non-checkout zone of the layout.

    Every visitor passes the entrance and the exit; a department zone is one
    of 1-``BLE_MAX_MIDDLE_ZONES`` browsed zones. Checkout is passed by buyers.
    """
    browse = min(BLE_MAX_MIDDLE_ZONES, len(layout.middle))
    department = (browse + 1) / 2 / len(layout.middle)
    shares = {zone: department for zone in layout.middle}
    shares[layout.entry] = 1.0
    if layout.exit is not None:
        shares[layout.exit] = 1.0
    return shares


# ---------------------------------------------------------------------------
# Marketing
# ---------------------------------------------------------------------------

# (archetype, channels, base impressions/day at the legacy 86-store fleet,
#  campaign duration in days). The archetype index in campaign IDs is the
#  1-based position in this list.
CAMPAIGN_ARCHETYPES: list[tuple[str, list[str], int, int]] = [
    ("seasonal_sale", ["FACEBOOK", "GOOGLE", "EMAIL"], 1000, 7),
    ("product_launch", ["INSTAGRAM", "YOUTUBE", "DISPLAY"], 2000, 14),
    ("loyalty_program", ["EMAIL", "SOCIAL"], 500, 30),
    ("flash_sale", ["SOCIAL", "SEARCH"], 5000, 1),  # runs ~1 day in 7
]

# channel -> (lo, hi) cost-per-impression band in dollars (uniform draw)
CHANNEL_COSTS: dict[str, tuple[float, float]] = {
    "EMAIL": (0.005, 0.05),
    "DISPLAY": (0.10, 0.50),
    "SOCIAL": (0.08, 0.25),
    "SEARCH": (0.15, 0.75),
    "FACEBOOK": (0.10, 0.40),
    "GOOGLE": (0.25, 1.50),
    "INSTAGRAM": (0.08, 0.35),
    "YOUTUBE": (0.30, 1.50),
}
MARKETING_CHANNELS = list(CHANNEL_COSTS)

# Device types are upper case; cost multipliers scale the channel band.
DEVICE_WEIGHTS: list[tuple[str, float]] = [("MOBILE", 0.6), ("DESKTOP", 0.3), ("TABLET", 0.1)]
DEVICE_MULTIPLIERS: dict[str, float] = {"MOBILE": 1.2, "DESKTOP": 0.8, "TABLET": 0.9}
DEVICE_TYPES = [device for device, _ in DEVICE_WEIGHTS]

# ---------------------------------------------------------------------------
# Online fulfillment
# ---------------------------------------------------------------------------

ONLINE_FULFILLMENT_MIX: list[tuple[str, float]] = [
    ("SHIP_FROM_DC", 0.60),
    ("SHIP_FROM_STORE", 0.30),
    ("BOPIS", 0.10),
]
FULFILLMENT_MODES = [mode for mode, _ in ONLINE_FULFILLMENT_MIX]

# Order-level outcomes and stage delays shared by history (online_orders.py)
# and the live stream. Cancelled orders have zero totals, no payment and no
# fulfillment; backordered (never BOPIS) orders ship but are not delivered yet.
ONLINE_CANCEL_RATE = 0.02
ONLINE_BACKORDER_RATE = 0.08
# Promo codes with their percentage discount; each order puts a promo on a
# share of its lines drawn uniformly from ONLINE_PROMO_LINE_SHARE.
ONLINE_PROMOS: list[tuple[str, int]] = [("PROMO05", 5), ("PROMO10", 10), ("PROMO20", 20)]
ONLINE_PROMO_LINE_SHARE = (0.10, 0.30)
# Stage delays in seconds, uniform in [low, high): picking after the order
# (BOPIS orders are picked for the pickup counter, ship orders for the
# carrier), shipping after picking, delivery after shipping, and the BOPIS
# customer pickup after the order is ready.
ONLINE_PICK_SECONDS = {"BOPIS": (4 * 3600, 24 * 3600), "SHIP": (30 * 60, 240 * 60)}
ONLINE_SHIP_SECONDS = (2 * 3600, 4 * 3600)
ONLINE_DELIVER_SECONDS = (86400, 3 * 86400)
ONLINE_PICKUP_SECONDS = (3600, 24 * 3600)
# Relative weight of online orders in each UTC hour (0-23). Most customers
# are in US time zones, so orders peak in the US evening (00-03 UTC) and are
# fewest in the US night (07-10 UTC).
ONLINE_HOURLY_WEIGHTS = [
    6.5, 6.5, 6.0, 5.0, 3.8, 2.6, 1.6, 1.0, 0.8, 0.9, 1.3, 2.0,
    2.9, 3.6, 4.1, 4.4, 4.6, 4.7, 4.7, 4.8, 5.0, 5.4, 5.9, 6.3,
]  # fmt: skip


"""Calibrated thresholds shared by the generators and their consumers.

Rules (``fabric/rules/definitions.kql``), KQL functions, the semantic model and
the live stream keep literal values in their own languages. Each literal must
equal the constant here; ``tests/generation/test_calibration.py`` parses those
sources and fails when one drifts.

The stream generator parameters below sit next to the thresholds they are
calibrated against, so a normal truck stays under the dwell SLA and a late
truck exceeds it.
"""


# ---------------------------------------------------------------------------
# Truck dwell (minutes from arrival to departure)
# ---------------------------------------------------------------------------

# One SLA for every consumer: the dwell-breach rule and the semantic model's
# breach-day measure. A typical consolidated load dwells 30-75 minutes, so a
# site-day averaging above 90 is a real outlier.
TRUCK_DWELL_SLA_MINUTES = 90
# Unload model shared by history and the live stream (network.unload_minutes):
# a truck dwells the base minutes plus its load at the site's unload rate
# (units per minute times a per-store dock factor), plus a little noise, never
# less than the minimum and never more than the maximum. A typical
# consolidated store load stays inside the normal range, under the SLA; only an
# unusually large load (a store served late after a backlog) breaches it.
TRUCK_NORMAL_DWELL_MINUTES_MIN = 30
TRUCK_NORMAL_DWELL_MINUTES_MAX = 75
TRUCK_UNLOAD_BASE_MINUTES = 25
TRUCK_UNLOAD_UNITS_PER_MINUTE = 120
TRUCK_UNLOAD_NOISE_MINUTES = 6
TRUCK_DOCK_FACTOR_RANGE = (0.8, 1.2)
TRUCK_UNLOAD_MINUTES_MAX = 180
# History: unloading (and the store's stock receipt) starts this long after
# the truck arrives.
TRUCK_CHECKIN_MINUTES = 15
# Rule evaluation window for completed lifecycles.
TRUCK_DWELL_RULE_WINDOW_MINUTES = 5

# ---------------------------------------------------------------------------
# Inventory
# ---------------------------------------------------------------------------

# A stockout is reported when a store's on-hand balance drops below this
# quantity, that is when a sale takes it to zero: the crossing history's
# fact_stockouts records. Reorder points and priorities come from the shared
# replenishment policy (inventory_policy.reorder_levels / reorder_priority).
STOCKOUT_QUANTITY_THRESHOLD = 1
# Rule evaluation windows.
STOCKOUT_RULE_WINDOW_MINUTES = 5
URGENT_REORDER_RULE_WINDOW_MINUTES = 10

# ---------------------------------------------------------------------------
# Truck logistics (history and stream_supply)
# ---------------------------------------------------------------------------

# Reorders wait at the DC for the store's daily delivery slot, so one truck
# carries a store's consolidated load; then the next free truck of the DC's
# fleet loads for TRUCK_LOAD_MINUTES, drives the lane (one-way minutes, a stable
# draw per DC and store) and returns.
TRUCK_LOAD_MINUTES = 45
TRUCK_TRANSIT_MINUTES_MIN = 60
TRUCK_TRANSIT_MINUTES_MAX = 240
# Delivery routes (network.plan_routes): each DC truck serves a fixed set of
# stores, one delivery a day each, at slots spaced by the longest possible
# trip (load, both legs of the lane, the maximum unload) plus this turnaround,
# which also absorbs the hour lost when clocks spring forward.
TRUCK_TURNAROUND_MINUTES = 60

# ---------------------------------------------------------------------------
# Zone arrivals
# ---------------------------------------------------------------------------

# Arrival-surge rule: per (store, zone) over the window, alert when arrivals
# exceed the surge count or the arrival-weighted expected dwell exceeds the
# long-dwell seconds.
ZONE_SURGE_WINDOW_MINUTES = 5
ZONE_SURGE_ARRIVALS = 50
ZONE_LONG_DWELL_SECONDS = 600

# ---------------------------------------------------------------------------
# High-value receipts
# ---------------------------------------------------------------------------

# A receipt is high value when its total exceeds this multiple of the store
# type's calibrated average ticket (profile.avg_ticket_target).
HIGH_VALUE_TICKET_MULTIPLE = 3.0
HIGH_VALUE_RULE_WINDOW_MINUTES = 5
# Store type whose threshold the static reference rule carries.
REFERENCE_STORE_TYPE = "supercenter"


def high_value_receipt_threshold(profile: StoreTypeProfile) -> float:
    """Receipt total (dollars) above which a receipt counts as high value."""
    return round(HIGH_VALUE_TICKET_MULTIPLE * profile.avg_ticket_target, 2)


# ---------------------------------------------------------------------------
# Anomaly detection (series_decompose_anomalies)
# ---------------------------------------------------------------------------

ANOMALY_LOOKBACK_HOURS = 168
ANOMALY_THRESHOLD = 1.5
ANOMALY_SEASONALITY = -1  # autodetect
ANOMALY_TREND = "avg"
# Severity by |anomaly_score|, highest band first; below the last band is "low".
ANOMALY_SEVERITY_BANDS: list[tuple[str, float]] = [
    ("critical", 3.0),
    ("high", 2.0),
    ("medium", 1.5),
]


def anomaly_severity(score: float) -> str:
    """Severity label the anomaly functions assign to an anomaly score."""
    for label, bound in ANOMALY_SEVERITY_BANDS:
        if abs(score) >= bound:
            return label
    return "low"


"""Store calendars and live-stream volume calibration.

Historical traffic (``traffic.py``) draws visitors per store, local day and
local hour from the profile's hourly, daily and monthly weights, the store's
``daily_traffic_multiplier`` and its operating hours. The live stream uses the
same weights, evaluated in each store's own time zone, so live volume and its
time-of-day curve continue the history.

The stream is driven by a ``rate`` source. Each source row is assigned to one
lane (store visits, online orders, marketing) and is
kept with an emit probability, so the kept rows form a Poisson process with
the calibrated rate (thinning). :func:`plan_lanes` sizes the rate and the
lane shares so the emit probability never needs to exceed 1 at peak.
Inventory and logistics have no lane: they follow from the emitted purchases
(``stream_supply``).

Pure Python plus Column helpers; ``build_notebooks.py`` inlines this module
into the stream notebook.
"""

from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from math import ceil
from zoneinfo import ZoneInfo

from pyspark.sql import Column
from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Operating hours
# ---------------------------------------------------------------------------

# operating_hours literal -> [open_hour, close_hour); "24h" maps to 0/24. Store
# types may declare other "open-close" patterns (StoreTypeProfile); these are
# the historical defaults.
OPERATING_HOURS_MAP: dict[str, tuple[int, int]] = {
    "6-22": (6, 22),
    "7-22": (7, 22),
    "7-23": (7, 23),
    "24h": (0, 24),
}
# Unknown formats fall back to standard hours.
DEFAULT_OPERATING_WINDOW = (6, 22)
# Stores are closed on Christmas Day, any year.
CLOSED_MONTH_DAYS = frozenset({(12, 25)})
# A 24-hour store "closes" at 23:59 and reopens at midnight, like history.
ALL_DAY_CLOSE_SECONDS = 23 * 3600 + 59 * 60


def operating_window(operating_hours: str | None) -> tuple[int, int]:
    """``[open_hour, close_hour)`` of an ``operating_hours`` value."""
    if operating_hours is None:
        return DEFAULT_OPERATING_WINDOW
    try:
        return parse_operating_hours(operating_hours)
    except ValueError:
        return DEFAULT_OPERATING_WINDOW


def is_closed_date(day: date) -> bool:
    """True on dates every store is closed."""
    return (day.month, day.day) in CLOSED_MONTH_DAYS


def closed_date_column(local_day: Column) -> Column:
    """Column form of :func:`is_closed_date`."""
    condition: Column = F.lit(False)
    for month, day in sorted(CLOSED_MONTH_DAYS):
        condition = condition | ((F.month(local_day) == month) & (F.dayofmonth(local_day) == day))
    return condition


def _close_seconds(close_hour: int) -> int:
    return ALL_DAY_CLOSE_SECONDS if close_hour == 24 else close_hour * 3600


# ---------------------------------------------------------------------------
# Traffic weights (the historical model in traffic.py)
# ---------------------------------------------------------------------------

# traffic.py draws visitors at ``transactions_per_store_day / 0.225`` and
# converts about this share of them into receipts.
VISIT_CONVERSION_CALIBRATION = 0.225
# Visit conversion: logistic in a store effect and a store/day/hour effect.
CONVERSION_LOGIT_INTERCEPT = 1.2657
CONVERSION_STORE_SD = 0.25
CONVERSION_HOUR_SD = 0.20


def hour_shares(hourly_weights: list[float], operating_hours: str | None) -> list[float]:
    """Share of a store-day's visits in each local hour (0 while closed)."""
    open_hour, close_hour = operating_window(operating_hours)
    masked = [w if open_hour <= h < close_hour else 0.0 for h, w in enumerate(hourly_weights)]
    total = sum(masked)
    return [w / total if total > 0 else 0.0 for w in masked]


def daily_factors(profile: StoreTypeProfile) -> list[float]:
    """Day-of-week multipliers, Monday first, averaging 1."""
    return [w * 7 / sum(profile.daily_weights) for w in profile.daily_weights]


def monthly_factors(profile: StoreTypeProfile) -> list[float]:
    """Month multipliers, January first, averaging 1."""
    return [w * 12 / sum(profile.monthly_weights) for w in profile.monthly_weights]


def online_hour_factors() -> list[float]:
    """Online-order multiplier of each UTC hour, averaging 1."""
    total = sum(ONLINE_HOURLY_WEIGHTS)
    return [w * 24 / total for w in ONLINE_HOURLY_WEIGHTS]


def online_hour_factor_column(ts: Column) -> Column:
    """Column form of :func:`online_hour_factors` for a UTC timestamp column."""
    return F.element_at(F.array(*[F.lit(w) for w in online_hour_factors()]), F.hour(ts) + 1)


def daily_factor_column(profile: StoreTypeProfile, day: Column) -> Column:
    """Column form of :func:`daily_factors` for a date column."""
    return F.element_at(
        F.array(*[F.lit(w) for w in daily_factors(profile)]),
        F.pmod(F.dayofweek(day) + 5, F.lit(7)) + 1,
    )


def monthly_factor_column(profile: StoreTypeProfile, day: Column) -> Column:
    """Column form of :func:`monthly_factors` for a date column."""
    return F.element_at(F.array(*[F.lit(w) for w in monthly_factors(profile)]), F.month(day))


def visit_conversion(draws: seeded_draws, store_id: Column, day: Column, hour: Column) -> Column:
    """Probability that a visit in this store, local day and local hour buys."""
    return 1.0 / (
        1.0
        + F.exp(
            F.lit(CONVERSION_LOGIT_INTERCEPT)
            - CONVERSION_STORE_SD * draws.gauss([store_id], "conversion-store")
            - CONVERSION_HOUR_SD * draws.gauss([store_id, day, hour], "conversion-hour")
        )
    )


def store_visits_per_hour(
    profile: StoreTypeProfile,
    transactions_per_store_day: float,
    multiplier: float,
    operating_hours: str | None,
    local: datetime,
) -> float:
    """Expected visits in the local hour starting at ``local`` (weather excluded)."""
    if is_closed_date(local.date()):
        return 0.0
    return (
        transactions_per_store_day
        / VISIT_CONVERSION_CALIBRATION
        * daily_factors(profile)[local.weekday()]
        * monthly_factors(profile)[local.month - 1]
        * multiplier
        * hour_shares(profile.hourly_weights, operating_hours)[local.hour]
    )


# ---------------------------------------------------------------------------
# Store open/close boundaries
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class StoreHours:
    store_id: int
    operating_hours: str
    time_zone: str


def store_boundaries(
    store: StoreHours, after: datetime, until: datetime
) -> list[tuple[datetime, str]]:
    """Open and close instants (UTC) of ``store`` in ``(after, until]``, in time order.

    Hours are local wall-clock times, so the UTC instants move with daylight
    saving time. Every open day has exactly one ``opened`` and one ``closed``.
    """
    if after.tzinfo is None or until.tzinfo is None:
        raise ValueError("boundary window must be timezone-aware")
    zone = ZoneInfo(store.time_zone)
    open_hour, close_hour = operating_window(store.operating_hours)
    day = after.astimezone(zone).date() - timedelta(days=1)
    last = until.astimezone(zone).date()
    found: list[tuple[datetime, str]] = []
    while day <= last:
        if not is_closed_date(day):
            for seconds, operation in (
                (open_hour * 3600, "opened"),
                (_close_seconds(close_hour), "closed"),
            ):
                # Wall-clock arithmetic: add the offset to local midnight, then
                # normalize through UTC so DST days keep the local hour.
                local = datetime(day.year, day.month, day.day) + timedelta(seconds=seconds)
                instant = local.replace(tzinfo=zone).astimezone(timezone.utc)
                if after < instant <= until:
                    found.append((instant, operation))
        day += timedelta(days=1)
    return sorted(found)


def is_open_at(store: StoreHours, instant: datetime) -> bool:
    """True when ``store`` is open at the UTC ``instant``."""
    local = instant.astimezone(ZoneInfo(store.time_zone))
    if is_closed_date(local.date()):
        return False
    open_hour, close_hour = operating_window(store.operating_hours)
    second_of_day = local.hour * 3600 + local.minute * 60 + local.second
    return open_hour * 3600 <= second_of_day < _close_seconds(close_hour)


# ---------------------------------------------------------------------------
# Lane plan: rate-source sizing and per-lane emit probabilities
# ---------------------------------------------------------------------------

# Marketing volumes in vocabulary.CAMPAIGN_ARCHETYPES are per day at this
# legacy fleet size (marketing.py scales them by store_count / 86).
MARKETING_LEGACY_FLEET_SIZE = 86

STREAM_LANES = ("visits", "online", "marketing")


@dataclass(frozen=True)
class LanePlan:
    """How rate-source rows are split into lanes and thinned.

    ``shares[lane]`` is the fraction of source rows assigned to a lane (the
    rest are idle). A lane row is kept with probability
    ``min(1, factors[lane] * time_factor)``, where the time factor is the
    lane's own calendar weight at the row's time (1 for flat lanes).
    """

    rows_per_second: float
    shares: dict[str, float]
    factors: dict[str, float]
    peaks: dict[str, float]
    clipped: bool


def _visit_peak_factor(profile: StoreTypeProfile, operating_hours: list[str]) -> float:
    patterns = sorted(set(operating_hours)) or ["24h"]
    peak_share = max(max(hour_shares(profile.hourly_weights, hours)) for hours in patterns)
    return max(daily_factors(profile)) * max(monthly_factors(profile)) * peak_share


def plan_lanes(
    profile: StoreTypeProfile,
    *,
    store_count: int,
    multiplier_total: float,
    operating_hours: list[str],
    transactions_per_store_day: float,
    online_orders_per_day: float,
    marketing_impressions_per_day: float,
    volume_scale: float = 1.0,
    rows_per_second: float | None = None,
    growth: float = 1.0,
) -> LanePlan:
    """Size the rate source and the lanes from the historical configuration.

    ``multiplier_total`` is the sum of the stores' ``daily_traffic_multiplier``
    and ``marketing_impressions_per_day`` the legacy-fleet total of every
    campaign archetype. The default rate is the smallest whole number of rows
    per second that covers the peak of every lane; an explicit
    ``rows_per_second`` only changes the resolution, not the volume, unless it
    is below the peak (``clipped``). ``growth`` is the volume trend multiplier
    (``pricing.volume_growth``) of the stream's dates; the lanes apply it per
    row, so it only sizes the peaks.
    """
    if volume_scale <= 0:
        raise ValueError("volume_scale must be positive")
    visit_peak_factor = _visit_peak_factor(profile, operating_hours)
    visits_per_second = (
        transactions_per_store_day / VISIT_CONVERSION_CALIBRATION / 3600.0 * multiplier_total
    )
    online_per_second = online_orders_per_day / 86400.0
    marketing_per_second = (
        store_count / MARKETING_LEGACY_FLEET_SIZE * marketing_impressions_per_day / 86400.0
    )
    # Mean events per second of each lane at time factor 1, and its peak factor.
    base = {
        "visits": (visits_per_second, visit_peak_factor * growth),
        "online": (
            online_per_second,
            max(monthly_factors(profile)) * max(online_hour_factors()) * growth,
        ),
        "marketing": (marketing_per_second, 1.0),
    }
    peaks = {lane: rate * peak * volume_scale for lane, (rate, peak) in base.items()}
    total_peak = sum(peaks.values())
    rate = float(rows_per_second) if rows_per_second else float(max(1, ceil(total_peak)))
    if rate <= 0:
        raise ValueError("rows_per_second must be positive")
    capacity = max(rate, total_peak)
    shares = {lane: peaks[lane] / capacity for lane in STREAM_LANES}
    factors = {
        lane: (base[lane][0] * volume_scale / (rate * shares[lane]) if shares[lane] > 0 else 0.0)
        for lane in STREAM_LANES
    }
    return LanePlan(rate, shares, factors, peaks, rate < total_peak)


"""Shelf prices over time and the demand response to them.

Each SKU's shelf price is a pure function of the seed, the SKU and the week,
shared by history (receipts, online orders, the published price history) and
the live stream:

- **Trend.** Regular prices follow a per-department inflation trend
  (``DEPARTMENT_PRICING``), anchored on ``dim_products.SalePrice`` at the
  configuration's anchor date.
- **Price reviews.** Every ``PRICE_REVIEW_WEEKS`` weeks (each SKU on its own
  phase) the regular price is reset within ``REGULAR_PRICE_SPREAD`` of trend.
- **Markdowns.** ``MARKDOWN_PROBABILITY`` of review periods carry a one-to-three
  week markdown of 10-35%.

Volume also trends: store traffic and online orders compound
``annual_volume_growth`` a year from the anchor date (:func:`volume_growth`).

Demand responds to price: within a department, a SKU's choice weight scales
with ``(price / trend price) ** elasticity``, with department elasticities
between about -0.8 (staples) and -2.5 (electronics). Markdowns and expensive
review periods therefore move units, which is what elasticity models measure.

Weeks count from ``PRICE_EPOCH`` (a Monday), not from the history window, so
the same SKU has the same price on the same day in every run.
"""

from dataclasses import dataclass
from datetime import date

from pyspark.sql import Column
from pyspark.sql import functions as F


PRICE_EPOCH = date(2000, 1, 3)
PRICE_REVIEW_WEEKS = 13
REGULAR_PRICE_SPREAD = 0.06
MARKDOWN_PROBABILITY = 0.3
MARKDOWN_WEEKS = (1, 3)
MARKDOWN_DEPTH = (0.10, 0.35)
DAYS_PER_YEAR = 365.25

# (department-name token, annual inflation, price elasticity of choice). The
# first token found in a lower-cased department name applies.
DEPARTMENT_PRICING: list[tuple[str, float, float]] = [
    ("electronic", -0.03, -2.5),
    ("jewel", 0.04, -0.8),
    ("watch", 0.04, -0.8),
    ("toy", 0.02, -2.0),
    ("seasonal", 0.02, -2.2),
    ("footwear", 0.02, -1.8),
    ("apparel", 0.02, -1.8),
    ("clothing", 0.02, -1.8),
    ("accessor", 0.03, -1.4),
    ("sport", 0.02, -1.8),
    ("garden", 0.03, -1.6),
    ("home", 0.025, -1.6),
    ("automotive", 0.03, -1.2),
    ("office", 0.02, -1.4),
    ("tool", 0.035, -1.2),
    ("building", 0.04, -1.0),
    ("electrical", 0.035, -1.1),
    ("plumbing", 0.035, -1.1),
    ("paint", 0.035, -1.2),
    ("hardware", 0.035, -1.2),
    ("pet", 0.03, -1.0),
    ("baby", 0.03, -0.9),
    ("health", 0.03, -1.1),
    ("beauty", 0.03, -1.3),
    ("household", 0.03, -1.0),
    ("fresh", 0.035, -0.9),
    ("grocery", 0.03, -0.8),
]
DEFAULT_INFLATION = 0.025
DEFAULT_ELASTICITY = -1.5


def department_pricing(department: str) -> tuple[float, float]:
    """(annual inflation, elasticity) of a department name."""
    name = department.lower()
    for token, inflation, elasticity in DEPARTMENT_PRICING:
        if token in name:
            return inflation, elasticity
    return DEFAULT_INFLATION, DEFAULT_ELASTICITY


@dataclass(frozen=True)
class PriceSchedule:
    """Price schedule anchored on ``dim_products.SalePrice`` at ``anchor_date``."""

    seed: int
    anchor_date: date


@dataclass(frozen=True)
class ShelfPrice:
    """A SKU's prices on one day (pure-Python form of :func:`shelf_price_struct`)."""

    regular_cents: int
    price_cents: int
    trend_cents: float
    markdown: bool


def week_index(day: date) -> int:
    return (day - PRICE_EPOCH).days // 7


def markdown_window(
    review_draw: float, length_draw: float, start_draw: float
) -> tuple[int, int] | None:
    """``[first, last)`` weeks into a review period of its markdown, or None."""
    if review_draw >= MARKDOWN_PROBABILITY:
        return None
    low, high = MARKDOWN_WEEKS
    length = low + int(length_draw * (high - low + 1))
    first = int(start_draw * (PRICE_REVIEW_WEEKS - length + 1))
    return first, first + length


def shelf_price(
    base_cents: int,
    inflation: float,
    anchor: date,
    day: date,
    phase: int,
    review_draw: float,
    markdown_draws: tuple[float, float, float, float],
) -> ShelfPrice:
    """A SKU's regular and shelf price on ``day`` given its draws.

    ``phase`` offsets the SKU's review calendar; ``review_draw`` sets the
    period's regular price within the spread; ``markdown_draws`` are (has
    markdown, length, start, depth).
    """
    week = week_index(day)
    period = (week + phase) // PRICE_REVIEW_WEEKS
    period_start = period * PRICE_REVIEW_WEEKS - phase
    years = (period_start * 7 + (PRICE_EPOCH - anchor).days) / DAYS_PER_YEAR
    trend_years = (day - anchor).days / DAYS_PER_YEAR
    regular = (
        base_cents * (1 + inflation) ** years * (1 + REGULAR_PRICE_SPREAD * (2 * review_draw - 1))
    )
    window = markdown_window(*markdown_draws[:3])
    marked = window is not None and window[0] <= week - period_start < window[1]
    low, high = MARKDOWN_DEPTH
    depth = low + markdown_draws[3] * (high - low) if marked else 0.0
    regular_cents = max(1, int(regular + 0.5))
    return ShelfPrice(
        regular_cents=regular_cents,
        price_cents=max(1, int(regular * (1 - depth) + 0.5)),
        trend_cents=base_cents * (1 + inflation) ** trend_years,
        markdown=marked,
    )


def _department_column(department: Column, index: int, default: float) -> Column:
    branches = [
        (F.lower(department).contains(token), F.lit(values[index]))
        for token, *values in DEPARTMENT_PRICING
    ]
    return case_when(branches).otherwise(F.lit(default))


def inflation_column(department: Column) -> Column:
    return _department_column(department, 0, DEFAULT_INFLATION)


def elasticity_column(department: Column) -> Column:
    return _department_column(department, 1, DEFAULT_ELASTICITY)


def shelf_price_struct(
    schedule: PriceSchedule,
    product_id: Column,
    base_price: Column,
    inflation: Column,
    day: Column,
) -> Column:
    """``struct<regular_cents long, price_cents long, trend_cents double, markdown boolean>``.

    Column form of :func:`shelf_price` for a SKU (``base_price`` in dollars,
    ``dim_products.SalePrice``; ``inflation`` its department's yearly trend,
    :func:`inflation_column`) on a date column.
    """
    draws = seeded_draws(schedule.seed)
    epoch_offset = (PRICE_EPOCH - schedule.anchor_date).days
    base_cents = F.round(base_price * 100)
    growth = F.lit(1.0) + inflation
    week = F.floor(F.datediff(day, F.lit(PRICE_EPOCH)) / 7)
    phase = draws.h64([product_id], "price-phase") % PRICE_REVIEW_WEEKS
    calendar = F.struct(
        week.alias("week"),
        F.floor((week + phase) / PRICE_REVIEW_WEEKS).alias("period"),
        phase.alias("phase"),
    )

    def in_period(c: Column) -> Column:
        period_start = c["period"] * PRICE_REVIEW_WEEKS - c["phase"]
        keys = [product_id, c["period"]]
        period_draws = F.struct(
            draws.u(keys, "price-review").alias("review"),
            draws.u(keys, "markdown").alias("markdown"),
            draws.u(keys, "markdown-length").alias("length"),
            draws.u(keys, "markdown-start").alias("start"),
            draws.u(keys, "markdown-depth").alias("depth"),
        )

        def price(r: Column) -> Column:
            years = (period_start * 7 + F.lit(epoch_offset)) / DAYS_PER_YEAR
            low, high = MARKDOWN_WEEKS
            length = low + F.floor(r["length"] * (high - low + 1))
            first = F.floor(r["start"] * (PRICE_REVIEW_WEEKS - length + 1))
            into = c["week"] - period_start
            marked = (
                (r["markdown"] < F.lit(MARKDOWN_PROBABILITY))
                & (into >= first)
                & (into < first + length)
            )
            depth_low, depth_high = MARKDOWN_DEPTH
            return F.struct(
                (
                    base_cents
                    * F.pow(growth, years)
                    * (1 + REGULAR_PRICE_SPREAD * (2 * r["review"] - 1))
                ).alias("regular"),
                F.when(marked, F.lit(depth_low) + r["depth"] * (depth_high - depth_low))
                .otherwise(F.lit(0.0))
                .alias("depth"),
                marked.alias("markdown"),
            )

        return bind(period_draws, price)

    def shelf(r: Column) -> Column:
        trend_years = F.datediff(day, F.lit(schedule.anchor_date)) / DAYS_PER_YEAR
        return F.struct(
            F.greatest(F.lit(1), F.floor(r["regular"] + 0.5)).cast("long").alias("regular_cents"),
            F.greatest(F.lit(1), F.floor(r["regular"] * (1 - r["depth"]) + 0.5))
            .cast("long")
            .alias("price_cents"),
            (base_cents * F.pow(growth, trend_years)).alias("trend_cents"),
            r["markdown"].alias("markdown"),
        )

    # Each step binds its inputs once, which keeps the expression small.
    return bind(bind(calendar, in_period), shelf)


def demand_response(price_cents: Column, trend_cents: Column, elasticity: Column) -> Column:
    """Choice-weight multiplier of a SKU priced off its trend: ``(price / trend) ** elasticity``."""
    return F.pow(price_cents / trend_cents, elasticity)


def volume_growth(annual_growth: float, anchor: date, day: date) -> float:
    """Year-over-year volume trend: 1 at ``anchor``, compounding ``annual_growth`` a year."""
    return (1.0 + annual_growth) ** ((day - anchor).days / DAYS_PER_YEAR)


def volume_growth_column(annual_growth: float, anchor: date, day: Column) -> Column:
    """Column form of :func:`volume_growth` for a date column."""
    return F.pow(F.lit(1.0 + annual_growth), F.datediff(day, F.lit(anchor)) / DAYS_PER_YEAR)


"""Shared historical/live shopping behavior, expressed as deterministic Spark columns."""

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window



def repeat_purchase_probability(department: Column) -> Column:
    """Consumables permit multi-unit purchases more often than durable goods."""
    consumable = F.lower(department).rlike(
        "grocery|fresh|food|household|personal|health|beauty|pet|baby"
    )
    return F.when(consumable, F.lit(0.22)).otherwise(F.lit(0.04))


def retail_quantity(draws: seeded_draws, keys: list, department: Column) -> Column:
    return (
        F.lit(1)
        + F.floor(
            F.log(F.lit(1.0) - draws.u(keys, "retail-quantity"))
            / F.log(repeat_purchase_probability(department))
        )
    ).cast("int")


def retail_basket_size(
    draws: seeded_draws,
    purchase_keys: list,
    shopper: Column,
    mean: Column,
    offer: Column,
    promo_rate: float,
) -> Column:
    """Mission mixtures with a calibrated mean and persistent customer budgets.

    ``shopper`` keys the budget: the customer (``customers.shopper_key``), so
    a customer's baskets share one budget, or the purchase when anonymous.
    """
    trip = draws.u(purchase_keys, "retail-mission")
    mission = (
        F.when(trip < 0.30, 0.30).when(trip < 0.70, 0.90).when(trip < 0.90, 1.60).otherwise(2.50)
    ) / F.lit(1.02)
    budget = F.exp(0.35 * draws.gauss([shopper], "retail-budget") - F.lit(0.35**2 / 2))
    # Exposure is decided before demand: coupons increase additional-item demand.
    lift = (F.lit(1.0) + offer.cast("double") * 0.20) / (1.0 + promo_rate * 0.20)
    extra = F.greatest(mean - 1.0, F.lit(0.0)) * mission * budget * lift
    return (1 + draws.negative_binomial(purchase_keys, "retail-basket", extra, 3)).cast("int")


def retail_catalog(
    products: DataFrame, dates: DataFrame, draws: seeded_draws, schedule: PriceSchedule
) -> DataFrame:
    """One launch-eligible weighted product CDF per day and department.

    Popularity is persistent per SKU, with cross-sectional price sensitivity,
    a long tail, and a demand response to the SKU's shelf price that day
    (markdowns and price reviews, ``pricing.demand_response``). Each entry
    carries the day's shelf price (``unit_cents``) and the SKU's complementary
    partner priced the same day (``baskets.basket_products``).
    Arrays keep selection a bounded lookup rather than a receipt x catalog join.
    """
    eligible = (
        dates.select("event_date")
        .distinct()
        .join(F.broadcast(products), F.col("launch_date") <= F.col("event_date"))
    )
    day = F.col("event_date")
    eligible = eligible.withColumn(
        "_price",
        shelf_price_struct(
            schedule, F.col("product_id"), F.col("SalePrice"), F.col("inflation"), day
        ),
    ).withColumn(
        "_partner_price",
        shelf_price_struct(
            schedule, F.col("partner_id"), F.col("partner_price"), F.col("inflation"), day
        ),
    )
    eligible = eligible.withColumn(
        "_popularity",
        F.exp(0.8 * draws.gauss(["product_id"], "retail-popularity") - 0.32)
        / F.pow(F.col("SalePrice"), 1.15)
        * demand_response(
            F.col("_price.price_cents"), F.col("_price.trend_cents"), F.col("elasticity")
        ),
    ).withColumn("unit_cents", F.col("_price.price_cents"))
    # Floating-point sums run in product_id order over the whole department, so
    # they are bit-identical whatever order the shuffle delivers rows in.
    by_product = Window.partitionBy("event_date", "department").orderBy("product_id")
    ordered = by_product.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    whole = by_product.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
    taxable_share = (
        F.when(F.col("taxability") == "TAXABLE", 1.0)
        .when(F.col("taxability") == "REDUCED_RATE", 0.5)
        .otherwise(0.0)
    )
    eligible = (
        eligible.withColumn("_total_weight", F.sum("_popularity").over(whole))
        .withColumn("_probability", F.col("_popularity") / F.col("_total_weight"))
        .withColumn("_cdf", F.sum("_probability").over(ordered))
        .withColumn(
            "_unit_value",
            F.col("unit_cents") / 100.0 / (1.0 - repeat_purchase_probability(F.col("department"))),
        )
        .withColumn(
            "_expected_line_value",
            F.sum(F.col("_probability") * F.col("_unit_value")).over(whole),
        )
        .withColumn(
            "_expected_taxable_value",
            F.sum(F.col("_probability") * F.col("_unit_value") * taxable_share).over(whole),
        )
        .withColumn(
            "partner_ok",
            F.col("partner_id").isNotNull() & (F.col("partner_launch") <= F.col("event_date")),
        )
        .withColumn("partner_cents", F.col("_partner_price.price_cents"))
    )
    return eligible.groupBy("event_date", "department").agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    "_cdf",
                    "product_id",
                    "SalePrice",
                    "taxability",
                    "unit_cents",
                    "partner_id",
                    "partner_cents",
                    "partner_taxability",
                    "partner_ok",
                )
            )
        ).alias("_catalog"),
        F.first("_expected_line_value").alias("_expected_line_value"),
        F.first("_expected_taxable_value").alias("_expected_taxable_value"),
    )


def pick_retail_product(catalog: Column, uniform: Column) -> Column:
    """Binary CDF search avoids scanning the catalog for every basket line."""
    size = F.size(catalog)
    initial = F.struct(F.lit(1).alias("lo"), size.alias("hi"))

    def narrow(bounds: Column, _step: Column) -> Column:
        middle = F.floor((bounds["lo"] + bounds["hi"]) / 2).cast("int")
        left = F.element_at(catalog, middle)["_cdf"] > uniform
        return F.struct(
            F.when(left, bounds["lo"]).otherwise(F.least(middle + 1, size)).alias("lo"),
            F.when(left, middle).otherwise(bounds["hi"]).alias("hi"),
        )

    steps = F.sequence(F.lit(1), (F.ceil(F.log2(size)) + 1).cast("int"))
    selected = F.aggregate(steps, initial, narrow)["lo"]
    # The final entry absorbs floating-point CDF accumulation error.
    return F.when(size > 0, F.element_at(catalog, selected)).otherwise(
        F.raise_error("Cannot select a product from an empty catalog")
    )


def calibrated_basket_mean(
    expected_line: Column,
    expected_taxable: Column,
    tax_rate: Column,
    target: float,
    promo_rate: float,
    nominal_lines: float,
    complement_rate: float = 0.0,
) -> Column:
    """Drawn basket lines so the average ticket meets ``target``.

    ``complement_rate`` is the share of drawn lines that bring a complementary
    product along (``baskets.with_complements``); those extra lines count
    towards the ticket too.
    """
    affordable_lines = F.lit(target) / (
        (expected_line + expected_taxable * tax_rate)
        * (1.0 - promo_rate * 0.20)
        * (1.0 + complement_rate)
    )
    return F.when(
        (expected_line > 0)
        & (expected_line < float("inf"))
        & (expected_taxable >= 0)
        & (tax_rate >= 0),
        F.least(affordable_lines, F.lit(nominal_lines)),
    ).otherwise(F.raise_error("Basket calibration requires valid catalog prices and tax"))


"""Customer lifecycle shared by history and the live stream.

Every customer has, as pure functions of the seed and their ID:

- a **home store** (``dim_customers.HomeStoreID``), drawn by store traffic, where
  most of their visits happen;
- a **visit frequency**: a long-tailed annual trip rate around the store type's
  ``customer_visits_per_year``, capped at three trips a week, replicated as
  selection tickets;
- a **segment** and a persistent **basket budget** (the draw baskets use);
- a **signup day**: pre-history customers are already active when history
  starts, the rest sign up uniformly across the history (the acquisition
  curve), so the base turns over instead of starting complete;
- a **churn day** from a feature-driven hazard: budget and convenience
  shoppers, infrequent visitors and small spenders churn sooner. The base
  hazard is calibrated so ``annual_churn_rate`` of the customers active at a
  given time churn within a year;
- a **taper**: most churners visit less and less over their last
  ``TAPER_DAYS``, the early-warning signal churn models learn from; the rest
  stop abruptly.

Receipts thin visits by this lifecycle: a store visit is identified with
``identify_probability``, picks a ticket from its store's pool (or, for
``1 - HOME_STORE_SHARE`` of visits, another store's), and keeps the customer
with their activity that day; otherwise the receipt is anonymous. So each
customer's trips form a thinned Poisson process at their own rate, and the
anonymous share of receipts is ``anonymous_share`` on average.

``build_notebooks.py`` inlines this module into the setup and stream notebooks.
"""

from dataclasses import dataclass
from datetime import date
from functools import cached_property, lru_cache
from math import exp

import numpy as np
from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Shopping segments (datagen CustomerJourney distribution) and their relative
# churn hazard.
SEGMENT_WEIGHTS: list[tuple[str, float]] = [
    ("BUDGET", 0.35),
    ("CONVENIENCE", 0.25),
    ("QUALITY", 0.20),
    ("BRAND_LOYAL", 0.20),
]
SEGMENT_HAZARD = {"BUDGET": 1.3, "CONVENIENCE": 1.1, "QUALITY": 0.8, "BRAND_LOYAL": 0.6}
# Trip rate: lognormal around the store type's mean, capped at three a week.
VISIT_RATE_SIGMA = 1.0
MAX_VISITS_PER_YEAR = 156.0
# Selection tickets of a customer at the mean trip rate (ticket resolution).
TICKETS_PER_MEAN_VISITOR = 6
# Hazard elasticities: infrequent visitors and small spenders churn sooner.
FREQUENCY_HAZARD_EXPONENT = -0.35
SPEND_HAZARD_COEFFICIENT = -0.25
# Share of a customer's identified visits made at their home store.
HOME_STORE_SHARE = 0.85
# Most churners taper: activity falls linearly to TAPER_FLOOR over the last
# TAPER_DAYS before churn.
TAPER_DAYS = 60
TAPER_FLOOR = 0.15
TAPERED_SHARE = 0.75
# Pre-history customers signed up up to this many days before history.
PRE_HISTORY_MAX_TENURE_DAYS = 1500
PRE_HISTORY_MIN_TENURE_DAYS = 30
# Share of all customers active at a time for a typical 18-month history at the
# default churn rate; sizes the default customer count (config.generation).
NOMINAL_ACTIVE_SHARE = 0.75
DAYS_PER_YEAR = 365.25
# Online orders try this many customers for one active that day.
ONLINE_CUSTOMER_ATTEMPTS = 4

_GAUSS_NODES, _GAUSS_WEIGHTS = np.polynomial.hermite_e.hermegauss(24)
_GAUSS_WEIGHTS = _GAUSS_WEIGHTS / _GAUSS_WEIGHTS.sum()
_BUDGET_NODES, _BUDGET_WEIGHTS = np.polynomial.hermite_e.hermegauss(9)
_BUDGET_WEIGHTS = _BUDGET_WEIGHTS / _BUDGET_WEIGHTS.sum()


def relative_visit_rate(visit_gauss: float) -> float:
    """Trip rate relative to the mean for a standard-normal propensity draw."""
    return exp(VISIT_RATE_SIGMA * visit_gauss - VISIT_RATE_SIGMA**2 / 2)


def hazard_multiplier(segment: str, relative_rate: float, budget_gauss: float) -> float:
    """Relative churn hazard of a customer's segment, trip rate and basket budget."""
    return (
        SEGMENT_HAZARD[segment]
        * relative_rate**FREQUENCY_HAZARD_EXPONENT
        * exp(SPEND_HAZARD_COEFFICIENT * budget_gauss)
    )


def activity(day: int, signup_day: int, churn_day: int, tapered: bool) -> float:
    """Share of a customer's trips that happen on ``day`` (days since history start)."""
    if day < signup_day or day >= churn_day:
        return 0.0
    taper_start = churn_day - TAPER_DAYS
    if tapered and day >= taper_start:
        return 1.0 - (1.0 - TAPER_FLOOR) * (day - taper_start) / TAPER_DAYS
    return 1.0


def _customer_types() -> list[tuple[float, float, float]]:
    """(headcount weight, ticket weight, hazard multiplier) over the population."""
    types = []
    for segment, segment_weight in SEGMENT_WEIGHTS:
        for g, g_weight in zip(_GAUSS_NODES, _GAUSS_WEIGHTS):
            rate = relative_visit_rate(float(g))
            for b, b_weight in zip(_BUDGET_NODES, _BUDGET_WEIGHTS):
                weight = segment_weight * float(g_weight) * float(b_weight)
                types.append((weight, rate, hazard_multiplier(segment, rate, float(b))))
    return types


@lru_cache(maxsize=16)
def calibrate_base_hazard(annual_churn_rate: float) -> float:
    """Base yearly hazard so ``annual_churn_rate`` of active customers churn in a year."""
    if not 0.0 < annual_churn_rate < 1.0:
        raise ValueError("annual_churn_rate must be between 0 and 1")
    types = _customer_types()

    def churned(base: float) -> float:
        return sum(w * (1.0 - exp(-base * m)) for w, _, m in types)

    low, high = 1e-9, 50.0
    for _ in range(200):
        middle = (low + high) / 2
        if churned(middle) < annual_churn_rate:
            low = middle
        else:
            high = middle
    return (low + high) / 2


@dataclass(frozen=True)
class CustomerLifecycle:
    """Lifecycle parameters for one history window."""

    seed: int
    history_start: date
    history_end: date
    annual_churn_rate: float
    anonymous_share: float
    visits_per_year: float

    @property
    def history_days(self) -> int:
        return (self.history_end - self.history_start).days + 1

    @cached_property
    def base_hazard(self) -> float:
        return calibrate_base_hazard(self.annual_churn_rate)

    @cached_property
    def pre_history_share(self) -> float:
        """Customers already active at history start; the rest join during history.

        Sized so acquisitions over the history replace the customers who churn.
        """
        mean_hazard = self.base_hazard * sum(w * m for w, _, m in _customer_types())
        return 1.0 / (1.0 + mean_hazard * self.history_days / DAYS_PER_YEAR)

    @property
    def ticket_unit(self) -> float:
        return self.visits_per_year / TICKETS_PER_MEAN_VISITOR

    def expected_active_share(self) -> float:
        """Ticket-weighted share of customers active on an average history day."""
        types = _customer_types()
        base, pre = self.base_hazard, self.pre_history_share
        years = self.history_days / DAYS_PER_YEAR
        taper_years = TAPER_DAYS / DAYS_PER_YEAR
        total = sum(w * rate for w, rate, _ in types)
        steps = 48
        share = 0.0
        for step in range(steps):
            t = years * (step + 0.5) / steps
            for weight, rate, multiplier in types:
                hazard = base * multiplier
                taper_loss = 1.0 - TAPERED_SHARE * (1.0 - TAPER_FLOOR) * hazard * taper_years / 2
                pre_active = exp(-hazard * t)
                joined_active = (1.0 - exp(-hazard * t)) / (hazard * years)
                share += weight * rate * taper_loss * (pre * pre_active + (1 - pre) * joined_active)
        return share / total / steps

    @cached_property
    def identify_probability(self) -> float:
        """Chance a store visit tries a customer, so receipts are ``anonymous_share`` anonymous."""
        return min(1.0, (1.0 - self.anonymous_share) / self.expected_active_share())


def lifecycle_for(cfg, visits_per_year: float) -> CustomerLifecycle:
    """The lifecycle of a generation config (``GenerationConfig``)."""
    return CustomerLifecycle(
        seed=cfg.seed,
        history_start=cfg.start_date,
        history_end=cfg.end_date,
        annual_churn_rate=cfg.annual_churn_rate,
        anonymous_share=cfg.anonymous_share,
        visits_per_year=visits_per_year,
    )


# ---------------------------------------------------------------------------
# Column forms
# ---------------------------------------------------------------------------


def customer_key(customer_id: Column) -> Column:
    """Hash key of a customer's persistent draws (shared with basket budgets)."""
    return customer_id.cast("string")


def shopper_key(customer_id: Column, purchase_key: Column) -> Column:
    """Basket-draw key: the customer, or the purchase itself when anonymous."""
    return F.coalesce(customer_key(customer_id), F.concat(F.lit("anon-"), purchase_key))


def budget_gauss(draws: seeded_draws, key: Column) -> Column:
    """Persistent basket budget draw (``behavior.retail_basket_size``)."""
    return draws.gauss([key], "retail-budget")


def relative_rate_column(draws: seeded_draws, customer_id: Column) -> Column:
    """Column form of :func:`relative_visit_rate` for a customer."""
    gauss = draws.gauss([customer_key(customer_id)], "visit-propensity")
    return F.exp(F.lit(VISIT_RATE_SIGMA) * gauss - VISIT_RATE_SIGMA**2 / 2)


def visit_tickets(lc: CustomerLifecycle, customer_id: Column) -> Column:
    """Selection tickets: the customer's trip rate in units of ``ticket_unit``."""
    draws = seeded_draws(lc.seed)
    rate = F.least(
        relative_rate_column(draws, customer_id) * lc.visits_per_year,
        F.lit(MAX_VISITS_PER_YEAR),
    )
    return F.greatest(F.lit(1), F.round(rate / lc.ticket_unit)).cast("int")


def _segment_hazard(segment_draw: Column) -> Column:
    """Hazard factor of the segment a uniform draw picks (``SEGMENT_WEIGHTS`` order)."""
    total = sum(weight for _, weight in SEGMENT_WEIGHTS)
    cumulative, branches = 0.0, []
    for name, weight in SEGMENT_WEIGHTS[:-1]:
        cumulative += weight / total
        branches.append((segment_draw < F.lit(cumulative), F.lit(SEGMENT_HAZARD[name])))
    return case_when(branches).otherwise(F.lit(SEGMENT_HAZARD[SEGMENT_WEIGHTS[-1][0]]))


def lifecycle_struct(lc: CustomerLifecycle, customer_id: Column) -> Column:
    """``struct<signup_day int, churn_day long, tapered boolean>`` (days since history start)."""
    draws = seeded_draws(lc.seed)
    key = [customer_key(customer_id)]
    pre = lc.pre_history_share
    inputs = F.struct(
        draws.u(key, "customer-signup").alias("join"),
        draws.u(key, "customer-tenure").alias("tenure"),
        draws.u(key, "customer-churn").alias("churn"),
        draws.u(key, "customer-taper").alias("taper"),
        draws.u(key, "customer-segment").alias("segment"),
        relative_rate_column(draws, customer_id).alias("rate"),
        budget_gauss(draws, customer_key(customer_id)).alias("budget"),
    )

    def build(r: Column) -> Column:
        hazard = (
            F.lit(lc.base_hazard)
            * _segment_hazard(r["segment"])
            * F.pow(r["rate"], F.lit(FREQUENCY_HAZARD_EXPONENT))
            * F.exp(F.lit(SPEND_HAZARD_COEFFICIENT) * r["budget"])
        )
        is_pre = r["join"] < F.lit(pre)
        tenure = PRE_HISTORY_MIN_TENURE_DAYS + F.floor(
            r["tenure"] * (PRE_HISTORY_MAX_TENURE_DAYS - PRE_HISTORY_MIN_TENURE_DAYS)
        )
        joined = F.floor((r["join"] - F.lit(pre)) / F.lit(1.0 - pre) * lc.history_days)
        signup_day = F.when(is_pre, -tenure).otherwise(joined).cast("int")
        # Exponential lifetime from the hazard, counted from history start for
        # pre-history customers (memoryless) and from signup for the rest.
        lifetime_days = F.floor(-F.log(F.lit(1.0) - r["churn"]) / hazard * DAYS_PER_YEAR)
        churn_start = F.when(is_pre, F.lit(0)).otherwise(joined).cast("long")
        return F.struct(
            signup_day.alias("signup_day"),
            (churn_start + lifetime_days).cast("long").alias("churn_day"),
            (r["taper"] < F.lit(TAPERED_SHARE)).alias("tapered"),
        )

    # The draws are evaluated once and bound to a lambda variable, which
    # keeps the expression (and query planning) small.
    return bind(inputs, build)


def activity_column(lc: CustomerLifecycle, customer_id: Column, day: Column) -> Column:
    """Column form of :func:`activity` for a customer and a date column (0 when NULL)."""
    offset = F.datediff(day, F.lit(lc.history_start))

    def from_life(life: Column) -> Column:
        taper_start = life["churn_day"] - TAPER_DAYS
        tapered = F.lit(1.0) - F.lit(1.0 - TAPER_FLOOR) * (offset - taper_start) / F.lit(
            float(TAPER_DAYS)
        )
        return (
            F.when(
                customer_id.isNull()
                | (offset < life["signup_day"])
                | (offset >= life["churn_day"]),
                F.lit(0.0),
            )
            .when(life["tapered"] & (offset >= taper_start), tapered)
            .otherwise(F.lit(1.0))
        )

    return bind(lifecycle_struct(lc, customer_id), from_life)


# ---------------------------------------------------------------------------
# Ticket pools
# ---------------------------------------------------------------------------


def store_picker(stores: list[tuple[int, float]]) -> Column:
    """A literal CDF over ``(store_id, multiplier)`` for store picks weighted by traffic.

    A literal (at most ``store_count`` entries) rather than a joined frame:
    joining a one-row frame lets the optimizer push upstream filters into the
    join condition, which copies large expressions into the plan.
    """
    if not stores:
        raise ValueError("customer lifecycles need at least one store")
    total = sum(weight for _, weight in stores)
    cumulative, entries = 0.0, []
    for store_id, weight in sorted(stores):
        cumulative += weight / total
        entries.append(
            F.struct(
                F.lit(cumulative).alias("_cdf"), F.lit(int(store_id)).cast("long").alias("store_id")
            )
        )
    return F.array(*entries)


def picked_store(picker: Column, draw: Column) -> Column:
    """The store of a :func:`store_picker` CDF at a uniform draw."""
    return bind(picker, lambda cdf: pick_retail_product(cdf, draw)["store_id"]).cast("long")


def with_home_store(seed: int, customers: DataFrame, picker: Column) -> DataFrame:
    """Add ``HomeStoreID``, drawn in proportion to store traffic, to customers with ``ID``.

    Depends on the seed and the stores only, so dimensions stay independent
    of the history window.
    """
    draws = seeded_draws(seed)
    draw = draws.u([customer_key(F.col("ID"))], "customer-home-store")
    return customers.withColumn("HomeStoreID", picked_store(picker, draw))


def ticket_pool(lc: CustomerLifecycle, customers: DataFrame) -> DataFrame:
    """``pool_store, rank, customer_id``: each customer's tickets, ranked within their home store.

    ``customers`` has ``ID`` and ``HomeStoreID``.
    """
    tickets = customers.select(
        F.col("ID").cast("long").alias("customer_id"),
        F.col("HomeStoreID").cast("long").alias("pool_store"),
    ).where(F.col("pool_store").isNotNull())
    tickets = tickets.withColumn(
        "_ticket", F.explode(F.sequence(F.lit(1), visit_tickets(lc, F.col("customer_id"))))
    )
    order = Window.partitionBy("pool_store").orderBy("customer_id", "_ticket")
    return tickets.select("pool_store", F.row_number().over(order).alias("rank"), "customer_id")


def pool_sizes(pool: DataFrame) -> DataFrame:
    """``pool_store, pool_tickets``."""
    return pool.groupBy("pool_store").agg(F.count("*").alias("pool_tickets"))


def assign_customers(
    lc: CustomerLifecycle,
    visits: DataFrame,
    pool: DataFrame,
    sizes: DataFrame,
    picker: Column,
    key: str,
    store: str,
    day: str,
) -> DataFrame:
    """Add ``customer_id`` to store visits (NULL for anonymous receipts).

    ``key`` names the visit's unique ID, ``store`` its store and ``day`` its
    local date. Away visits use a store from ``picker`` (:func:`store_picker`).
    """
    draws = seeded_draws(lc.seed)
    k = [F.col(key)]
    frame = visits.withColumn(
        "_pool_store",
        F.when(draws.u(k, "customer-home-visit") < F.lit(HOME_STORE_SHARE), F.col(store)).otherwise(
            picked_store(picker, draws.u(k, "customer-away-store"))
        ),
    ).join(F.broadcast(sizes.withColumnRenamed("pool_store", "_pool_store")), "_pool_store", "left")
    frame = frame.withColumn(
        "_rank", (draws.h64(k, "customer-ticket") % F.col("pool_tickets") + 1).cast("long")
    )
    picked = pool.select(
        F.col("pool_store").alias("_pool_store"),
        F.col("rank").alias("_rank"),
        F.col("customer_id").alias("_picked"),
    )
    frame = frame.join(picked, ["_pool_store", "_rank"], "left")
    keep = (draws.u(k, "customer-identified") < F.lit(lc.identify_probability)) & (
        draws.u(k, "customer-active") < activity_column(lc, F.col("_picked"), F.col(day))
    )
    return frame.withColumn("customer_id", F.when(keep, F.col("_picked")).cast("long")).drop(
        "_pool_store", "pool_tickets", "_rank", "_picked"
    )


def assign_online_customers(
    lc: CustomerLifecycle,
    orders: DataFrame,
    pool: DataFrame,
    sizes: DataFrame,
    picker: Column,
    key: str,
    day: str,
) -> DataFrame:
    """Add ``customer_id`` to online orders: the first of a few tries active that day.

    Orders whose tries all land on inactive customers are dropped (about
    ``(1 - active share) ** ONLINE_CUSTOMER_ATTEMPTS`` of them).
    """
    draws = seeded_draws(lc.seed)
    tries = orders.select(
        F.col(key).alias("_order"),
        F.col(day).alias("_day"),
        F.explode(F.sequence(F.lit(1), F.lit(ONLINE_CUSTOMER_ATTEMPTS))).alias("_try"),
    )
    k = [F.col("_order"), F.col("_try")]
    tries = tries.withColumn(
        "_pool_store", picked_store(picker, draws.u(k, "online-customer-store"))
    ).join(F.broadcast(sizes.withColumnRenamed("pool_store", "_pool_store")), "_pool_store")
    tries = tries.withColumn(
        "_rank", (draws.h64(k, "online-customer-ticket") % F.col("pool_tickets") + 1).cast("long")
    ).join(
        pool.select(
            F.col("pool_store").alias("_pool_store"),
            F.col("rank").alias("_rank"),
            F.col("customer_id").alias("_picked"),
        ),
        ["_pool_store", "_rank"],
    )
    active = draws.u(k, "online-customer-active") < activity_column(
        lc, F.col("_picked"), F.col("_day")
    )
    chosen = (
        tries.where(active)
        .groupBy("_order")
        .agg(F.min_by("_picked", "_try").cast("long").alias("customer_id"))
        .withColumnRenamed("_order", key)
    )
    return orders.join(chosen, key)


"""Basket-level line handling shared by history and the live stream.

Both producers build a purchase's lines as an array of structs and pass it
through the same steps:

1. :func:`with_complements` - each line may bring its complementary product
   (a fixed partner in the same department, see :func:`complement_partners`),
   so baskets carry real product affinities.
2. :func:`merge_lines` - repeated SKUs collapse into one line with the summed
   quantity, numbered in first-seen order.
3. :func:`basket_promotion` - an offered purchase gets at most one promotion
   code: one of the ``PROMO_CATALOG`` codes in season whose minimum is met by
   the basket subtotal and that has an eligible line. The code discounts only
   the lines in its department scope (``promo_scopes``); BOGO discounts every
   second unit of eligible lines with two or more units.

Line structs carry ``line_num``, ``product_id``, ``quantity``, ``unit_cents``,
``taxability`` and ``department``; complements also need ``partner_id``,
``partner_cents``, ``partner_taxability`` and ``partner_ok`` (the partner is
launched).
"""

from collections.abc import Callable

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Share of lines that bring their complementary product along.
COMPLEMENT_RATE = 0.15
# Promotion code -> department-name tokens it applies to; None applies to all.
# A code whose tokens match none of a store type's departments applies to all.
PROMO_SCOPES: dict[str, list[str] | None] = {
    "SAVE10": None,
    "SAVE15": None,
    "SAVE20": None,
    "SAVE25": None,
    "CLEARANCE30": [
        "seasonal", "clothing", "apparel", "footwear", "accessor", "home", "toy",
        "electronic", "sport", "garden", "lawn",
    ],
    "BOGO50": [
        "grocery", "fresh", "food", "household", "personal", "health", "beauty", "pet", "baby",
    ],
    "NEWYEAR15": ["health", "sport", "beauty", "fitness"],
    "SPRINGSALE20": ["garden", "lawn", "home", "clothing", "apparel", "paint"],
    "SUMMER25": ["garden", "lawn", "sport", "seasonal", "outdoor", "clothing", "apparel"],
    "BACKTOSCHOOL20": ["office", "clothing", "apparel", "electronic", "baby", "kids"],
    "BFRIDAY30": [
        "electronic", "toy", "home", "clothing", "apparel", "jewel", "accessor", "tool",
    ],
    "BFRIDAY40": [
        "electronic", "toy", "home", "clothing", "apparel", "jewel", "accessor", "tool",
    ],
    "HOLIDAY20": ["toy", "seasonal", "electronic", "jewel", "fragrance", "home", "accessor"],
}  # fmt: skip
# Seasonal codes are favored while in season.
SEASONAL_CODE_WEIGHT = 3.0


def promo_scopes(departments: list[str]) -> dict[str, list[str] | None]:
    """Each code's department names among ``departments`` (None = every department)."""
    scopes: dict[str, list[str] | None] = {}
    for code, *_rest in PROMO_CATALOG:
        tokens = PROMO_SCOPES.get(code)
        matched = (
            [name for name in departments if any(t in name.lower() for t in tokens)]
            if tokens is not None
            else []
        )
        scopes[code] = sorted(matched) if matched else None
    return scopes


def receipt_offer(draws: seeded_draws, key: Column, promo_rate: float) -> Column:
    """Whether a purchase presents a promotion (``promo_rate`` of purchases)."""
    return draws.u([key], "offer") < F.lit(promo_rate)


def _in_scope(department: Column, scope: list[str] | None) -> Column:
    return F.lit(True) if scope is None else department.isin(*scope)


def _discountable(scope: list[str] | None, kind: str) -> Callable[[Column], Column]:
    """Whether a line can take a code of this scope and kind (BOGO needs 2+ units)."""

    def check(line: Column) -> Column:
        units = F.lit(True) if kind != "BOGO" else line["quantity"] >= 2
        return _in_scope(line["department"], scope) & units

    return check


def basket_promotion(
    draws: seeded_draws,
    key: Column,
    lines: Column,
    month: Column,
    offer: Column,
    profile: StoreTypeProfile,
) -> Column:
    """Lines with ``promo_code`` and ``discount_cents`` from one basket-level code.

    The code is drawn among the codes eligible for the basket (in season,
    basket gross subtotal at or above the code's minimum, at least one
    discountable line); lines outside its scope, and BOGO lines with one unit,
    keep no code and no discount. Purchases without an offer get no codes.
    """
    scopes = promo_scopes(list(profile.department_weights))
    gross = F.aggregate(
        lines, F.lit(0).cast("long"), lambda acc, line: acc + line["unit_cents"] * line["quantity"]
    )

    def choose(basket_cents: Column) -> Column:
        candidates = []
        for code, _pct, months, min_cents, kind in PROMO_CATALOG:
            eligible = F.exists(lines, _discountable(scopes[code], kind)) & (
                basket_cents >= F.lit(min_cents)
            )
            if months is not None:
                eligible = eligible & month.isin(*months)
            weight = SEASONAL_CODE_WEIGHT if months is not None else 1.0
            # Weighted random choice: the largest u ** (1 / weight) among eligible codes.
            score = F.when(eligible, F.pow(draws.u([key, F.lit(code)], "pcode"), 1.0 / weight))
            candidates.append(F.struct(score.alias("score"), F.lit(code).alias("code")))
        eligible_codes = F.filter(F.array(*candidates), lambda entry: entry["score"].isNotNull())
        return F.when(offer, F.array_max(eligible_codes)["code"])

    def priced(line: Column, code: Column) -> Column:
        line_cents = line["unit_cents"] * line["quantity"]
        discount: Column = F.lit(0).cast("long")
        for name, pct, _months, _min_cents, kind in PROMO_CATALOG:
            applies = (code == F.lit(name)) & _in_scope(line["department"], scopes[name])
            if kind == "BOGO":
                amount = F.floor(
                    F.floor(line["quantity"] / 2) * line["unit_cents"] * pct / 100.0 + 0.5
                )
            else:
                amount = F.floor(line_cents * pct / 100.0 + 0.5)
            discount = F.when(applies, amount.cast("long")).otherwise(discount)
        discount = F.coalesce(discount, F.lit(0).cast("long"))
        return line.withField("promo_code", F.when(discount > 0, code)).withField(
            "discount_cents", discount
        )

    # The subtotal and the chosen code are evaluated once per basket and bound
    # to lambda variables, which keeps the expression (and planning) small.
    code = bind(gross, choose)
    return bind(code, lambda chosen: F.transform(lines, lambda line: priced(line, chosen)))


def merge_lines(lines: Column) -> Column:
    """Collapse repeated SKUs into one line (summed quantity), renumbered 1..n."""
    products = F.array_distinct(F.transform(lines, lambda line: line["product_id"]))

    def merged(product_id: Column, index: Column) -> Column:
        same = F.filter(lines, lambda line: line["product_id"] == product_id)
        quantity = F.aggregate(same, F.lit(0), lambda acc, line: acc + line["quantity"].cast("int"))
        first = F.element_at(same, 1)
        return first.withField("line_num", (index + 1).cast("int")).withField(
            "quantity", quantity.cast("int")
        )

    return F.transform(products, merged)


def with_complements(draws: seeded_draws, key: Column, lines: Column) -> Column:
    """Lines followed, for ``COMPLEMENT_RATE`` of them, by one unit of their partner."""

    def expand(line: Column) -> Column:
        brings = line["partner_ok"] & (
            draws.u([key, line["line_num"]], "complement") < F.lit(COMPLEMENT_RATE)
        )
        partner = (
            line.withField("product_id", line["partner_id"])
            .withField("quantity", F.lit(1))
            .withField("unit_cents", line["partner_cents"])
            .withField("taxability", line["partner_taxability"])
        )
        return F.when(brings, F.array(line, partner)).otherwise(F.array(line))

    return F.flatten(F.transform(lines, expand))


def complement_partners(products: DataFrame, seed: int) -> DataFrame:
    """``product_id, partner_id``: a fixed complementary product in the same department.

    Products are ranked within their department by a seeded hash; a product's
    partner sits a seeded, non-zero offset further round the ranking, so each
    pair is stable for the whole history and the live stream. Departments
    with one product have no partner.
    """
    draws = seeded_draws(seed)
    ranked = products.select(
        F.col("product_id").cast("long").alias("product_id"), "department"
    ).withColumn(
        "_rank",
        F.row_number().over(
            Window.partitionBy("department").orderBy(
                draws.h64([F.col("product_id")], "complement-rank"), "product_id"
            )
        ),
    )
    ranked = ranked.withColumn("_size", F.count("*").over(Window.partitionBy("department")))
    offset = draws.h64([F.col("product_id")], "complement-offset") % (F.col("_size") - 1) + 1
    wanted = ranked.where(F.col("_size") > 1).withColumn(
        "_partner_rank", F.pmod(F.col("_rank") - 1 + offset, F.col("_size")) + 1
    )
    partners = ranked.select(
        "department",
        F.col("_rank").alias("_partner_rank"),
        F.col("product_id").alias("partner_id"),
    )
    return wanted.join(partners, ["department", "_partner_rank"]).select("product_id", "partner_id")


def basket_products(dim_products: DataFrame, seed: int) -> DataFrame:
    """Catalog input of ``behavior.retail_catalog``: SKUs with their complementary partner.

    ``product_id, SalePrice, taxability, department, launch_date, inflation,
    elasticity`` plus the partner's ``partner_id, partner_price, partner_taxability, partner_launch``
    (NULL for a department of one product).
    """
    products = dim_products.select(
        F.col("ID").cast("long").alias("product_id"),
        "SalePrice",
        "taxability",
        F.col("Department").alias("department"),
        F.to_date("LaunchDate").alias("launch_date"),
        # Department price trend and elasticity (pricing.DEPARTMENT_PRICING),
        # resolved once per SKU here instead of in every price expression.
        inflation_column(F.col("Department")).alias("inflation"),
        elasticity_column(F.col("Department")).alias("elasticity"),
    )
    partner = products.select(
        F.col("product_id").alias("partner_id"),
        F.col("SalePrice").alias("partner_price"),
        F.col("taxability").alias("partner_taxability"),
        F.col("launch_date").alias("partner_launch"),
    )
    return products.join(complement_partners(products, seed), "product_id", "left").join(
        partner, "partner_id", "left"
    )


def line_tax_cents(net_cents: Column, taxability: Column, bps: Column) -> Column:
    """Integer basis-point tax of a line's net cents (datagen ``_tax_cents``)."""
    multiplier = (
        F.when(taxability == "TAXABLE", 100).when(taxability == "REDUCED_RATE", 50).otherwise(0)
    ).cast("long")
    return F.floor((net_cents * bps * multiplier + F.lit(500_000)) / F.lit(1_000_000)).cast("long")


def price_basket(
    draws: seeded_draws,
    key: Column,
    lines: Column,
    month: Column,
    offer: Column,
    bps: Column,
    profile: StoreTypeProfile,
) -> Column:
    """Lines with the basket promotion, ``ext_cents`` (net) and ``tax_cents``."""
    promoted = basket_promotion(draws, key, lines, month, offer, profile)

    def finish(line: Column) -> Column:
        gross = line["unit_cents"] * line["quantity"]
        net = F.greatest(F.lit(0).cast("long"), gross - line["discount_cents"])
        return line.withField("ext_cents", net).withField(
            "tax_cents", line_tax_cents(net, line["taxability"], bps)
        )

    return F.transform(promoted, finish)


def sum_lines(lines: Column, field: str) -> Column:
    """Sum of a long field over a basket's lines."""
    return F.aggregate(lines, F.lit(0).cast("long"), lambda acc, line: acc + line[field])


"""Visit-first store traffic shared by receipt generation and footfall facts."""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window



def generate_hourly_visits(
    spark: SparkSession,
    stores: DataFrame,
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
) -> DataFrame:
    draws = seeded_draws(cfg.seed)
    hours = stores.select(
        F.col("ID").alias("store_id"),
        "GeographyID",
        "daily_traffic_multiplier",
        "operating_hours",
    )
    is_24 = F.col("operating_hours") == "24h"
    parts = F.split("operating_hours", "-")
    hours = hours.withColumn(
        "hour",
        F.explode(
            F.sequence(
                F.when(is_24, 0).otherwise(parts.getItem(0).cast("int")),
                F.when(is_24, 24).otherwise(parts.getItem(1).cast("int")) - 1,
            )
        ),
    ).withColumn(
        "_hour_weight",
        F.element_at(F.array(*[F.lit(w) for w in profile.hourly_weights]), F.col("hour") + 1),
    )
    hours = hours.withColumn(
        "_hour_share",
        F.col("_hour_weight") / F.sum("_hour_weight").over(Window.partitionBy("store_id")),
    )
    days = date_grid(spark, cfg.start_date, cfg.end_date).filter(
        ~((F.month("day") == 12) & (F.dayofmonth("day") == 25))
    )
    grid = hours.crossJoin(days)
    daily = daily_factor_column(profile, F.col("day"))
    monthly = monthly_factor_column(profile, F.col("day"))
    # Nearby stores share weather; store/hour conversion remains heterogeneous.
    weather = F.exp(0.15 * draws.gauss(["GeographyID", "day"], "weather") - 0.15**2 / 2)
    conversion = visit_conversion(draws, F.col("store_id"), F.col("day"), F.col("hour"))
    grid = grid.withColumn(
        "_visits_mean",
        F.lit(cfg.transactions_per_store_day / VISIT_CONVERSION_CALIBRATION)
        * daily
        * monthly
        * weather
        * volume_growth_column(cfg.annual_volume_growth, cfg.anchor_date, F.col("day"))
        * F.col("daily_traffic_multiplier")
        * F.col("_hour_share"),
    ).withColumn("_conversion", conversion)
    grid = grid.withColumn(
        "visitors",
        draws.negative_binomial(["store_id", "day", "hour"], "visitors", F.col("_visits_mean"), 24),
    )
    visitor_ids = F.when(
        F.col("visitors") > 0, F.sequence(F.lit(1).cast("long"), F.col("visitors"))
    ).otherwise(F.array().cast("array<long>"))
    grid = grid.withColumn(
        "n_receipts",
        F.aggregate(
            visitor_ids,
            F.lit(0).cast("long"),
            lambda count, visitor: (
                count
                + (
                    draws.u(["store_id", "day", "hour", visitor], "visit-conversion")
                    < F.col("_conversion")
                ).cast("long")
            ),
        ),
    )
    return grid.withColumn(
        "hour_ts",
        F.timestamp_seconds(
            F.unix_timestamp(F.col("day").cast("timestamp")) + F.col("hour") * 3600
        ),
    ).select("store_id", "day", "hour", "hour_ts", "visitors", "n_receipts")


"""Receipts fact group, Spark-native.

Randomness: every stochastic decision derives a uniform double from
xxhash64(key columns, salt) via `runtime.seeded_draws` — partition-
arrangement-independent, so output is deterministic for a (config, seed) pair
regardless of cluster shape. The seed is folded into the salt delimiter inside
`seeded_draws`, keeping generators decoupled from one another.

Visits precede purchase decisions. Negative-binomial baskets combine shopping
missions and persistent budgets, calibrated against the eligible catalog.
Products are chosen by price-elastic popularity at the day's shelf price
(pricing.py); each basket then gains complementary products, merges repeated
SKUs and takes at most one promotion code (baskets.py, shared with the live
stream). Customers follow their lifecycle (customers.py); about 45% of
receipts are anonymous.

Money is integer cents end-to-end; tax replicates datagen's basis-point
integer formula exactly:
    rate_bps = round(rate * 10000); mult = 100/50/0 by taxability;
    tax = (ext_cents * rate_bps * mult + 500_000) // 1_000_000
implemented with Spark integer `DIV` so no float rounding is involved.
"""

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Per-department seasonal demand multipliers by month (Jan..Dec), keyed by a
# lowercase token matched as a substring of the department name so the one
# table works across store types (datagen SeasonalPatterns category effects).
SEASONAL_LIFT: dict[str, list[float]] = {
    "electronics": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.0, 1.0, 2.5, 2.0],
    "seasonal": [0.6, 0.7, 0.9, 1.0, 1.3, 1.5, 1.5, 1.2, 1.0, 1.4, 2.0, 2.5],
    "garden": [0.5, 0.6, 1.3, 1.8, 2.0, 1.8, 1.4, 1.0, 0.8, 0.6, 0.5, 0.5],
    "home": [0.8, 0.8, 1.2, 1.5, 1.6, 1.4, 1.2, 1.0, 0.9, 0.8, 1.0, 1.1],
    "sport": [0.9, 0.9, 1.1, 1.3, 1.5, 1.6, 1.6, 1.3, 1.1, 1.0, 1.1, 1.2],
    "clothing": [1.0, 0.9, 1.1, 1.1, 1.0, 1.0, 1.0, 1.4, 1.2, 1.0, 1.3, 1.5],
    "apparel": [1.0, 0.9, 1.1, 1.1, 1.0, 1.0, 1.0, 1.4, 1.2, 1.0, 1.3, 1.5],
    "office": [1.1, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.6, 1.4, 1.0, 1.0, 1.1],
    "toys": [0.8, 0.8, 0.8, 0.9, 0.9, 1.0, 1.0, 1.0, 1.0, 1.1, 2.0, 3.0],
    "baby": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.2],
    "grocery": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.3, 1.4],
    "fresh": [1.0, 1.0, 1.0, 1.0, 1.05, 1.1, 1.1, 1.05, 1.0, 1.05, 1.3, 1.4],
    "health": [1.3, 1.05, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.1],
    "beauty": [1.2, 1.05, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.2, 1.5],
    "automotive": [1.0, 1.0, 1.05, 1.1, 1.2, 1.2, 1.2, 1.1, 1.0, 1.0, 1.0, 1.0],
}


# Weather simulation (datagen EventPatterns): a per-store-day weather state
# scales foot traffic. Seasonal odds make winter snow/storm and summer sun more
# likely. Only the traffic multiplier surfaces (there is no weather column).
WEATHER_MULTS = [1.1, 1.0, 0.7, 0.6, 0.5]  # sunny, cloudy, rainy, snowy, stormy
WEATHER_P_WINTER = [0.25, 0.30, 0.15, 0.20, 0.10]
WEATHER_P_SUMMER = [0.55, 0.25, 0.15, 0.00, 0.05]
WEATHER_P_SHOULDER = [0.40, 0.30, 0.20, 0.05, 0.05]

# Shopping trip archetypes (datagen ShoppingBehaviorType): each trip is a quick
# run / normal run / family shop / bulk stock-up, giving multi-modal basket
# sizes. Multipliers scale the store-type basket_lambda; weighted mean ~1.0 so
# overall item volume is preserved.
TRIP_TYPES: list[tuple[str, float, float]] = [  # (name, weight, basket multiplier)
    ("QUICK_TRIP", 0.30, 0.30),
    ("GROCERY_RUN", 0.40, 0.90),
    ("FAMILY_SHOPPING", 0.20, 1.60),
    ("BULK_SHOPPING", 0.10, 2.50),
]


def _cdf_pick_lit(u: Column, probs: list[float], values: list[float]) -> Column:
    """Inverse-CDF pick returning the chosen literal value."""
    acc = 0.0
    expr: Column | None = None
    for p, v in list(zip(probs, values))[:-1]:
        acc += p
        cond = u < F.lit(acc)
        expr = F.when(cond, F.lit(v)) if expr is None else expr.when(cond, F.lit(v))
    return expr.otherwise(F.lit(values[-1])) if expr is not None else F.lit(values[0])


def _weather_mult(u: Column, month: Column) -> Column:
    """Per-store-day weather traffic multiplier with seasonal weather odds."""
    return (
        F.when(month.isin(12, 1, 2), _cdf_pick_lit(u, WEATHER_P_WINTER, WEATHER_MULTS))
        .when(month.isin(6, 7, 8), _cdf_pick_lit(u, WEATHER_P_SUMMER, WEATHER_MULTS))
        .otherwise(_cdf_pick_lit(u, WEATHER_P_SHOULDER, WEATHER_MULTS))
    )


def _trip_basket_mult(u: Column) -> Column:
    """Pick a shopping-trip archetype's basket-size multiplier (inverse-CDF)."""
    return _cdf_pick_lit(u, [w for _, w, _ in TRIP_TYPES], [m for _, _, m in TRIP_TYPES])


def _seasonal_factor(dept_name: str, month_col: Column) -> Column:
    """Monthly demand multiplier for a department (1.0 if no token matches)."""
    key = dept_name.lower()
    lifts = next((m for token, m in SEASONAL_LIFT.items() if token in key), None)
    if lifts is None:
        return F.lit(1.0)
    expr = case_when((month_col == F.lit(m), F.lit(lifts[m - 1])) for m in range(1, 13))
    return expr.otherwise(F.lit(1.0))


def _with_seasonal_department(
    df: DataFrame, u_col: Column, dept_weights: dict[str, float]
) -> DataFrame:
    """Pick a department per row via inverse-CDF over base weights scaled by the
    seasonal lift for the row's month.

    Intermediate per-department weight (``_w*``) and cumulative-CDF (``_c*``)
    columns are materialized so Catalyst codegen stays small even for store
    types with many departments (an inline single expression overflows the
    64 KB JVM method limit and forces interpreted fallback). They are dropped
    before returning.
    """
    depts = list(dept_weights)
    month = F.month(F.col("event_date"))
    out = df
    for i, dn in enumerate(depts):
        out = out.withColumn(f"_w{i}", F.lit(float(dept_weights[dn])) * _seasonal_factor(dn, month))
    wt = F.col("_w0")
    for i in range(1, len(depts)):
        wt = wt + F.col(f"_w{i}")
    out = out.withColumn("_wt", wt)
    prev: str | None = None
    for i in range(len(depts) - 1):
        term = F.col(f"_w{i}") / F.col("_wt")
        out = out.withColumn(f"_c{i}", term if prev is None else F.col(prev) + term)
        prev = f"_c{i}"
    expr: Column | None = None
    for i, dn in enumerate(depts[:-1]):
        cond = u_col < F.col(f"_c{i}")
        expr = F.when(cond, F.lit(dn)) if expr is None else expr.when(cond, F.lit(dn))
    dept_col = expr.otherwise(F.lit(depts[-1])) if expr is not None else F.lit(depts[0])
    out = out.withColumn("department", dept_col)
    drop_cols = (
        [f"_w{i}" for i in range(len(depts))] + ["_wt"] + [f"_c{i}" for i in range(len(depts) - 1)]
    )
    return out.drop(*drop_cols)


def customer_pools(
    lifecycle: CustomerLifecycle, dims: dict[str, DataFrame]
) -> tuple[DataFrame, DataFrame, Column]:
    """(ticket pool, pool sizes, store picker) of the customer dimension.

    Replicated selection tickets give persistent, long-tailed visit frequency;
    the dimension-sized expansion avoids receipt x customer inequality joins.
    """
    pool = ticket_pool(lifecycle, dims["dim_customers"])
    stores = dims["dim_stores"].select("ID", "daily_traffic_multiplier").collect()
    picker = store_picker(
        [(int(row["ID"]), float(row["daily_traffic_multiplier"])) for row in stores]
    )
    return pool, pool_sizes(pool), picker


def _assign_customers(
    receipts: DataFrame, dims: dict[str, DataFrame], lifecycle: CustomerLifecycle
) -> DataFrame:
    """Resolve each receipt's customer_id (NULL for anonymous shoppers).

    Most identified visits come from the store's own home customers, the
    rest from another store's; each customer is kept with their lifecycle
    activity on the receipt's local date (generation.customers).
    """
    pool, sizes, picker = customer_pools(lifecycle, dims)
    return assign_customers(
        lifecycle, receipts, pool, sizes, picker, "receipt_id_ext", "store_id", "event_date"
    )


def _fmt(cents_col: Column) -> Column:
    """Integer cents -> 'XX.XX' string (no thousands separators)."""
    return F.format_string("%.2f", cents_col / F.lit(100.0))


def _open_close(operating_hours: str) -> tuple[int, int]:
    """Parse an ``operating_hours`` string into ``[open, close)`` integer hours."""
    if operating_hours == "24h":
        return 0, 24
    lo, hi = operating_hours.split("-")
    return int(lo), int(hi)


def _open_close_cols(operating_hours: Column) -> tuple[Column, Column]:
    """Column form of :func:`_open_close` for ``[open, close)`` hour bounds.

    ``"24h"`` maps to ``[0, 24)``; ``"7-22"`` maps to ``[7, 22)``. The close hour
    is exclusive, so a ``"7-22"`` store transacts in hours 7..21 inclusive.
    """
    is_24 = operating_hours == "24h"
    parts = F.split(operating_hours, "-")
    open_h = F.when(is_24, F.lit(0)).otherwise(parts.getItem(0).cast("int"))
    close_h = F.when(is_24, F.lit(24)).otherwise(parts.getItem(1).cast("int"))
    return open_h, close_h


def _pick_hour_static(u: Column, hourly_weights: list[float], open_h: int, close_h: int) -> Column:
    """Inverse-CDF hour pick for a single, statically known operating window.

    Weights outside ``[open_h, close_h)`` are zeroed and the CDF renormalised
    over the open hours, so the pick can never land on a closed hour. The
    cumulative fractions are Python floats (not column arithmetic), keeping the
    generated expression small.
    """
    masked = [w if open_h <= h < close_h else 0.0 for h, w in enumerate(hourly_weights)]
    total = sum(masked) or 1.0
    chain: Column | None = None
    acc = 0.0
    for h in range(close_h - 1):
        acc += masked[h]
        cond = u < F.lit(acc / total)
        chain = F.when(cond, F.lit(h)) if chain is None else chain.when(cond, F.lit(h))
    last_open = close_h - 1
    return F.lit(last_open) if chain is None else chain.otherwise(F.lit(last_open))


def _pick_hour(
    u: Column, hourly_weights: list[float], operating_hours: Column, patterns: list[str]
) -> Column:
    """Masked inverse-CDF hour pick that respects each store's operating window.

    Operating hours come from a small fixed set of patterns, so the per-pattern
    static CDF (cheap Python-float chain) is selected by the store's
    ``operating_hours`` string — avoiding a per-row column CDF that would
    explode the Catalyst expression tree.
    """
    expr: Column | None = None
    for pat in patterns:
        open_h, close_h = _open_close(pat)
        branch = _pick_hour_static(u, hourly_weights, open_h, close_h)
        cond = operating_hours == F.lit(pat)
        expr = F.when(cond, branch) if expr is None else expr.when(cond, branch)
    # Every store carries one of ``patterns``; the fallback only guards nulls.
    return F.lit(12) if expr is None else expr.otherwise(F.lit(12))


def generate_receipts_group(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Generate fact_receipts, fact_receipt_lines, fact_payments (in-store only)."""

    d = seeded_draws(cfg.seed)

    stores = dims["dim_stores"].select(
        F.col("ID").alias("store_id"),
        "tax_rate",
        "daily_traffic_multiplier",
        "operating_hours",
        F.col("GeographyID").alias("store_geo_id"),
    )
    grid = generate_hourly_visits(spark, dims["dim_stores"], profile, cfg).join(stores, "store_id")
    earlier_hours = (
        Window.partitionBy("store_id", "day")
        .orderBy("hour")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    grid = grid.withColumn(
        "_prior_receipts", F.coalesce(F.sum("n_receipts").over(earlier_hours), F.lit(0))
    )
    receipts = (
        grid.filter(F.col("n_receipts") > 0)
        .withColumn("_hour_seq", F.explode(F.sequence(F.lit(1), F.col("n_receipts"))))
        .withColumn("seq", F.col("_prior_receipts") + F.col("_hour_seq"))
        .withColumn("minute", (d.h64(["store_id", "day", "seq"], "min") % 60).cast("int"))
        .withColumn("second", (d.h64(["store_id", "day", "seq"], "sec") % 60).cast("int"))
        .withColumn(
            "event_ts",
            F.make_timestamp(
                F.year("day"),
                F.month("day"),
                F.dayofmonth("day"),
                F.col("hour"),
                F.col("minute"),
                F.col("second"),
            ),
        )
        .withColumn("event_date", F.col("day"))
        # RCP(3) + yyyyMMddHHmm(12) + store(4) + seq(6) = 25 chars, unique by
        # construction (store_id, day, seq) is a key
        .withColumn(
            "receipt_id_ext",
            F.concat(
                F.lit("RCP"),
                F.date_format("event_ts", "yyyyMMddHHmm"),
                fixed_width_decimal(F.col("store_id"), 4, "receipt store ID"),
                fixed_width_decimal(F.col("seq"), 6, "daily receipt sequence"),
            ),
        )
        .withColumn("trace_id", F.concat(F.lit("TRC"), F.col("receipt_id_ext")))
    )

    # home-store customers by lifecycle activity, or anonymous shoppers
    lifecycle = lifecycle_for(cfg, profile.resolved_customer_visits_per_year())
    receipts = _assign_customers(receipts, dims, lifecycle).withColumn(
        "_shopper", shopper_key(F.col("customer_id"), F.col("receipt_id_ext"))
    )

    receipts = receipts.withColumn(
        "_offer", receipt_offer(d, F.col("receipt_id_ext"), profile.promo_rate)
    ).withColumn(
        "tender_type",
        tender_column(d, ["receipt_id_ext"]),
    )

    # --- lines: explode baskets, weighted department -> price- and
    # promo-elastic product within department, restricted to products already
    # launched on the sale day (IMP-010 no-pre-launch-sales invariant).
    products = basket_products(dims["dim_products"], cfg.seed)

    product_departments = {r.department for r in products.select("department").distinct().collect()}
    unknown = set(profile.department_weights) - product_departments
    if unknown:
        raise ValueError(
            f"profile department_weights reference departments missing from the catalog: {sorted(unknown)}"
        )

    # Catalog days come from the history window, not from the receipts: deriving
    # them from receipts lets the optimizer push the visit filter into the
    # traffic grid and copy its large expressions into the plan.
    sale_dates = date_grid(spark, cfg.start_date, cfg.end_date, "event_date")
    catalog = retail_catalog(products, sale_dates, d, PriceSchedule(cfg.seed, cfg.anchor_date))
    department_weight = sum(
        (
            F.when(
                F.col("department") == name,
                F.lit(weight) * _seasonal_factor(name, F.month("event_date")),
            ).otherwise(0.0)
            for name, weight in profile.department_weights.items()
        ),
        F.lit(0.0),
    )
    calibration = (
        catalog.withColumn("_weight", department_weight)
        .groupBy("event_date")
        .agg(
            (F.sum(F.col("_weight") * F.col("_expected_line_value")) / F.sum("_weight")).alias(
                "_line_value"
            ),
            (F.sum(F.col("_weight") * F.col("_expected_taxable_value")) / F.sum("_weight")).alias(
                "_taxable_value"
            ),
        )
    )
    receipts = (
        receipts.join(calibration, "event_date")
        .withColumn(
            "_basket_mean",
            calibrated_basket_mean(
                F.col("_line_value"),
                F.col("_taxable_value"),
                F.col("tax_rate"),
                profile.avg_ticket_target,
                profile.promo_rate,
                profile.basket_lambda,
                COMPLEMENT_RATE,
            ),
        )
        .withColumn(
            "basket_n",
            retail_basket_size(
                d,
                ["receipt_id_ext"],
                F.col("_shopper"),
                F.col("_basket_mean"),
                F.col("_offer"),
                profile.promo_rate,
            ),
        )
    )

    exploded = receipts.select(
        "receipt_id_ext",
        "event_ts",
        "event_date",
        "store_id",
        "tax_rate",
        "basket_n",
        "_offer",
        "_shopper",
    ).withColumn("line_num", F.explode(F.sequence(F.lit(1), F.col("basket_n"))))
    exploded = _with_seasonal_department(
        exploded,
        F.when(
            d.u(["receipt_id_ext", "line_num"], "mission-coherence") < 0.60,
            F.when(
                d.u(["receipt_id_ext"], "favorite-mission") < 0.50,
                d.u(["_shopper"], "favorite-department"),
            ).otherwise(d.u(["receipt_id_ext"], "mission-department")),
        ).otherwise(d.u(["receipt_id_ext", "line_num"], "dept")),
        profile.department_weights,
    )

    key = ["receipt_id_ext", "line_num"]
    drawn = (
        exploded.join(catalog, ["event_date", "department"])
        .withColumn("_product", pick_retail_product(F.col("_catalog"), d.u(key, "prod")))
        .select(
            "receipt_id_ext",
            F.struct(
                F.col("line_num").cast("int").alias("line_num"),
                F.col("_product.product_id").alias("product_id"),
                retail_quantity(d, key, F.col("department")).alias("quantity"),
                F.col("_product.unit_cents").alias("unit_cents"),
                F.col("_product.taxability").alias("taxability"),
                F.col("department"),
                F.col("_product.partner_id").alias("partner_id"),
                F.col("_product.partner_cents").alias("partner_cents"),
                F.col("_product.partner_taxability").alias("partner_taxability"),
                F.col("_product.partner_ok").alias("partner_ok"),
            ).alias("_line"),
        )
        .groupBy("receipt_id_ext")
        .agg(F.array_sort(F.collect_list("_line")).alias("_lines"))
    )
    # Basket-level steps shared with the live stream: complementary products,
    # repeated SKUs merged into one line, and at most one promotion code whose
    # threshold the basket meets; tax is integer basis-point math on the net.
    # Each step is its own projection so the optimizer keeps the array
    # expressions small instead of inlining one into the next.
    receipt_key = F.col("receipt_id_ext")
    receipts = (
        receipts.join(drawn, "receipt_id_ext")
        .withColumn("_lines", with_complements(d, receipt_key, F.col("_lines")))
        .withColumn("_lines", merge_lines(F.col("_lines")))
        .withColumn(
            "_lines",
            price_basket(
                d,
                receipt_key,
                F.col("_lines"),
                F.month(F.col("event_date")),
                F.col("_offer"),
                F.round(F.col("tax_rate") * 10000).cast("long"),
                profile,
            ),
        )
    )

    fact_receipt_lines = (
        receipts.select("receipt_id_ext", "event_ts", "event_date", F.explode("_lines").alias("_l"))
        .select(
            "receipt_id_ext",
            "event_ts",
            "event_date",
            F.col("_l.line_num").alias("line_num"),
            F.col("_l.product_id").alias("product_id"),
            F.col("_l.quantity").alias("quantity"),
            _fmt(F.col("_l.unit_cents")).alias("unit_price"),
            F.col("_l.unit_cents").alias("unit_cents"),
            _fmt(F.col("_l.ext_cents")).alias("ext_price"),
            F.col("_l.ext_cents").alias("ext_cents"),
            F.col("_l.promo_code").alias("promo_code"),
        )
        .select(*column_names("fact_receipt_lines"))
    )

    # --- header rollup from the priced basket
    receipts = (
        receipts.withColumn("subtotal_cents", sum_lines(F.col("_lines"), "ext_cents"))
        .withColumn("discount_cents", sum_lines(F.col("_lines"), "discount_cents"))
        .withColumn("tax_cents", sum_lines(F.col("_lines"), "tax_cents"))
    )
    fact_receipts = (
        receipts.withColumn("total_cents", F.col("subtotal_cents") + F.col("tax_cents"))
        .withColumn("receipt_type", F.lit("SALE"))
        .withColumn("payment_method", F.col("tender_type"))
        # IMP-007: discount is applied before tax, so subtotal_cents (kept as
        # the existing net subtotal) plus discount_cents recovers the
        # pre-discount gross subtotal. attribution fields default NULL here;
        # attribution.py enriches the ~5% of receipts selected for attribution.
        .withColumn("gross_subtotal_cents", F.col("subtotal_cents") + F.col("discount_cents"))
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .withColumn("campaign_id", F.lit(None).cast("string"))
        .withColumn("impression_id_ext", F.lit(None).cast("string"))
        .select(
            "receipt_id_ext",
            "trace_id",
            "event_ts",
            "event_date",
            "store_id",
            "customer_id",
            "receipt_type",
            "tender_type",
            "subtotal_cents",
            _fmt(F.col("discount_cents")).alias("discount_amount"),
            "tax_cents",
            "total_cents",
            _fmt(F.col("subtotal_cents")).alias("subtotal_amount"),
            _fmt(F.col("tax_cents")).alias("tax_amount"),
            _fmt(F.col("total_cents")).alias("total_amount"),
            "payment_method",
            # Legacy semantic-model column (sourceColumn: Subtotal); same value
            # as subtotal_amount per the TMDL contract.
            _fmt(F.col("subtotal_cents")).alias("Subtotal"),
            "gross_subtotal_cents",
            "discount_cents",
            "attribution_journey_id",
            "campaign_id",
            "impression_id_ext",
            F.hour("event_ts").alias("local_hour"),
        )
        .select(*column_names("fact_receipts"))
    )

    # --- payments (one per receipt; in-store, so order_id_ext is NULL)
    fact_payments = (
        fact_receipts.withColumn("order_id_ext", F.lit(None).cast("string"))
        .withColumn("amount_cents", F.col("total_cents"))
        .withColumn(
            "transaction_id",
            F.concat(
                F.lit("TXN_"),
                F.unix_timestamp("event_ts").cast("string"),
                F.lit("_"),
                fixed_width_decimal(
                    d.h64(["receipt_id_ext"], "txn") % 1_000_000,
                    6,
                    "payment transaction suffix",
                ),
            ),
        )
        .withColumn("status", payment_status_column(d, ["receipt_id_ext"], F.col("payment_method")))
        .withColumn("decline_reason", decline_reason_column(d, ["receipt_id_ext"], F.col("status")))
        .withColumn(
            "processing_time_ms",
            processing_ms_column(d, ["receipt_id_ext"], F.col("payment_method")),
        )
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .select(*column_names("fact_payments"))
    )

    return {
        "fact_receipts": fact_receipts,
        "fact_receipt_lines": fact_receipt_lines,
        "fact_payments": fact_payments,
    }


"""Store ops + foot traffic facts, Spark-native.

fact_store_ops: exactly two rows (opened/closed) per store-day, with the
open/close hours parsed from `dim_stores.operating_hours` ("24h" or
"open-close", from the store type's profile). Dec 25 is skipped entirely
(stores closed for Christmas, any year).

Footfall uses the same independently generated visitors as receipt conversion
and counts zone passages, not shares: the store type's zones
(``vocabulary.ble_layout``), passed as in a visit's path
(``vocabulary.visit_zone_path``, shared with BLE and the live stream). Every
visitor passes the entrance and the exit, checkout counts the hour's buyers,
and each department zone the visitors who browse it.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F



BASE_CONVERSION = 0.20
PEAK_HOURS = (12, 13, 17, 18, 19)
PEAK_MULTIPLIER = 1.3
WEEKEND_MULTIPLIER = 0.9
# Baseline browsers per store-hour (scaled by store size + hour-of-day) so that
# open hours with zero receipts still emit foot traffic.
BASE_HOURLY_BROWSERS = 8


def generate_store_ops(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> DataFrame:
    """Two rows (opened/closed) per store-day, skipping Dec 25 of any year."""

    stores = dims["dim_stores"].select(F.col("ID").alias("store_id"), "operating_hours")
    store_ids = [r.store_id for r in stores.select("store_id").collect()]

    grid = (
        store_day_grid(spark, store_ids, cfg.start_date, cfg.end_date, cfg.seed, "store_ops")
        .filter(~((F.month("day") == 12) & (F.dayofmonth("day") == 25)))
        .join(stores, "store_id")
    )

    # operating_hours ("24h" or "open-close") -> open/close hour; unknown
    # formats fall back to standard hours instead of silent NULL event_ts
    open_hour, close_hour = _open_close_cols(F.col("operating_hours"))
    default_open, default_close = DEFAULT_OPERATING_WINDOW
    grid = grid.withColumn("open_hour", F.coalesce(open_hour, F.lit(default_open))).withColumn(
        "close_hour", F.coalesce(close_hour, F.lit(default_close))
    )

    day_ts = F.unix_timestamp(F.col("day").cast("timestamp"))
    open_ts = F.timestamp_seconds(day_ts + F.col("open_hour") * 3600)
    # 24h stores "close" at 23:59:00 the same day; others at close_hour:00
    close_secs = F.when(F.col("close_hour") == 24, F.lit(23 * 3600 + 59 * 60)).otherwise(
        F.col("close_hour") * 3600
    )
    close_ts = F.timestamp_seconds(day_ts + close_secs)

    ops = grid.select(
        "store_id",
        F.col("day").alias("event_date"),
        F.explode(
            F.array(
                F.struct(F.lit("opened").alias("operation_type"), open_ts.alias("event_ts")),
                F.struct(F.lit("closed").alias("operation_type"), close_ts.alias("event_ts")),
            )
        ).alias("op"),
    ).select(
        F.col("op.event_ts").alias("event_ts"),
        F.concat(
            F.lit("TRC-OPS-"),
            F.col("store_id"),
            F.lit("-"),
            F.col("event_date"),
            F.lit("-"),
            F.col("op.operation_type"),
        ).alias("trace_id"),
        F.col("store_id").cast("long").alias("store_id"),
        F.col("op.operation_type").alias("operation_type"),
        "event_date",
    )
    return ops.select(*column_names("fact_store_ops"))


def generate_foot_traffic(
    spark: SparkSession,
    receipts: DataFrame,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> DataFrame:
    """Per store-hour-zone passage counts of the hour's visitors and buyers."""

    d = seeded_draws(cfg.seed)

    del receipts
    profile = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type).profile
    layout = ble_layout(profile)
    shares = zone_passage_shares(layout)
    grid = generate_hourly_visits(spark, dims["dim_stores"], profile, cfg)

    zone_structs = [
        F.struct(
            F.lit(zone).alias("zone"),
            F.lit(shares.get(zone, -1.0)).alias("share"),
            F.lit(layout.dwell[zone][0]).alias("dwell_lo"),
            F.lit(layout.dwell[zone][1]).alias("dwell_hi"),
        )
        for zone in layout.zones
    ]
    ft = grid.select(
        "store_id",
        "hour_ts",
        "visitors",
        "n_receipts",
        F.explode(F.array(*zone_structs)).alias("z"),
    )
    visitors = F.col("visitors").cast("double")
    share = F.col("z.share")
    # Department passages: binomial counts (normal approximation) of the
    # hour's visitors; checkout counts the hour's buyers (share -1).
    spread = F.sqrt(visitors * share * (1.0 - share))
    browsed = F.least(
        visitors,
        F.greatest(
            F.lit(0.0),
            F.round(
                visitors * share
                + spread
                * d.gauss([F.col("store_id"), F.col("hour_ts"), F.col("z.zone")], "ft_count")
            ),
        ),
    )
    count = (
        F.when(share < 0, F.col("n_receipts").cast("double"))
        .when(share >= 1.0, visitors)
        .otherwise(browsed)
    )
    dwell_u = d.u([F.col("store_id"), F.col("hour_ts"), F.col("z.zone")], "ft_dwell")
    ft = ft.select(
        F.col("hour_ts").alias("event_ts"),
        F.concat(
            F.lit("TRC-FT-"),
            F.col("store_id"),
            F.lit("-"),
            F.date_format("hour_ts", "yyyyMMddHH"),
            F.lit("-"),
            F.col("z.zone"),
        ).alias("trace_id"),
        F.col("store_id").cast("long").alias("store_id"),
        sensor_id_column(F.col("store_id"), F.col("z.zone")).alias("sensor_id"),
        F.col("z.zone").alias("zone"),
        F.floor(F.col("z.dwell_lo") + dwell_u * (F.col("z.dwell_hi") - F.col("z.dwell_lo") + 1))
        .cast("long")
        .alias("dwell_seconds"),
        count.cast("long").alias("count"),
        F.to_date("hour_ts").alias("event_date"),
    )
    return ft.select(*column_names("fact_foot_traffic"))


"""Marketing impressions fact, Spark-native.

Ports datagen's marketing_campaign.py constants: four campaign archetypes,
each with its own channel mix and base impressions/day. Daily volumes are
scaled by `cfg.store_count / 86` — 86 is the legacy datagen fleet size, so a
fleet of 86 stores reproduces datagen's nominal volumes. `flash_sale` runs
only ~1 day in 7 (a per-day uniform gate, u < 1/7).

Per impression: channel uniform over the archetype's channels; device
MOBILE/DESKTOP/TABLET at 60/30/10 with cost multipliers 1.2/0.8/0.9; cost a
uniform draw within the channel's dollar band times the device multiplier;
a uniformly-assigned customer supplies `customer_ad_id` (dim_customers.AdId),
with `customer_id` populated for ~5% of impressions (CRM match) else NULL. A
CRM match needs the customer to be signed up and not yet churned that day
(generation.customers), so lapsed customers stop showing up as matched.
All draws via `runtime.seeded_draws`, so output is deterministic for a
(config, seed) pair regardless of cluster shape.

Schema note: fact_marketing is TMDL-bound with DUAL columns — PascalCase
`CustomerId`/`CostCents` (the TMDL sourceColumns) are exact mirrors of the
snake_case plan columns `customer_id`/`cost_cents`, and `__index_level_0__`
is the legacy pandas index (hash-derived via legacy_index).
"""

from datetime import date

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F


ARCHETYPES = CAMPAIGN_ARCHETYPES

LEGACY_FLEET_SIZE = 86  # datagen's store count; volumes scale store_count/86
# Background (non-journey) impressions carry a customer_id for this CRM-matched share.
CRM_MATCH_RATE = 0.05


def campaign_window_start(day: Column, history_start: date, duration: Column) -> Column:
    """Start of the campaign window a day falls in.

    Campaigns of an archetype run back to back for its duration, anchored on
    the first history day, so the live stream continues the same campaigns.
    """
    return F.date_add(
        F.lit(history_start),
        (F.datediff(day, F.lit(history_start)) / duration).cast("int") * duration,
    )


def impression_device_cost(
    draws: seeded_draws, key: list, channel: Column, device_salt: str, cost_salt: str
) -> tuple[Column, Column]:
    """(device, cost in dollars) of an impression.

    The device is MOBILE/DESKTOP/TABLET at 60/30/10; the cost is uniform in
    the channel's band times the device's multiplier.
    """
    device = draws.pick_by_weights(key, device_salt, DEVICE_WEIGHTS)
    multiplier = case_when((device == name, m) for name, m in DEVICE_MULTIPLIERS.items())
    low = case_when((channel == name, c_lo) for name, (c_lo, _) in CHANNEL_COSTS.items())
    high = case_when((channel == name, c_hi) for name, (_, c_hi) in CHANNEL_COSTS.items())
    return device, (low + draws.u(key, cost_salt) * (high - low)) * multiplier


def generate_marketing(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
    profile: StoreTypeProfile | None = None,
) -> DataFrame:
    """Ad-impression facts: day x archetype grid exploded to impressions."""

    d = seeded_draws(cfg.seed)
    if profile is None:
        profile = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type).profile
    lifecycle = lifecycle_for(cfg, profile.resolved_customer_visits_per_year())
    scale = cfg.store_count / LEGACY_FLEET_SIZE

    # --- Spark-native day x archetype grid
    rows = [
        (idx + 1, name, channels, base, dur)
        for idx, (name, channels, base, dur) in enumerate(ARCHETYPES)
    ]
    archetypes = spark.createDataFrame(
        rows,
        "archetype_idx int, archetype string, "
        "channels array<string>, base_impressions int, duration int",
    )
    grid = (
        date_grid(spark, cfg.start_date, cfg.end_date)
        .withColumn(
            "day_offset",
            F.datediff(F.col("day"), F.lit(cfg.start_date)),
        )
        .crossJoin(archetypes)
        .withColumn(
            # Campaigns span their archetype duration: the day's impressions
            # belong to the window [start + k*duration, ...) it falls in.
            "campaign_start",
            campaign_window_start(F.col("day"), cfg.start_date, F.col("duration")),
        )
    )

    # flash_sale runs ~1 day in 7 (uniform gate per day)
    grid = grid.filter(
        (F.col("archetype") != "flash_sale")
        | (d.u([F.col("day")], "mk_flash_gate") < F.lit(1.0 / 7.0))
    )

    # per-day impression count: clamped normal around the scaled base
    lam = F.col("base_impressions") * F.lit(scale)
    n_imps = F.greatest(
        F.lit(1),
        F.round(lam + d.gauss(["day", "archetype"], "mk_n") * F.sqrt(lam)),
    ).cast("int")
    grid = grid.withColumn("n_impressions", n_imps)

    # --- explode to impressions; build IDs
    imps = (
        grid.withColumn("seq", F.explode(F.sequence(F.lit(1), F.col("n_impressions"))))
        .withColumn(
            # campaign_id keyed on the campaign window so it spans multiple days
            "campaign_id",
            campaign_id_column(F.col("campaign_start"), F.col("archetype_idx")),
        )
        .withColumn(
            # impression id stays day-unique (campaign_id now repeats across days)
            "impression_id_ext",
            F.concat(
                F.lit("IMP"),
                F.date_format("day", "yyyyMMdd"),
                fixed_width_decimal(F.col("archetype_idx"), 2, "impression archetype index"),
                fixed_width_decimal(F.col("seq"), 7, "daily impression sequence"),
            ),
        )
        # drops the 'IMP' prefix: CREAT + day + archetype + seq
        .withColumn(
            "creative_id",
            F.concat(F.lit("CREAT"), F.substring("impression_id_ext", 4, 30)),
        )
    )

    key = [F.col("impression_id_ext")]

    # --- channel uniform over the archetype's channels
    ch_idx = F.floor(d.u(key, "mk_channel") * F.size("channels")).cast("int")
    imps = imps.withColumn("channel", F.element_at("channels", ch_idx + 1))

    # --- device pick (60/30/10); cost uniform in the channel band times the
    # device multiplier
    device, cost = impression_device_cost(d, key, F.col("channel"), "mk_device", "mk_cost")
    imps = imps.withColumn("device", device).withColumn("_cost_dollars", cost)
    cost_dollars = F.col("_cost_dollars")
    imps = imps.withColumn("cost_cents", F.round(cost_dollars * 100).cast("long")).withColumn(
        "cost", F.format_string("%.2f", cost_dollars)
    )

    # --- uniform customer per impression; ~5% CRM-matched (customer_id set).
    # A CRM match needs a customer who is signed up and not churned that day.
    imps = imps.withColumn(
        "customer_pick",
        (d.h64(key, "mk_customer") % F.lit(cfg.customer_count) + 1).cast("long"),
    )
    customers = dims["dim_customers"].select(
        F.col("ID").alias("customer_pick"), F.col("AdId").alias("customer_ad_id")
    )
    imps = imps.join(customers, "customer_pick", "left")
    imps = imps.withColumn(
        "customer_id",
        F.when(
            (d.u(key, "mk_crm") < F.lit(CRM_MATCH_RATE))
            & (activity_column(lifecycle, F.col("customer_pick"), F.col("day")) > 0),
            F.col("customer_pick").cast("double"),
        ),
    )

    # --- event_ts uniform within the day
    day_ts = F.unix_timestamp(F.col("day").cast("timestamp"))
    event_ts = F.timestamp_seconds(day_ts + F.floor(d.u(key, "mk_ts") * F.lit(86400)))

    out = (
        imps.select(
            event_ts.alias("event_ts"),
            F.concat(F.lit("TRC-MKT-"), F.col("impression_id_ext")).alias("trace_id"),
            "channel",
            "campaign_id",
            "creative_id",
            "customer_ad_id",
            "customer_id",
            "cost_cents",
            # TMDL-bound PascalCase duals: exact mirrors of the snake_case columns
            F.col("customer_id").alias("CustomerId"),
            F.col("cost_cents").alias("CostCents"),
            "impression_id_ext",
            "cost",
            "device",
            F.col("day").alias("event_date"),
        )
        .withColumn(
            "__index_level_0__",
            legacy_index("impression_id_ext"),
        )
        .withColumn(
            # IMP-007: NULL for organic/background impressions. attribution.py
            # unions in the two synthetic per-journey touches, which carry a
            # non-NULL attribution_journey_id.
            "attribution_journey_id",
            F.lit(None).cast("string"),
        )
    )
    return out.select(*column_names("fact_marketing"))


"""Deterministic marketing-attribution fact (IMP-007), Spark-native.

Builds ``ag.fact_marketing_attribution``: one row per in-store SALE receipt or
online order, including attributed, unattributed, declined, and
reconciliation-failed outcomes, using deterministic last-touch attribution
within an inclusive 7-day window.

Pipeline:

1. Purchase candidates are built from SALE receipts and online order headers,
   each left-joined to its own payment row. A purchase is "reconciled" only
   when it has an APPROVED payment whose ``amount_cents`` exactly equals the
   purchase's ``total_cents``; anything else is ``PAYMENT_DECLINED`` (a
   payment exists but is DECLINED) or ``RECONCILIATION_FAILED`` (no payment
   row at all — e.g. a cancelled online order — or an approved payment that
   doesn't reconcile to the total).
2. A deterministic, seed-bound ~5% share of the reconciled purchases is
   selected for attribution (``seeded_draws.u`` keyed on the purchase's own
   id, so selection is independent of cluster shape and stable for a given
   (config, seed)). Each selected purchase gets a unique
   ``attribution_journey_id`` and two synthetic ad-impression touches for the
   SAME customer/campaign/journey: an "older" touch (3-7 days before
   purchase) and a "newer" touch (1 minute-2 days before purchase). Both land
   inside the 7-day attribution window, and the two offset ranges never
   overlap, so the newer touch is always the more recent of the two.
3. The two touches are unioned into ``fact_marketing`` (customer_ad_id/
   customer_id populated, ``attribution_journey_id`` set). Last-touch
   attribution is then DERIVED — not copied — via ``_last_touch``: a real
   window/rank join keyed on ``touch_ts <= purchase_ts <= touch_ts + 7d``,
   ordered by ``touch_ts`` desc then ``impression_id_ext`` desc. Because the
   "newer" touch's timestamp is always later than the "older" touch's (by
   construction — the offset ranges never overlap), the newer touch always
   wins the rank; the winner is computed, not assumed, so a swapped
   offset/ordering bug would surface as a failing rank/tie-break test rather
   than silently passing.
4. All other reconciled purchases get ``UNATTRIBUTED_NO_JOURNEY`` (no journey
   was selected for them); ``attributed_revenue_cents`` is 0 for every
   non-ATTRIBUTED row and equals ``net_subtotal_cents`` (tax-exclusive) for
   ATTRIBUTED rows.
5. Selected receipts/orders/payments/promotions are enriched in place with
   the *derived* ``attribution_journey_id``/``campaign_id``/
   ``impression_id_ext`` — every other row keeps the NULL placeholders the
   base generators already emit.
"""

from datetime import date

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


ATTRIBUTION_MODEL = "LAST_TOUCH_7D"
ATTRIBUTION_WINDOW_DAYS = 7
SELECTION_RATE = 0.05  # deterministic bounded share (~5%) of eligible purchases

# Linked journeys reuse the product-launch and loyalty campaign families from
# marketing.py. Their 14/30-day durations provide real background spend while
# still allowing a conversion up to seven days after the selected touch.
JOURNEY_CAMPAIGNS = [
    (2, CAMPAIGN_ARCHETYPES[1][1], CAMPAIGN_ARCHETYPES[1][3]),
    (3, CAMPAIGN_ARCHETYPES[2][1], CAMPAIGN_ARCHETYPES[2][3]),
]

# Touch offsets (seconds before purchase_ts). The ranges never overlap
# (older_min = 3d > newer_max = 2d), so the "newer" touch is always the more
# recent one of the pair — attribution is still DERIVED by real ranking (see
# _last_touch); this just keeps the pipeline's own construction sane.
OLDER_OFFSET_MIN_S = 3 * 86400
OLDER_OFFSET_MAX_S = ATTRIBUTION_WINDOW_DAYS * 86400
NEWER_OFFSET_MIN_S = 60
NEWER_OFFSET_MAX_S = 2 * 86400


def _last_touch(touches: DataFrame, purchases: DataFrame) -> DataFrame:
    """Deterministic last-touch join: one output row per input purchase row.

    ``touches`` columns: match_key, touch_ts, impression_id_ext, campaign_id,
    creative_id, channel, customer_ad_id, customer_id.
    ``purchases`` columns: match_key, purchase_ts (any other columns are
    passed through unchanged on the output).

    For each purchase, selects the touch with
    ``touch_ts <= purchase_ts <= touch_ts + ATTRIBUTION_WINDOW_DAYS days``,
    ranked by the greatest ``touch_ts``, ties broken by ``impression_id_ext``
    descending — a real window/rank computation, not a lookup of a
    pre-known "right answer". Purchases with no qualifying touch keep all
    touch columns NULL (left join).
    """
    window_s = ATTRIBUTION_WINDOW_DAYS * 86400
    t = touches.select(
        F.col("match_key").alias("_t_match_key"),
        F.col("touch_ts"),
        F.col("impression_id_ext"),
        F.col("campaign_id"),
        F.col("creative_id"),
        F.col("channel"),
        F.col("customer_ad_id"),
        F.col("customer_id").alias("touch_customer_id"),
    )
    p_cols = purchases.columns
    cond = (
        (F.col("_t_match_key") == F.col("match_key"))
        & (F.col("touch_ts").cast("long") <= F.col("purchase_ts").cast("long"))
        & (F.col("touch_ts").cast("long") >= F.col("purchase_ts").cast("long") - F.lit(window_s))
    )
    joined = purchases.join(t, cond, "left")
    w = Window.partitionBy("match_key").orderBy(
        F.col("touch_ts").desc_nulls_last(),
        F.col("impression_id_ext").desc_nulls_last(),
    )
    ranked = (
        joined.withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn", "_t_match_key")
        .withColumn(
            "lag_seconds",
            F.when(
                F.col("touch_ts").isNotNull(),
                F.col("purchase_ts").cast("long") - F.col("touch_ts").cast("long"),
            ).cast("long"),
        )
    )
    return ranked.select(
        *p_cols,
        "touch_ts",
        "impression_id_ext",
        "campaign_id",
        "creative_id",
        "channel",
        "customer_ad_id",
        "touch_customer_id",
        "lag_seconds",
    )


def _journey_touch_rows(
    selected: DataFrame,
    d: seeded_draws,
    ts_col: str,
    suffix: str,
) -> DataFrame:
    """Build one fact_marketing row per selected journey for a single touch
    ("older" or "newer"). Device/cost draws mirror marketing.py's organic
    impressions, keyed on this touch's own impression id so the two touches
    of a journey draw independently of each other.
    """
    rows = selected.withColumn("touch_ts", F.col(ts_col)).withColumn(
        "impression_id_ext",
        F.concat(F.lit("IMPJRN"), F.col("_jhash"), F.lit(suffix[0].upper())),
    )
    key = [F.col("impression_id_ext")]
    device, cost = impression_device_cost(
        d, key, F.col("channel"), f"attr_device_{suffix}", f"attr_cost_{suffix}"
    )
    rows = rows.withColumn("device", device).withColumn("_cost_dollars", cost)
    cost_dollars = F.col("_cost_dollars")
    rows = (
        rows.withColumn("cost_cents", F.round(cost_dollars * 100).cast("long"))
        .withColumn("cost", F.format_string("%.2f", cost_dollars))
        .withColumn("trace_id", F.concat(F.lit("TRC-MKT-"), F.col("impression_id_ext")))
        .withColumn("creative_id", F.concat(F.col("creative_id"), F.lit(f"-{suffix.upper()}")))
        .withColumn("event_date", F.col("touch_ts").cast("date"))
    )
    out = rows.select(
        F.col("touch_ts").alias("event_ts"),
        "trace_id",
        "channel",
        "campaign_id",
        "creative_id",
        "customer_ad_id",
        F.col("customer_id").cast("double").alias("customer_id"),
        "cost_cents",
        F.col("customer_id").cast("double").alias("CustomerId"),
        F.col("cost_cents").alias("CostCents"),
        "impression_id_ext",
        "cost",
        "device",
        "event_date",
        "attribution_journey_id",
    ).withColumn("__index_level_0__", legacy_index("impression_id_ext"))
    return out.select(*column_names("fact_marketing"))


def with_journey_plan(purchases: DataFrame, d: seeded_draws, history_start: date) -> DataFrame:
    """Add a selected purchase's journey: ID, campaign, channel and two touch times.

    ``purchases`` needs ``match_key`` (the purchase's own ID) and
    ``purchase_ts``. Every draw is keyed on ``match_key``, so the batch and the
    live stream plan journeys the same way for their own purchases.
    """
    return (
        purchases.withColumn(
            "_jhash", F.hex(F.xxhash64(F.col("match_key"), F.lit(f"attr_journey|{d.seed}")))
        )
        .withColumn("attribution_journey_id", F.concat(F.lit("JRN"), F.col("_jhash")))
        .withColumn("creative_id", F.concat(F.lit("CREATJRN"), F.substring(F.col("_jhash"), 1, 12)))
        .withColumn(
            "_campaign_choice",
            (d.h64(["match_key"], "attr_campaign") % len(JOURNEY_CAMPAIGNS)).cast("int"),
        )
        .withColumn(
            "_campaign_index",
            F.when(F.col("_campaign_choice") == 0, F.lit(JOURNEY_CAMPAIGNS[0][0])).otherwise(
                F.lit(JOURNEY_CAMPAIGNS[1][0])
            ),
        )
        .withColumn(
            "_campaign_duration",
            F.when(F.col("_campaign_choice") == 0, F.lit(JOURNEY_CAMPAIGNS[0][2])).otherwise(
                F.lit(JOURNEY_CAMPAIGNS[1][2])
            ),
        )
        .withColumn(
            "channel",
            F.when(
                F.col("_campaign_choice") == 0,
                d.pick_by_weights(
                    ["match_key"],
                    "attr_channel_product",
                    [(channel, 1.0) for channel in JOURNEY_CAMPAIGNS[0][1]],
                ),
            ).otherwise(
                d.pick_by_weights(
                    ["match_key"],
                    "attr_channel_loyalty",
                    [(channel, 1.0) for channel in JOURNEY_CAMPAIGNS[1][1]],
                )
            ),
        )
        .withColumn(
            "older_offset",
            (
                F.lit(OLDER_OFFSET_MIN_S)
                + d.u(["match_key"], "attr_older") * F.lit(OLDER_OFFSET_MAX_S - OLDER_OFFSET_MIN_S)
            ).cast("long"),
        )
        .withColumn(
            "newer_offset",
            (
                F.lit(NEWER_OFFSET_MIN_S)
                + d.u(["match_key"], "attr_newer") * F.lit(NEWER_OFFSET_MAX_S - NEWER_OFFSET_MIN_S)
            ).cast("long"),
        )
        .withColumn(
            "touch_ts_older",
            F.timestamp_seconds(F.col("purchase_ts").cast("long") - F.col("older_offset")),
        )
        .withColumn(
            "touch_ts_newer",
            F.timestamp_seconds(F.col("purchase_ts").cast("long") - F.col("newer_offset")),
        )
        .withColumn("_campaign_touch_day", F.to_date("touch_ts_newer"))
        .withColumn(
            "_campaign_start",
            campaign_window_start(
                F.col("_campaign_touch_day"), history_start, F.col("_campaign_duration")
            ),
        )
        .withColumn(
            "campaign_id",
            campaign_id_column(F.col("_campaign_start"), F.col("_campaign_index")),
        )
    )


def generate_attribution(
    spark: SparkSession,
    t: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Derive fact_marketing_attribution and enrich receipts/orders/payments/
    promotions/marketing with the selected attribution mapping.

    Must run after fact_receipts, fact_receipt_lines, fact_online_order_headers,
    fact_payments, fact_promotions and the background fact_marketing all exist
    in ``t`` — wired in engine.py right after marketing.generate_marketing.
    """
    d = seeded_draws(cfg.seed)

    # --- 1. purchase candidates: SALE receipts + online orders, each
    # left-joined to its own payment.
    receipts = t["fact_receipts"].filter(F.col("receipt_type") == "SALE")
    payments = t["fact_payments"]
    store_pay = payments.filter(F.col("receipt_id_ext").isNotNull()).select(
        F.col("receipt_id_ext"),
        F.col("status").alias("payment_status"),
        F.col("amount_cents").alias("payment_cents"),
    )
    online_pay = payments.filter(F.col("order_id_ext").isNotNull()).select(
        F.col("order_id_ext"),
        F.col("status").alias("payment_status"),
        F.col("amount_cents").alias("payment_cents"),
    )

    store_purchases = receipts.join(store_pay, "receipt_id_ext", "left").select(
        F.col("receipt_id_ext").alias("match_key"),
        F.lit("STORE").alias("purchase_type"),
        F.col("receipt_id_ext"),
        F.lit(None).cast("string").alias("order_id_ext"),
        F.col("event_ts").alias("purchase_ts"),
        F.col("event_date"),
        F.col("store_id"),
        F.col("customer_id"),
        F.col("gross_subtotal_cents"),
        F.col("discount_cents"),
        F.col("subtotal_cents").alias("net_subtotal_cents"),
        F.col("tax_cents"),
        F.col("total_cents"),
        F.col("payment_status"),
        F.coalesce(F.col("payment_cents"), F.lit(0).cast("long")).alias("payment_cents"),
    )

    online_purchases = (
        t["fact_online_order_headers"]
        .join(online_pay, "order_id_ext", "left")
        .select(
            F.col("order_id_ext").alias("match_key"),
            F.lit("ONLINE").alias("purchase_type"),
            F.lit(None).cast("string").alias("receipt_id_ext"),
            F.col("order_id_ext"),
            F.col("event_ts").alias("purchase_ts"),
            F.col("event_date"),
            F.lit(None).cast("long").alias("store_id"),
            F.col("customer_id"),
            F.col("gross_subtotal_cents"),
            F.col("discount_cents"),
            F.col("subtotal_cents").alias("net_subtotal_cents"),
            F.col("tax_cents"),
            F.col("total_cents"),
            F.col("payment_status"),
            F.coalesce(F.col("payment_cents"), F.lit(0).cast("long")).alias("payment_cents"),
        )
    )

    purchases_all = store_purchases.unionByName(online_purchases)

    # --- 2. reconciliation status: an APPROVED payment whose amount exactly
    # equals the purchase total is the only path to attribution eligibility.
    reconciled = (F.col("payment_status") == "APPROVED") & (
        F.col("payment_cents") == F.col("total_cents")
    )
    purchases_all = purchases_all.withColumn(
        "base_status",
        F.when(F.col("payment_status").isNull(), F.lit("RECONCILIATION_FAILED"))
        .when(F.col("payment_status") == "DECLINED", F.lit("PAYMENT_DECLINED"))
        .when(~reconciled, F.lit("RECONCILIATION_FAILED"))
        .otherwise(F.lit("ELIGIBLE")),
    )

    # --- 3. deterministic ~5% selection of eligible (reconciled) purchases,
    # with a known customer to anchor the journey's touches to.
    earliest_purchase_ts = F.date_add(F.lit(cfg.start_date), ATTRIBUTION_WINDOW_DAYS).cast(
        "timestamp"
    )
    eligible = purchases_all.filter(
        (F.col("base_status") == "ELIGIBLE")
        & F.col("customer_id").isNotNull()
        # Keep generated touches inside the requested historical range.
        & (F.col("purchase_ts") >= earliest_purchase_ts)
    )
    selected = (
        eligible.withColumn("_sel_u", d.u(["match_key"], "attr_select"))
        .filter(F.col("_sel_u") < F.lit(SELECTION_RATE))
        .drop("_sel_u")
    )

    selected = with_journey_plan(selected, d, cfg.start_date)

    cust_ad = t["dim_customers"].select(
        F.col("ID").alias("customer_id"), F.col("AdId").alias("customer_ad_id")
    )
    selected = selected.join(cust_ad, "customer_id", "left")

    # --- 4. the two linked touches per journey, unioned into fact_marketing.
    older_rows = _journey_touch_rows(selected, d, "touch_ts_older", "older")
    newer_rows = _journey_touch_rows(selected, d, "touch_ts_newer", "newer")
    fact_marketing = t["fact_marketing"].unionByName(older_rows).unionByName(newer_rows)

    # --- 5. derive last-touch attribution via a real window/rank join,
    # scoped to each journey's own two touches (never mixed with organic
    # background impressions or other journeys — deterministic regardless of
    # catalog size/config) but still a genuine ranking computation.
    touches = older_rows.select(
        F.col("attribution_journey_id").alias("match_key"),
        F.col("event_ts").alias("touch_ts"),
        "impression_id_ext",
        "campaign_id",
        "creative_id",
        "channel",
        "customer_ad_id",
        "customer_id",
    ).unionByName(
        newer_rows.select(
            F.col("attribution_journey_id").alias("match_key"),
            F.col("event_ts").alias("touch_ts"),
            "impression_id_ext",
            "campaign_id",
            "creative_id",
            "channel",
            "customer_ad_id",
            "customer_id",
        )
    )
    purchase_for_rank = selected.select(
        F.col("attribution_journey_id").alias("match_key"), "purchase_ts"
    )
    best_touch = _last_touch(touches, purchase_for_rank).select(
        F.col("match_key").alias("attribution_journey_id"),
        "touch_ts",
        "impression_id_ext",
        "campaign_id",
        "creative_id",
        "channel",
        "customer_ad_id",
        "lag_seconds",
    )

    attributed = (
        selected.drop(
            "channel",
            "campaign_id",
            "creative_id",
            "customer_ad_id",
            "_jhash",
            "older_offset",
            "newer_offset",
            "touch_ts_older",
            "touch_ts_newer",
            "_campaign_choice",
            "_campaign_index",
            "_campaign_duration",
            "_campaign_touch_day",
            "_campaign_start",
        )
        .join(best_touch, "attribution_journey_id")
        .withColumn("attribution_status", F.lit("ATTRIBUTED"))
    )

    non_attributed = (
        purchases_all.join(selected.select("match_key"), "match_key", "left_anti")
        .withColumn(
            "attribution_status",
            F.when(F.col("base_status") == "ELIGIBLE", F.lit("UNATTRIBUTED_NO_JOURNEY")).otherwise(
                F.col("base_status")
            ),
        )
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .withColumn("touch_ts", F.lit(None).cast("timestamp"))
        .withColumn("impression_id_ext", F.lit(None).cast("string"))
        .withColumn("campaign_id", F.lit(None).cast("string"))
        .withColumn("creative_id", F.lit(None).cast("string"))
        .withColumn("channel", F.lit(None).cast("string"))
        .withColumn("customer_ad_id", F.lit(None).cast("string"))
        .withColumn("lag_seconds", F.lit(None).cast("long"))
    )

    combined = attributed.unionByName(non_attributed)
    fact_marketing_attribution = (
        combined.withColumn(
            "attribution_id",
            F.concat(
                F.lit("ATTR-"),
                F.col("purchase_type"),
                F.lit("-"),
                F.col("match_key"),
            ),
        )
        .withColumn("attribution_model", F.lit(ATTRIBUTION_MODEL))
        .withColumn("attribution_window_days", F.lit(ATTRIBUTION_WINDOW_DAYS).cast("int"))
        .withColumn(
            "attributed_revenue_cents",
            F.when(
                F.col("attribution_status") == "ATTRIBUTED", F.col("net_subtotal_cents")
            ).otherwise(F.lit(0).cast("long")),
        )
        .select(*column_names("fact_marketing_attribution"))
    )

    # --- 6. enrich receipts/orders/payments/promotions with the DERIVED
    # (not pre-assigned) journey/campaign/impression for attributed purchases.
    enrich_map = attributed.select(
        "match_key", "purchase_type", "attribution_journey_id", "campaign_id", "impression_id_ext"
    )
    store_enrich = enrich_map.filter(F.col("purchase_type") == "STORE").select(
        F.col("match_key").alias("receipt_id_ext"),
        "attribution_journey_id",
        "campaign_id",
        "impression_id_ext",
    )
    online_enrich = enrich_map.filter(F.col("purchase_type") == "ONLINE").select(
        F.col("match_key").alias("order_id_ext"),
        "attribution_journey_id",
        "campaign_id",
        "impression_id_ext",
    )
    pay_enrich = enrich_map.select(F.col("match_key").alias("pay_key"), "attribution_journey_id")

    fact_receipts = (
        t["fact_receipts"]
        .drop("attribution_journey_id", "campaign_id", "impression_id_ext")
        .join(store_enrich, "receipt_id_ext", "left")
        .select(*column_names("fact_receipts"))
    )
    fact_online_order_headers = (
        t["fact_online_order_headers"]
        .drop("attribution_journey_id", "campaign_id", "impression_id_ext")
        .join(online_enrich, "order_id_ext", "left")
        .select(*column_names("fact_online_order_headers"))
    )
    fact_payments = (
        t["fact_payments"]
        .drop("attribution_journey_id")
        .withColumn("pay_key", F.coalesce(F.col("receipt_id_ext"), F.col("order_id_ext")))
        .join(pay_enrich, "pay_key", "left")
        .drop("pay_key")
        .select(*column_names("fact_payments"))
    )
    fact_promotions = (
        t["fact_promotions"]
        .drop("attribution_journey_id")
        .join(
            store_enrich.select("receipt_id_ext", "attribution_journey_id"),
            "receipt_id_ext",
            "left",
        )
        .select(*column_names("fact_promotions"))
    )

    return {
        "fact_receipts": fact_receipts,
        "fact_online_order_headers": fact_online_order_headers,
        "fact_payments": fact_payments,
        "fact_promotions": fact_promotions,
        "fact_marketing": fact_marketing,
        "fact_marketing_attribution": fact_marketing_attribution,
    }


"""Causal, streaming inventory policy, independent of Spark.

Each SKU learns an EWMA of attempted daily demand (including lost sales).
There are no backorders: rejected whole lines have no stock commitment.
Inventory position includes outstanding deliveries and accepted restock returns.

Deliveries follow the store's delivery slot, as in the live stream: every SKU a
store reorders when a day closes ships on that store's next consolidated load,
which leaves the DC at the store's daily slot (``network.plan_routes``), drives
the lane and is unloaded on arrival. The SKU simulation only needs the slot and
the lane time, so it stays independent of other SKUs; ``inventory.py``
consolidates the reorders into capacity-bounded shipments and puts them on
trucks. The DC is an unlimited just-in-time supplier. This is a stock/service
simulation, not a vehicle-routing or supplier-capacity model.
"""

from dataclasses import dataclass
from datetime import date, datetime, timedelta
from datetime import time as inventory_time
from hashlib import sha256
from heapq import heappop, heappush
from math import ceil
from typing import Iterable, Iterator


CASE_PACK = 12


def stable_int(seed: int, *keys: object) -> int:
    """Stable across processes, Python hash seeds, and Spark partition counts."""
    value = "|".join(str(key) for key in (seed, *keys))
    return int.from_bytes(sha256(value.encode("utf-8")).digest()[:8], "big")


def opening_stock(seed: int, store_id: int, product_id: int, rate: float = 1.0) -> int:
    """Opening shelf stock for a SKU expected to sell ``rate`` units a day.

    A stable draw between the reorder point and the order-up-to target, so a
    store opens with 4-8 days of cover and its SKUs do not all reorder on the
    first day. ``rate`` is a prior from the demand model, never generated demand.
    """
    point, target = reorder_levels(rate)
    return point + 1 + stable_int(seed, store_id, product_id, "opening") % (target - point)


def delivery_slot_minute(seed: int, store_id: int) -> int:
    """Fallback daily delivery slot (minute of the day) of a store without a route."""
    return stable_int(seed, int(store_id), "slot") % 1440


def lane_transit_minutes(seed: int, dc_id: int, store_id: int) -> int:
    """One-way drive between a store and its DC (a stable draw per lane)."""
    span = TRUCK_TRANSIT_MINUTES_MAX - TRUCK_TRANSIT_MINUTES_MIN + 1
    return TRUCK_TRANSIT_MINUTES_MIN + stable_int(seed, int(dc_id), int(store_id), "lane") % span


# The replenishment policy, shared by history (``simulate_sku``) and the live
# stream (``stream_supply``).


def smooth_demand(rate: float, attempted: int) -> float:
    """Close a day: fold its attempted demand into the EWMA daily demand rate."""
    return 0.75 * rate + 0.25 * attempted


def decay_demand(rate: float, quiet_days: int) -> float:
    """EWMA rate after ``quiet_days`` closed days without any attempted demand."""
    return rate * 0.75 ** max(0, quiet_days)


def reorder_levels(rate: float) -> tuple[int, int]:
    """(reorder point, order-up-to target) for a smoothed daily demand ``rate``."""
    point = max(CASE_PACK, ceil(rate * 4))
    target = max(point + CASE_PACK, ceil(rate * 8))
    return point, target


def reorder_quantity(target: int, position: int) -> int:
    """Whole cases that lift the inventory position to the target."""
    return ceil((target - position) / CASE_PACK) * CASE_PACK


def reorder_priority(on_hand: int, rate: float) -> str:
    return "URGENT" if on_hand == 0 else ("HIGH" if on_hand < rate * 2 else "NORMAL")


@dataclass(frozen=True)
class Demand:
    event_ts: datetime
    receipt_id: str
    line_num: int
    quantity: int
    return_ts: datetime | None = None
    return_id: str | None = None

    @property
    def trace_id(self) -> str:
        return f"TRC-INV-{self.receipt_id}-{self.line_num}"


@dataclass
class InventoryEvent:
    store_id: int
    product_id: int
    event_ts: datetime
    trace_id: str
    kind: str
    quantity: int
    source: str
    balance: int = 0
    receipt_id: str | None = None
    line_num: int | None = None
    requested_quantity: int = 0
    unfulfilled_quantity: int = 0
    reorder_point: int | None = None
    priority: str | None = None
    shipment_id: str | None = None
    leg: int | None = None
    scheduled_ts: datetime | None = None
    loading_ts: datetime | None = None
    depart_ts: datetime | None = None
    eta: datetime | None = None
    unloading_ts: datetime | None = None
    etd: datetime | None = None


def shipment_base_id(dispatch: datetime, store_id: int) -> str:
    """A store's consolidated load of one dispatch day (``inventory.py`` adds the leg)."""
    return f"SHP-{dispatch:%Y%m%d}-{int(store_id):04d}"


def shipment_event(
    store_id: int,
    product_id: int,
    day: date,
    quantity: int,
    *,
    slot_minute: int,
    transit_minutes: int,
) -> InventoryEvent:
    """The SKU's line on its store's next load: the day after the reorder, at the slot.

    Timing depends only on the store's slot and lane, so every SKU the store
    reorders on ``day`` shares one dispatch and one arrival. Stock is received
    when unloading starts; ``inventory.py`` adds the unload duration (``etd``).
    """
    if not 0 <= slot_minute < 1440 or transit_minutes < 0:
        raise ValueError("invalid delivery slot or lane time")
    loading = datetime.combine(day + timedelta(days=1), inventory_time()) + timedelta(
        minutes=slot_minute
    )
    departure = loading + timedelta(minutes=TRUCK_LOAD_MINUTES)
    arrival = departure + timedelta(minutes=transit_minutes)
    shipment_id = shipment_base_id(loading, store_id)
    return InventoryEvent(
        store_id,
        product_id,
        datetime.combine(day, inventory_time(23, 59, 59)),
        f"TRC-PLAN-{shipment_id}-{product_id}",
        "SHIPMENT",
        quantity,
        shipment_id,
        shipment_id=shipment_id,
        leg=0,
        scheduled_ts=datetime.combine(day + timedelta(days=1), inventory_time()),
        loading_ts=loading,
        depart_ts=departure,
        eta=arrival,
        unloading_ts=arrival + timedelta(minutes=TRUCK_CHECKIN_MINUTES),
    )


def simulate_sku(
    demand: Iterable[Demand],
    *,
    store_id: int,
    product_id: int,
    start_date: date,
    end_date: date,
    seed: int,
    daily_prior: float = 1.0,
    slot_minute: int | None = None,
    transit_minutes: int | None = None,
) -> Iterator[InventoryEvent]:
    """Consume sorted demand with O(one SKU + pending deliveries/returns) state.

    ``daily_prior`` is the SKU's expected daily demand: it sizes the opening
    stock and starts the demand estimate, so history has no start-up stockout
    wave. ``slot_minute`` and ``transit_minutes`` are the store's delivery slot
    and lane (``network.plan_routes``); they default to the stable fallbacks.
    """
    if daily_prior < 0 or end_date < start_date:
        raise ValueError("invalid inventory simulation bounds")
    if slot_minute is None:
        slot_minute = delivery_slot_minute(seed, store_id)
    if transit_minutes is None:
        transit_minutes = lane_transit_minutes(seed, 0, store_id)
    on_hand = opening_stock(seed, store_id, product_id, daily_prior)
    yield InventoryEvent(
        store_id,
        product_id,
        datetime.combine(start_date, inventory_time()),
        f"TRC-INIT-ST-{store_id}-{product_id}",
        "INITIAL",
        on_hand,
        "SEED",
        balance=on_hand,
    )
    pending: list[tuple[datetime, str, InventoryEvent]] = []
    inbound = 0
    expected_returns = 0
    rate = daily_prior
    iterator = iter(demand)
    candidate = next(iterator, None)
    previous: tuple[datetime, str] | None = None
    day = start_date
    while day <= end_date:
        closing = datetime.combine(day, inventory_time(23, 59, 59))
        attempted = 0
        while candidate is not None and candidate.event_ts <= closing:
            key = (candidate.event_ts, candidate.trace_id)
            if candidate.event_ts.date() < start_date or (previous and key <= previous):
                raise ValueError("demand must have unique, chronological line keys")
            if candidate.quantity <= 0:
                raise ValueError("candidate demand quantity must be positive")
            previous = key
            while pending and pending[0][:2] <= key:
                _, _, event = heappop(pending)
                on_hand += event.quantity
                if event.kind == "RETURN":
                    expected_returns -= event.quantity
                else:
                    inbound -= event.quantity
                event.balance = on_hand
                yield event
            attempted += candidate.quantity
            fulfilled = candidate.quantity <= on_hand
            if fulfilled:
                on_hand -= candidate.quantity
            yield InventoryEvent(
                store_id,
                product_id,
                candidate.event_ts,
                candidate.trace_id,
                "SALE" if fulfilled else "LOST_SALE",
                -candidate.quantity if fulfilled else 0,
                "CUSTOMER_PURCHASE",
                balance=on_hand,
                receipt_id=candidate.receipt_id,
                line_num=candidate.line_num,
                requested_quantity=candidate.quantity,
                unfulfilled_quantity=0 if fulfilled else candidate.quantity,
            )
            if fulfilled and candidate.return_ts is not None:
                if candidate.return_ts <= candidate.event_ts or not candidate.return_id:
                    raise ValueError("restock returns must identify a later return")
                returned = InventoryEvent(
                    store_id,
                    product_id,
                    candidate.return_ts,
                    f"TRC-INV-{candidate.return_id}-{candidate.line_num}",
                    "RETURN",
                    candidate.quantity,
                    candidate.return_id,
                )
                heappush(pending, (returned.event_ts, returned.trace_id, returned))
                expected_returns += returned.quantity
            candidate = next(iterator, None)
        while pending and pending[0][0] <= closing:
            _, _, event = heappop(pending)
            on_hand += event.quantity
            if event.kind == "RETURN":
                expected_returns -= event.quantity
            else:
                inbound -= event.quantity
            event.balance = on_hand
            yield event
        rate = smooth_demand(rate, attempted)
        point, target = reorder_levels(rate)
        # Only near-term returns reduce the order. A distant return cannot
        # cover demand before the service lead time has elapsed.
        near_returns = (
            sum(
                event.quantity
                for ts, _, event in pending
                if event.kind == "RETURN" and ts <= closing + timedelta(days=4)
            )
            if expected_returns
            else 0
        )
        position = on_hand + inbound + near_returns
        if position <= point:
            quantity = reorder_quantity(target, position)
            priority = reorder_priority(on_hand, rate)
            yield InventoryEvent(
                store_id,
                product_id,
                closing,
                f"TRC-RO-{store_id}-{day}-{product_id}",
                "REORDER",
                quantity,
                "POLICY",
                balance=on_hand,
                reorder_point=point,
                priority=priority,
            )
            shipment = shipment_event(
                store_id,
                product_id,
                day,
                quantity,
                slot_minute=slot_minute,
                transit_minutes=transit_minutes,
            )
            yield shipment
            if shipment.unloading_ts is None:
                raise RuntimeError(f"shipment {shipment.source} has no unloading time")
            delivery = InventoryEvent(
                store_id,
                product_id,
                shipment.unloading_ts,
                f"TRC-SI-{shipment.source}-{product_id}",
                "INBOUND_SHIPMENT",
                shipment.quantity,
                shipment.source,
            )
            heappush(pending, (delivery.event_ts, delivery.trace_id, delivery))
            inbound += delivery.quantity
        day += timedelta(days=1)
    if candidate is not None:
        raise ValueError("candidate demand falls outside the simulation history")


"""Store-to-DC routing, delivery routes and DC truck fleets.

History (``inventory.py``) and the live stream (``stream_supply``) share all of
it, so both deliver the same way:

- **Routing.** Each store is served by a DC in its own state, else in its
  region, else any DC. Among those candidates the store goes to the DC serving
  the fewest stores so far (stores in ID order, ties to the lower DC ID), so DCs
  in one state share the load instead of the lowest ID serving everything.
- **Routes.** Each DC truck runs a fixed daily route: a set of stores in one
  time zone, each with a daily delivery slot. Slots on a route are spaced by
  the longest possible trip (load, the lane both ways, the maximum unload, a
  turnaround), so a truck on its route is never in two places at once. A
  store's reorders of one day leave on its next slot as one consolidated load.
- **Fleets.** ``dim_trucks`` gives each DC one truck per route (at least
  ``MIN_DC_FLEET``) plus a shared pool (``DCID`` NULL) for overflow loads; a DC
  without assigned trucks falls back to the pool.
- **Unloading.** Dwell is driven by load size and a per-store dock factor
  (``unload_minutes``), capped at ``TRUCK_UNLOAD_MINUTES_MAX``.
"""

from collections.abc import Iterable, Mapping
from dataclasses import dataclass
from datetime import datetime, time, timedelta, timezone
from zoneinfo import ZoneInfo

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


MIN_DC_FLEET = 2
POOL_TRUCK_SHARE = 0.15
MINUTES_PER_DAY = 1440


def assign_dcs(
    stores: Iterable[tuple[int, str, str]], dcs: Iterable[tuple[int, str, str]]
) -> dict[int, int]:
    """``{store_id: dc_id}``: same state, else same region, else any DC, least loaded.

    ``stores`` and ``dcs`` hold ``(id, state, region)``.
    """
    sites = sorted((int(dc), state, region) for dc, state, region in dcs)
    if not sites:
        raise ValueError("routing requires at least one distribution center")
    load = {dc: 0 for dc, _, _ in sites}
    assigned: dict[int, int] = {}
    for store_id, state, region in sorted((int(s), st, rg) for s, st, rg in stores):
        candidates = (
            [dc for dc, dc_state, _ in sites if dc_state == state]
            or [dc for dc, _, dc_region in sites if dc_region == region]
            or [dc for dc, _, _ in sites]
        )
        best = min(candidates, key=lambda dc: (load[dc], dc))
        load[best] += 1
        assigned[store_id] = best
    return assigned


@dataclass(frozen=True)
class DeliveryRoute:
    """A store's place on its DC's delivery routes."""

    dc_id: int
    route: int
    slot_minute: int
    transit_minutes: int


def route_budget_minutes(transit_minutes: int) -> int:
    """Longest possible round trip to a store, plus the turnaround."""
    return (
        TRUCK_LOAD_MINUTES
        + 2 * transit_minutes
        + TRUCK_UNLOAD_MINUTES_MAX
        + TRUCK_TURNAROUND_MINUTES
    )


def plan_routes(
    seed: int, store_dc: Mapping[int, int], store_zones: Mapping[int, str]
) -> dict[int, DeliveryRoute]:
    """Daily delivery routes of every DC, packed first-fit into trucks.

    Stores of one DC and time zone (in ID order) fill a truck's day with their
    round-trip budgets; a route starts at a stable minute and each store's slot
    follows the previous store's budget, wrapping around midnight. Slots are in
    the store's local time.
    """
    groups: dict[int, dict[str, list[int]]] = {}
    for store_id, dc_id in store_dc.items():
        zone = store_zones.get(int(store_id), "UTC")
        groups.setdefault(int(dc_id), {}).setdefault(zone, []).append(int(store_id))
    routes: dict[int, DeliveryRoute] = {}
    for dc_id in sorted(groups):
        index = 0
        for zone in sorted(groups[dc_id]):
            bins: list[list[tuple[int, int, int]]] = []
            used: list[int] = []
            for store_id in sorted(groups[dc_id][zone]):
                transit = lane_transit_minutes(seed, dc_id, store_id)
                budget = route_budget_minutes(transit)
                slot = next(
                    (b for b, minutes in enumerate(used) if minutes + budget <= MINUTES_PER_DAY),
                    None,
                )
                if slot is None:
                    bins.append([])
                    used.append(0)
                    slot = len(bins) - 1
                bins[slot].append((store_id, transit, budget))
                used[slot] += budget
            for offset, members in enumerate(bins):
                minute = stable_int(seed, dc_id, zone, offset, "route") % MINUTES_PER_DAY
                for store_id, transit, budget in members:
                    routes[store_id] = DeliveryRoute(dc_id, index + offset, minute, transit)
                    minute = (minute + budget) % MINUTES_PER_DAY
            index += len(bins)
    return routes


def routes_per_dc(routes: Mapping[int, DeliveryRoute]) -> dict[int, int]:
    """Number of daily routes each DC runs."""
    counts: dict[int, int] = {}
    for route in routes.values():
        counts[route.dc_id] = max(counts.get(route.dc_id, 0), route.route + 1)
    return counts


def fleet_sizes(dc_routes: Mapping[int, int], dc_ids: Iterable[int]) -> dict[int, int]:
    """Trucks assigned to each DC: one per route, at least ``MIN_DC_FLEET``."""
    return {int(dc): max(MIN_DC_FLEET, int(dc_routes.get(int(dc), 0))) for dc in sorted(dc_ids)}


def pool_size(assigned_trucks: int) -> int:
    """Shared pool trucks so that they are POOL_TRUCK_SHARE of the fleet."""
    return max(1, round(assigned_trucks * POOL_TRUCK_SHARE / (1 - POOL_TRUCK_SHARE)))


def route_truck(route: DeliveryRoute, fleets: Mapping[int, tuple[int, ...]]) -> int:
    """The DC truck that runs a store's route."""
    fleet = fleets[route.dc_id]
    return fleet[route.route % len(fleet)]


def next_slot_ms(since_ms: int, slot_minute: int, zone: str | None = None) -> int:
    """First instant at or after ``since_ms`` whose wall clock in ``zone`` is the slot."""
    tz = ZoneInfo(zone) if zone else timezone.utc
    local_day = datetime.fromtimestamp(since_ms / 1000, tz).date()
    for days in range(3):
        start = datetime.combine(local_day + timedelta(days=days), time(), tzinfo=tz)
        candidate = round((start + timedelta(minutes=slot_minute)).timestamp() * 1000)
        if candidate >= since_ms:
            return candidate
    raise RuntimeError("no delivery slot within two days")


def dock_factor(seed: int, store_id: int) -> float:
    """How slowly a store's dock unloads relative to the calibrated rate."""
    low, high = TRUCK_DOCK_FACTOR_RANGE
    return low + (high - low) * (stable_int(seed, int(store_id), "dock") % 1001) / 1000


def estimated_unload_minutes(seed: int, store_id: int, units: int) -> int:
    """Planned dwell of a load: base time plus the units at the site's rate."""
    handling = units / TRUCK_UNLOAD_UNITS_PER_MINUTE * dock_factor(seed, store_id)
    return min(
        TRUCK_UNLOAD_MINUTES_MAX,
        max(TRUCK_NORMAL_DWELL_MINUTES_MIN, round(TRUCK_UNLOAD_BASE_MINUTES + handling)),
    )


def unload_minutes(seed: int, store_id: int, units: int, shipment_id: str) -> int:
    """Actual dwell: the estimate plus a little noise per shipment, within the bounds."""
    span = 2 * TRUCK_UNLOAD_NOISE_MINUTES + 1
    noise = stable_int(seed, shipment_id, "unload") % span - TRUCK_UNLOAD_NOISE_MINUTES
    return min(
        TRUCK_UNLOAD_MINUTES_MAX,
        max(
            TRUCK_NORMAL_DWELL_MINUTES_MIN, estimated_unload_minutes(seed, store_id, units) + noise
        ),
    )


def _store_sites(
    dim_stores: DataFrame, dim_geographies: DataFrame, dim_distribution_centers: DataFrame
) -> tuple[list[tuple[int, str, str]], list[tuple[int, str, str]], dict[int, str]]:
    geo = {
        int(row["ID"]): (row["State"], row["Region"])
        for row in dim_geographies.select("ID", "State", "Region").collect()
    }
    stores, zones = [], {}
    for row in dim_stores.select("ID", "GeographyID", "TimeZone").collect():
        state, region = geo[int(row["GeographyID"])]
        stores.append((int(row["ID"]), state, region))
        zones[int(row["ID"])] = row["TimeZone"] or "UTC"
    dcs = []
    for row in dim_distribution_centers.select("ID", "GeographyID").collect():
        state, region = geo[int(row["GeographyID"])]
        dcs.append((int(row["ID"]), state, region))
    return stores, dcs, zones


def store_routes(
    seed: int,
    dim_stores: DataFrame,
    dim_geographies: DataFrame,
    dim_distribution_centers: DataFrame,
) -> dict[int, DeliveryRoute]:
    """Every store's DC and delivery route, from the (small) dimensions."""
    stores, dcs, zones = _store_sites(dim_stores, dim_geographies, dim_distribution_centers)
    return plan_routes(seed, assign_dcs(stores, dcs), zones)


def store_dc_map(
    dim_stores: DataFrame, dim_geographies: DataFrame, dim_distribution_centers: DataFrame
) -> DataFrame:
    """One ``(store_id, dc_id)`` row per store (see :func:`assign_dcs`)."""
    stores, dcs, _ = _store_sites(dim_stores, dim_geographies, dim_distribution_centers)
    spark = SparkSession.getActiveSession() or dim_stores.sparkSession
    rows = sorted(assign_dcs(stores, dcs).items())
    return spark.createDataFrame(rows, "store_id long, dc_id long").select(
        F.col("store_id"), F.col("dc_id")
    )


def dc_fleets(
    trucks: Iterable[tuple[int, int | float | None]], dc_ids: Iterable[int]
) -> dict[int, tuple[int, ...]]:
    """Truck IDs of each DC's fleet from ``(truck_id, DCID)`` pairs.

    A DC with no assigned trucks uses the pool trucks (``DCID`` is NULL).
    """
    assigned: dict[int, list[int]] = {}
    pool: list[int] = []
    for truck_id, dc_id in trucks:
        if dc_id is None:
            pool.append(int(truck_id))
        else:
            assigned.setdefault(int(dc_id), []).append(int(truck_id))
    fleets: dict[int, tuple[int, ...]] = {}
    for dc in sorted(int(dc) for dc in dc_ids):
        fleet = sorted(assigned.get(dc) or pool)
        if not fleet:
            raise ValueError(f"DC {dc} has neither assigned nor shared trucks")
        fleets[dc] = tuple(fleet)
    return fleets


"""Variable live baskets using the same catalog, mission, quantity and budget model."""

from dataclasses import dataclass
from datetime import date

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F



@dataclass(frozen=True)
class BasketCatalog:
    """Department lookup and basket calibration for one catalog date."""

    lookup: DataFrame
    line_value: float
    taxable_value: float


def prepare_basket_catalog(
    spark: SparkSession,
    products: DataFrame,
    profile: StoreTypeProfile,
    seed: int,
    catalog_date: date,
    anchor_date: date,
) -> BasketCatalog:
    """Build the live catalog as of ``catalog_date`` once per stream run.

    ``products`` is ``baskets.basket_products`` of ``dim_products``. Launch
    eligibility, shelf prices (``pricing``, anchored on ``anchor_date`` as in
    history) and the seasonal department weights all come from
    ``catalog_date``, not the clock, so a replayed micro-batch rebuilds the
    same baskets. The stream keeps the date in its checkpoint incarnation
    record and moves it forward on a restart that has no batch to replay.
    """
    draws = seeded_draws(seed)
    catalog = retail_catalog(
        products,
        spark.range(1).select(F.lit(catalog_date).alias("event_date")),
        draws,
        PriceSchedule(seed, anchor_date),
    )
    weight = sum(
        (
            F.when(
                F.col("department") == name,
                F.lit(value) * _seasonal_factor(name, F.month("event_date")),
            ).otherwise(0.0)
            for name, value in profile.department_weights.items()
        ),
        F.lit(0.0),
    )
    # Department arrays are bounded profile metadata, not a collected customer pool.
    entries = catalog.withColumn("_weight", weight).orderBy("department").collect()
    if not entries:
        raise ValueError("No launched products available for live baskets")
    total = sum(row["_weight"] for row in entries)
    missing = set(profile.department_weights) - {row["department"] for row in entries}
    if missing:
        raise ValueError(
            f"Live catalog has no launched products for departments: {sorted(missing)}"
        )
    cumulative = 0.0
    metadata = []
    for row in entries:
        cumulative += row["_weight"] / total
        metadata.append((row["department"], cumulative, row["_catalog"]))
    schema = (
        "department string, _cdf double, _catalog array<struct<_cdf:double,"
        "product_id:bigint,SalePrice:double,taxability:string,unit_cents:bigint,"
        "partner_id:bigint,partner_cents:bigint,partner_taxability:string,partner_ok:boolean>>"
    )
    lookup = spark.createDataFrame(metadata, schema).agg(
        F.sort_array(F.collect_list(F.struct("_cdf", "department", "_catalog"))).alias(
            "_departments"
        )
    )
    line_value = sum(row["_weight"] * row["_expected_line_value"] for row in entries) / total
    taxable_value = sum(row["_weight"] * row["_expected_taxable_value"] for row in entries) / total
    return BasketCatalog(lookup, line_value, taxable_value)


def attach_baskets(
    purchases: DataFrame,
    catalog: BasketCatalog,
    profile: StoreTypeProfile,
    seed: int,
    complement_rate: float = 0.0,
) -> DataFrame:
    """Attach ``_offer``, ``basket_n`` and ``basket_lines`` to purchase rows.

    Lines carry the catalog date's shelf price and the SKU's complementary
    partner; ``complement_rate`` sizes baskets that will bring complements
    along (``baskets.with_complements``, store purchases).

    Rows need ``receipt_id``, ``customer_id`` (NULL for anonymous shoppers)
    and ``store_bps``. Budgets and favorite departments are the customer's,
    or the purchase's own for anonymous shoppers, as in history.
    """
    draws = seeded_draws(seed)
    purchases = (
        purchases.join(F.broadcast(catalog.lookup))
        .withColumn("_shopper", shopper_key(F.col("customer_id"), F.col("receipt_id")))
        .withColumn("_offer", receipt_offer(draws, F.col("receipt_id"), profile.promo_rate))
        .withColumn(
            "basket_n",
            retail_basket_size(
                draws,
                ["receipt_id"],
                F.col("_shopper"),
                calibrated_basket_mean(
                    F.lit(catalog.line_value),
                    F.lit(catalog.taxable_value),
                    F.col("store_bps") / 10000.0,
                    profile.avg_ticket_target,
                    profile.promo_rate,
                    profile.basket_lambda,
                    complement_rate,
                ),
                F.col("_offer"),
                profile.promo_rate,
            ),
        )
    )

    def line(index):
        keys = [F.col("receipt_id"), index]
        department_draw = F.when(
            draws.u(keys, "mission-coherence") < 0.60,
            F.when(
                draws.u(["receipt_id"], "favorite-mission") < 0.50,
                draws.u(["_shopper"], "favorite-department"),
            ).otherwise(draws.u(["receipt_id"], "mission-department")),
        ).otherwise(draws.u(keys, "dept"))
        department = pick_retail_product(F.col("_departments"), department_draw)
        product = pick_retail_product(department["_catalog"], draws.u(keys, "prod"))
        quantity = retail_quantity(draws, keys, department["department"])
        unit = product["unit_cents"]
        return F.struct(
            index.alias("line_num"),
            product["product_id"].alias("product_id"),
            quantity.alias("quantity"),
            unit.alias("unit_cents"),
            (unit * quantity).alias("gross_cents"),
            product["taxability"].alias("taxability"),
            department["department"].alias("department"),
            product["partner_id"].alias("partner_id"),
            product["partner_cents"].alias("partner_cents"),
            product["partner_taxability"].alias("partner_taxability"),
            product["partner_ok"].alias("partner_ok"),
        )

    return purchases.withColumn(
        "basket_lines", F.transform(F.sequence(F.lit(1), F.col("basket_n")), line)
    ).drop("_departments", "_shopper")


def attach_stream_baskets(
    spark: SparkSession,
    purchases: DataFrame,
    products: DataFrame,
    profile: StoreTypeProfile,
    seed: int,
    catalog_date: date,
    anchor_date: date,
) -> DataFrame:
    """Attach baskets drawn from the catalog as of ``catalog_date`` (one-off form)."""
    catalog = prepare_basket_catalog(spark, products, profile, seed, catalog_date, anchor_date)
    return attach_baskets(purchases, catalog, profile, seed)


"""Run control for the live stream: checkpoints, Kusto write retries and restarts.

Pure Python with no Spark or Fabric imports. ``build_notebooks.py`` inlines it
into ``stream-events``, and the tests call it directly with fakes, so the
notebook's operational behavior is checked offline.
"""

import json
import re
import time
import uuid
from collections.abc import Callable
from dataclasses import dataclass, replace
from datetime import date, datetime, timedelta, timezone
from typing import Any, Protocol

# Event types declared end to end (KQL, mappings, EVENT_PAYLOADS, Silver) for
# the logistics incident model, which the live stream does not emit yet.
# Readiness doesn't expect fresh rows for them. Shrink this set as the model
# lands; tests/test_notebook_build.py checks it against the slots.
PENDING_EVENT_TYPES = frozenset({"logistics_incident_started", "logistics_incident_resolved"})

# Live tables that are quiet by design. Live volume follows the store hours and
# the historical configuration, so a healthy stream can leave these tables
# without new rows for a while: store operations happen only at opening and
# closing, store visits (and the stock moves, reorders and stockouts they
# cause) stop overnight while every store is closed, trucks run only when
# consolidated loads are due, online orders reach delivery a day or more after
# they are placed, and a small fleet has few online orders, impressions or
# promotions per hour. Readiness adds this quiet time to its freshness window
# for the table.
EVENT_TABLE_QUIET = {
    "store_opened": timedelta(hours=26),
    "store_closed": timedelta(hours=26),
    "customer_entered": timedelta(hours=12),
    "ble_ping_detected": timedelta(hours=12),
    "customer_zone_changed": timedelta(hours=12),
    "receipt_created": timedelta(hours=12),
    "receipt_line_added": timedelta(hours=12),
    "promotion_applied": timedelta(hours=12),
    "payment_processed": timedelta(hours=12),
    "online_order_created": timedelta(hours=12),
    "online_order_picked": timedelta(hours=12),
    "online_order_shipped": timedelta(hours=12),
    "online_order_line_added": timedelta(hours=12),
    "online_order_delivered": timedelta(hours=36),
    "inventory_updated": timedelta(hours=12),
    "reorder_triggered": timedelta(hours=12),
    "stockout_detected": timedelta(hours=24),
    "truck_arrived": timedelta(hours=12),
    "truck_departed": timedelta(hours=12),
    "ad_impression": timedelta(hours=6),
}


def event_table_max_age(event_type: str, base: timedelta) -> timedelta:
    """Freshness window of a live event table: ``base`` plus its quiet time."""

    return base + EVENT_TABLE_QUIET.get(event_type, timedelta(0))


# Live runs trigger a micro-batch every minute. Each batch makes one Kusto
# transaction per event table, so a longer trigger means fewer, larger extents.
DEFAULT_TRIGGER_SECONDS = 60
# Bounded (smoke) runs are short, so they trigger more often to show data quickly.
SMOKE_TRIGGER_SECONDS = 10
# After a stop, the rate source replays the whole downtime in one batch. Rows
# older than this many seconds behind the batch time are dropped instead.
DEFAULT_MAX_LAG_SECONDS = 600

# Kusto writes: attempts per table write, with exponential backoff between them.
WRITE_RETRY_ATTEMPTS = 5
WRITE_RETRY_BASE_SECONDS = 2.0
WRITE_RETRY_MAX_SECONDS = 60.0

# Supervised restarts of a failed live query. The counter resets once a
# restarted query has stayed healthy for RESTART_RESET_AFTER_SECONDS.
MAX_QUERY_RESTARTS = 5
RESTART_BASE_SECONDS = 30.0
RESTART_MAX_SECONDS = 600.0
RESTART_RESET_AFTER_SECONDS = 3600.0

_TRANSIENT_ERROR = re.compile(
    r"throttl"
    r"|too\s*many\s*requests"
    r"|\b(?:status|http|code)\b\D{0,8}(?:401|408|429|500|502|503|504)\b"
    r"|service\s*unavailable|server\s*busy|bad\s*gateway|gateway\s*time-?out"
    r"|internal\s*server\s*error"
    r"|timed?\s*out|timeout"
    r"|temporar(?:il)?y"
    r"|connection\s+(?:reset|refused|aborted|closed)"
    r"|token.{0,40}expir|expir.{0,40}token"
    r"|unauthori[sz]ed",
    re.IGNORECASE,
)


def stream_checkpoint_name(sink: str, run_seconds: int) -> str:
    """Return the sink checkpoint folder; bounded runs never share live offsets."""

    return f"{sink}-smoke" if int(run_seconds) > 0 else f"{sink}-live"


def trigger_interval_seconds(run_seconds: int, trigger_seconds: int) -> int:
    """Return the processing-time trigger for a live or bounded run."""

    if int(trigger_seconds) < 1:
        raise ValueError("trigger_seconds must be at least 1")
    if int(run_seconds) > 0:
        return min(int(trigger_seconds), SMOKE_TRIGGER_SECONDS)
    return int(trigger_seconds)


def _stream_error_text(exc: BaseException) -> str:
    parts = []
    seen: set[int] = set()
    current: BaseException | None = exc
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        parts.append(f"{type(current).__name__}: {current}")
        current = current.__cause__ or current.__context__
    return " | ".join(parts)


def is_transient_error(exc: BaseException) -> bool:
    """True for throttling, 5xx, timeouts, dropped connections and expired tokens."""

    return bool(_TRANSIENT_ERROR.search(_stream_error_text(exc)))


def backoff_seconds(attempt: int, base: float, ceiling: float) -> float:
    """Exponential backoff for the given 1-based retry number, capped at ``ceiling``."""

    return float(min(ceiling, base * 2 ** max(0, attempt - 1)))


def _stream_error_summary(exc: BaseException, limit: int = 300) -> str:
    text = " ".join(str(exc).split())
    return text if len(text) <= limit else text[: limit - 3] + "..."


def call_with_retry(
    operation: Callable[[int], Any],
    *,
    label: str,
    attempts: int = WRITE_RETRY_ATTEMPTS,
    base_seconds: float = WRITE_RETRY_BASE_SECONDS,
    max_seconds: float = WRITE_RETRY_MAX_SECONDS,
    is_transient: Callable[[BaseException], bool] = is_transient_error,
    sleep: Callable[[float], None] = time.sleep,
    log: Callable[[str], None] = print,
) -> Any:
    """Call ``operation(attempt)``, retrying transient failures with backoff.

    A permanent error, or a transient one on the last attempt, propagates
    unchanged so the caller fails the micro-batch and Spark does not commit it.
    """

    for attempt in range(1, attempts + 1):
        try:
            return operation(attempt)
        except Exception as exc:
            if attempt >= attempts or not is_transient(exc):
                raise
            delay = backoff_seconds(attempt, base_seconds, max_seconds)
            log(
                f"{label}: transient failure on attempt {attempt}/{attempts} "
                f"({type(exc).__name__}: {_stream_error_summary(exc)}); retrying in {delay:.0f}s"
            )
            sleep(delay)
    raise AssertionError("unreachable")  # pragma: no cover


def run_supervised(
    start_query: Callable[[], Any],
    *,
    run_seconds: int,
    max_restarts: int = MAX_QUERY_RESTARTS,
    base_seconds: float = RESTART_BASE_SECONDS,
    max_seconds: float = RESTART_MAX_SECONDS,
    healthy_after_seconds: float = RESTART_RESET_AFTER_SECONDS,
    is_restartable: Callable[[BaseException], bool] = is_transient_error,
    sleep: Callable[[float], None] = time.sleep,
    clock: Callable[[], float] = time.monotonic,
    log: Callable[[str], None] = print,
) -> int:
    """Run the streaming query; restart a failed live query a bounded number of times.

    A bounded run (``run_seconds > 0``) runs once and is stopped after its
    duration. A live run is restarted after a transient failure, with backoff,
    from the same checkpoint, so the failed micro-batch replays with the same
    tags and IDs. Permanent failures, and failures beyond ``max_restarts``,
    propagate. Returns the number of restarts used.
    """

    if int(run_seconds) > 0:
        query = start_query()
        try:
            query.awaitTermination(int(run_seconds))
        finally:
            query.stop()
        return 0

    restarts = 0
    while True:
        started = clock()
        query = start_query()
        try:
            query.awaitTermination()
            log(f"stream query stopped after {restarts} restart(s)")
            return restarts
        except Exception as exc:
            if clock() - started >= healthy_after_seconds:
                restarts = 0
            if restarts >= max_restarts or not is_restartable(exc):
                log(
                    f"stream query failed and will not be restarted "
                    f"(restarts used: {restarts}/{max_restarts}): {_stream_error_summary(exc)}"
                )
                raise
            restarts += 1
            delay = backoff_seconds(restarts, base_seconds, max_seconds)
            log(
                f"stream query failed ({type(exc).__name__}: {_stream_error_summary(exc)}); "
                f"restart {restarts}/{max_restarts} in {delay:.0f}s"
            )
            sleep(delay)


# ---------------------------------------------------------------------------
# Checkpoint incarnations
# ---------------------------------------------------------------------------
#
# The checkpoint root holds ``_metadata/stream_id`` and one Spark checkpoint per
# sink and run kind (``eventhouse-live``, ``eventhouse-smoke``, ...). Each Spark
# checkpoint has an incarnation record, ``_metadata/<checkpoint>.json``, with a
# random nonce that goes into every Kusto ingest tag and generated business ID,
# and the catalog date used to build live baskets. Spark restarts batch IDs at
# 0 when its checkpoint is deleted; the record makes that state visible, so a
# reset can never reuse the tags or IDs of an earlier incarnation.

_INCARNATION_PATTERN = re.compile(r"^[0-9a-f]{8}$")
INGEST_TAG_PREFIX = "retail-demo"


class CheckpointFs(Protocol):
    """The subset of ``mssparkutils.fs`` the checkpoint helpers use."""

    def exists(self, path: str) -> bool: ...

    def head(self, path: str, max_bytes: int) -> str: ...

    def put(self, path: str, content: str, overwrite: bool) -> bool: ...

    def mkdirs(self, path: str) -> bool: ...

    def ls(self, path: str) -> list[Any]: ...


class CheckpointStateError(RuntimeError):
    """The checkpoint folders disagree; starting would reuse or lose event identities."""


@dataclass(frozen=True)
class StreamIncarnation:
    """The identity and catalog snapshot a stream checkpoint runs with."""

    stream_id: str
    checkpoint: str
    incarnation: str
    catalog_date: date
    record_path: str
    # The batch Spark will replay first (offsets written, commit missing), if any.
    pending_batch: int | None
    # True when the record must be written (new incarnation or new catalog date).
    needs_write: bool
    resumed: bool

    @property
    def namespace(self) -> str:
        """Prefix for generated business IDs and trace IDs."""

        return f"{self.stream_id}-{self.incarnation}"

    def record(self) -> dict[str, str]:
        return {
            "stream_id": self.stream_id,
            "checkpoint": self.checkpoint,
            "incarnation": self.incarnation,
            "catalog_date": self.catalog_date.isoformat(),
        }


def ingest_tag(stream_id: str, incarnation: str, event_type: str, batch_id: int) -> str:
    """Kusto ingest-by tag for one table write of one micro-batch."""

    return f"{INGEST_TAG_PREFIX}:{stream_id}:{incarnation}:{event_type}:{int(batch_id)}"


def parse_ingest_tag(tag: str) -> dict[str, Any] | None:
    """Parse a stream ingest tag; Kusto may prefix it with ``ingest-by:``.

    Returns ``stream_id``, ``incarnation`` (None for tags written before
    incarnations existed), ``event_type`` and ``batch_id``, or None.
    """

    marker = tag.find(f"{INGEST_TAG_PREFIX}:")
    if marker < 0:
        return None
    parts = tag[marker:].split(":")
    if len(parts) == 5:
        _prefix, stream_id, incarnation, event_type, batch = parts
    elif len(parts) == 4:
        _prefix, stream_id, event_type, batch = parts
        incarnation = None
    else:
        return None
    if not batch.isdigit() or not stream_id or not event_type:
        return None
    return {
        "stream_id": stream_id,
        "incarnation": incarnation,
        "event_type": event_type,
        "batch_id": int(batch),
    }


def spark_log_batches(fs: CheckpointFs, directory: str) -> list[int]:
    """Batch IDs in a Spark ``offsets`` or ``commits`` log (ignores .crc/tmp files)."""

    if not fs.exists(directory):
        return []
    names = (str(getattr(entry, "name", entry)).rstrip("/") for entry in fs.ls(directory))
    return sorted(int(name) for name in names if name.isdigit())


def _new_incarnation() -> str:
    return uuid.uuid4().hex[:8]


def plan_incarnation(
    fs: CheckpointFs,
    root: str,
    checkpoint: str,
    stream_id: str,
    today: date,
    *,
    new_nonce: Callable[[], str] = _new_incarnation,
) -> StreamIncarnation:
    """Decide the incarnation a stream checkpoint starts with, without writing it.

    - No record and no Spark log: a new incarnation with today's catalog date.
    - Record and offsets: resume. A pending batch (offsets without a commit)
      replays with the recorded catalog date, so it regenerates the same
      baskets. With no pending batch the catalog date moves to today, so new
      launches enter the live catalog on restart.
    - Anything else (a record without offsets, offsets without a record,
      commits without offsets, or a record for another stream) is refused.
    """

    root = root.rstrip("/")
    record_path = f"{root}/_metadata/{checkpoint}.json"
    location = f"{root}/{checkpoint}"
    offsets = spark_log_batches(fs, f"{location}/offsets")
    commits = spark_log_batches(fs, f"{location}/commits")
    has_record = fs.exists(record_path)

    if commits and (not offsets or commits[-1] > offsets[-1]):
        raise CheckpointStateError(
            f"Spark checkpoint {location!r} has commits past its offsets log; it is "
            f"inconsistent. Delete {location!r} and {record_path!r} to start a new "
            "incarnation."
        )
    if not has_record:
        if offsets or commits:
            raise CheckpointStateError(
                f"Spark checkpoint {location!r} has offsets but no incarnation record "
                f"{record_path!r}, so the tags and IDs it already used are unknown. "
                f"Delete {location!r} to start a new incarnation."
            )
        incarnation = new_nonce()
        if not _INCARNATION_PATTERN.fullmatch(incarnation):
            raise ValueError(f"invalid incarnation nonce {incarnation!r}")
        return StreamIncarnation(
            stream_id, checkpoint, incarnation, today, record_path, None, True, False
        )

    record = json.loads(fs.head(record_path, 65536))
    if record.get("stream_id") != stream_id or record.get("checkpoint") != checkpoint:
        raise CheckpointStateError(
            f"Incarnation record {record_path!r} belongs to stream "
            f"{record.get('stream_id')!r} / {record.get('checkpoint')!r}, not "
            f"{stream_id!r} / {checkpoint!r}. Delete the whole checkpoint root to reset."
        )
    incarnation = str(record.get("incarnation", ""))
    if not _INCARNATION_PATTERN.fullmatch(incarnation):
        raise CheckpointStateError(f"Incarnation record {record_path!r} has no valid nonce")
    if not offsets:
        raise CheckpointStateError(
            f"Spark checkpoint {location!r} has no offsets log, but its incarnation "
            f"record {record_path!r} exists: the checkpoint was reset or never ran a "
            f"batch. Delete {record_path!r} as well to start a new incarnation with new "
            "tags and IDs, or restore the checkpoint to resume."
        )
    recorded_date = date.fromisoformat(str(record["catalog_date"]))
    pending = offsets[-1] if not commits or offsets[-1] > commits[-1] else None
    catalog_date = recorded_date if pending is not None else today
    return StreamIncarnation(
        stream_id,
        checkpoint,
        incarnation,
        catalog_date,
        record_path,
        pending,
        catalog_date != recorded_date,
        True,
    )


def persist_incarnation(fs: CheckpointFs, plan: StreamIncarnation) -> StreamIncarnation:
    """Write the incarnation record when the plan needs it; call just before starting."""

    if not plan.needs_write:
        return plan
    payload = {**plan.record(), "written_at": datetime.now(timezone.utc).isoformat()}
    fs.mkdirs(plan.record_path.rsplit("/", 1)[0])
    content = json.dumps(payload, sort_keys=True)
    if plan.resumed:
        fs.put(plan.record_path, content, True)
        return replace(plan, needs_write=False)
    try:
        created = fs.put(plan.record_path, content, False)
    except Exception:
        if not fs.exists(plan.record_path):
            raise
        created = False
    if not created:
        raise CheckpointStateError(
            f"Another run created {plan.record_path!r} while this one was starting; "
            "run only one stream per checkpoint."
        )
    return replace(plan, needs_write=False)


# ---------------------------------------------------------------------------
# History-to-stream handoff
# ---------------------------------------------------------------------------

# History ends yesterday (store-local), so the newest historical receipt is
# normally less than a day before the stream starts.
HANDOFF_MAX_GAP = timedelta(days=1)
# Local "yesterday" can end up to 14 hours after UTC midnight, so the newest
# historical receipt may legitimately be slightly after the stream start.
HANDOFF_OVERLAP_TOLERANCE = timedelta(hours=14)


class HandoffError(RuntimeError):
    """History and the live stream would leave a gap or overlap in the facts."""


@dataclass(frozen=True)
class HandoffCheck:
    ok: bool
    message: str


def check_history_handoff(
    history_end: datetime | None,
    stream_start: datetime,
    *,
    max_gap: timedelta = HANDOFF_MAX_GAP,
    overlap_tolerance: timedelta = HANDOFF_OVERLAP_TOLERANCE,
) -> HandoffCheck:
    """Compare the newest historical receipt time with the stream start (both UTC)."""

    if history_end is None:
        return HandoffCheck(
            False, "Silver fact_receipts has no rows, so there is no history to continue."
        )
    if history_end.tzinfo is None or stream_start.tzinfo is None:
        raise ValueError("history_end and stream_start must be timezone-aware UTC times")
    start = stream_start.astimezone(timezone.utc).isoformat(timespec="seconds")
    end = history_end.astimezone(timezone.utc).isoformat(timespec="seconds")
    if history_end > stream_start + overlap_tolerance:
        return HandoffCheck(
            False,
            f"History runs until {end}, after the stream start {start}: live events "
            "would duplicate the historical volume for the overlap.",
        )
    if history_end > stream_start:
        return HandoffCheck(
            True,
            f"History ends at {end}, slightly after the stream start {start} "
            "(store time zones); the overlap is within tolerance.",
        )
    gap = stream_start - history_end
    if gap > max_gap:
        return HandoffCheck(
            False,
            f"History ends at {end}, {gap.total_seconds() / 86400:.1f} days before the "
            f"stream start {start}: Silver and Gold facts will have a gap. Re-render "
            "and rerun setup so history ends yesterday.",
        )
    return HandoffCheck(True, "")


def enforce_history_handoff(
    check: HandoffCheck,
    *,
    resumed: bool,
    allow_gap: bool,
    log: Callable[[str], None] = print,
) -> None:
    """Fail a new stream on a bad handoff unless overridden; only warn on a resume.

    A resumed checkpoint continues its own events, so a gap after a stop is
    expected (the backlog cap skips the downtime) and is only reported.
    """

    if check.ok:
        if check.message:
            log(f"history handoff: {check.message}")
        return
    if resumed or allow_gap:
        reason = "resumed checkpoint" if resumed else "allow_history_gap=True"
        log(f"WARNING history handoff ({reason}): {check.message}")
        return
    raise HandoffError(f"{check.message} Set allow_history_gap=True to start anyway.")


"""Spark helpers for the live stream: bounded catch-up and existing-ID sampling."""

from pyspark.sql import Column, DataFrame, Window
from pyspark.sql import functions as F


def cap_backlog(
    frame: DataFrame, max_lag_seconds: int, timestamp_col: str = "timestamp"
) -> DataFrame:
    """Drop source rows more than ``max_lag_seconds`` older than the micro-batch time.

    After a stop, Spark's ``rate`` source emits every row for the downtime in
    the next micro-batch. This filter keeps only the newest ``max_lag_seconds``
    of them, so a resumed stream continues from about now. In a streaming
    query ``current_timestamp()`` is the batch timestamp Spark stores in the
    offset log, so a replayed batch keeps exactly the same rows.
    """

    if int(max_lag_seconds) < 1:
        raise ValueError("max_lag_seconds must be at least 1")
    floor = F.current_timestamp() - F.expr(f"INTERVAL {int(max_lag_seconds)} SECONDS")
    return frame.where(F.col(timestamp_col) >= floor)


def dense_id_lookup(dimension: DataFrame, id_col: str = "ID") -> tuple[DataFrame, int]:
    """Number a dimension's distinct, non-null IDs 0..n-1 and return (lookup, n).

    The lookup has ``_idx`` and ``_id`` columns. It is persisted because the
    stream joins it on every micro-batch. Gaps in the IDs (a custom layout or a
    filtered publish) are fine: rows only ever receive an ID that exists.
    """

    lookup = (
        dimension.select(F.col(id_col).cast("long").alias("_id"))
        .where(F.col("_id").isNotNull())
        .distinct()
        .withColumn("_idx", (F.row_number().over(Window.orderBy("_id")) - 1).cast("long"))
        .persist()
    )
    return lookup, lookup.count()


def attach_sampled_id(
    frame: DataFrame,
    key: Column,
    salt: str,
    seed: int,
    lookup: DataFrame,
    count: int,
    output: str,
) -> DataFrame:
    """Add ``output``: an existing ID picked deterministically per row from ``lookup``.

    The pick is ``xxhash64(key, "salt|seed") mod count``, the same draw the
    stream used for contiguous ``1..count`` ranges, so a dimension with IDs
    1..n gets exactly the IDs it got before.
    """

    if int(count) < 1:
        raise ValueError(f"cannot sample {output}: the ID lookup is empty")
    index = f"_{output}_idx"
    choice = lookup.select(F.col("_idx").alias(index), F.col("_id").alias(output))
    return (
        frame.withColumn(index, F.pmod(F.xxhash64(key, F.lit(f"{salt}|{seed}")), F.lit(int(count))))
        .join(F.broadcast(choice), index, "inner")
        .drop(index)
    )


"""Versioned state for the live stream, kept in Lakehouse Delta tables.

Stateful stream features (store open/closed, in-progress customer visits)
persist their state between micro-batches in ``<silver>._stream_state_<name>``.
Each table holds one row per entity version::

    (incarnation, batch_id, entity_key, payload, deleted)

Micro-batch N reads, for every entity, its latest version written before N,
and writes the versions it changes with ``batch_id = N``. The write first
replaces every row of the incarnation with ``batch_id >= N``, so when Spark
replays batch N (its commit was not recorded) the replay reads exactly the
state the first attempt read, regenerates identical events, and overwrites the
first attempt's versions instead of applying them twice.

State belongs to a checkpoint incarnation: a new incarnation finds no rows
and seeds its state once (``batch_id = -1``) from current Silver values.

The live stream keeps these state tables:

- ``_stream_state_stores``: each store's open/closed state and last change
- ``_stream_state_visits``: customer visits still in progress
- ``_stream_state_orders``: online orders whose lifecycle is still running
- ``_stream_state_inventory``: on-hand, inbound and smoothed demand of each
  store and product the stream has touched (``stream_supply``)
- ``_stream_state_logistics``: pending loads, shipments on the road and truck
  positions (``stream_supply``)
- ``_stream_state_meta``: the clock of the last micro-batch and the
  incarnation's inventory seed time

The transitions are pure Python over plain dicts (``apply_*_batch``), so they
are unit-tested without Spark. :class:`MemoryStateStore` keeps the rows in
memory with the same semantics as :class:`DeltaStateStore`, which is the
Fabric adapter. ``build_notebooks.py`` inlines this module into the stream
notebook.
"""

import json
from collections.abc import Callable, Iterable
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Any, Protocol, TypeVar

from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F


STATE_TABLE_PREFIX = "_stream_state_"
STATE_NAMES = ("stores", "visits", "orders", "inventory", "logistics", "meta")
SEED_BATCH_ID = -1
STATE_SCHEMA = (
    "incarnation string, batch_id long, entity_key string, payload string, deleted boolean"
)
# Superseded versions are deleted every this many batches.
COMPACT_EVERY_BATCHES = 60

StateMap = dict[str, dict[str, Any]]
Changes = dict[str, dict[str, Any] | None]


@dataclass(frozen=True)
class StateRow:
    incarnation: str
    batch_id: int
    entity_key: str
    payload: str | None
    deleted: bool = False


def encode_payload(value: dict[str, Any]) -> str:
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def decode_payload(text: str) -> dict[str, Any]:
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("state payload must be a JSON object")
    return value


def state_as_of(rows: Iterable[StateRow], incarnation: str, batch_id: int) -> StateMap:
    """State a batch sees: each entity's latest version written before ``batch_id``."""
    latest: dict[str, StateRow] = {}
    for row in rows:
        if row.incarnation != incarnation or row.batch_id >= batch_id:
            continue
        current = latest.get(row.entity_key)
        if current is None or row.batch_id > current.batch_id:
            latest[row.entity_key] = row
    return {
        key: decode_payload(row.payload)
        for key, row in sorted(latest.items())
        if not row.deleted and row.payload is not None
    }


def state_changes(before: StateMap, after: StateMap) -> Changes:
    """Entities whose value changed (``None`` marks a deletion)."""
    changes: Changes = {key: None for key in before if key not in after}
    changes.update({key: value for key, value in after.items() if before.get(key) != value})
    return dict(sorted(changes.items()))


def version_rows(incarnation: str, batch_id: int, changes: Changes) -> list[StateRow]:
    return [
        StateRow(
            incarnation,
            int(batch_id),
            key,
            None if value is None else encode_payload(value),
            value is None,
        )
        for key, value in sorted(changes.items())
    ]


def replace_batch(
    rows: Iterable[StateRow], incarnation: str, batch_id: int, new_rows: Iterable[StateRow]
) -> list[StateRow]:
    """Rows after writing batch ``batch_id``: drop the incarnation's ``>= batch_id`` rows first."""
    kept = [
        row for row in rows if not (row.incarnation == incarnation and row.batch_id >= batch_id)
    ]
    return kept + list(new_rows)


def superseded_rows(rows: Iterable[StateRow], incarnation: str, batch_id: int) -> list[StateRow]:
    """Versions before ``batch_id`` that no read at or after ``batch_id`` can see.

    An older version is hidden by a newer one written before ``batch_id``, and
    a deletion marker that is the latest version hides nothing.
    """
    latest: dict[str, StateRow] = {}
    candidates = [row for row in rows if row.incarnation == incarnation and row.batch_id < batch_id]
    for row in candidates:
        current = latest.get(row.entity_key)
        if current is None or row.batch_id > current.batch_id:
            latest[row.entity_key] = row
    return [row for row in candidates if latest[row.entity_key] is not row or row.deleted]


def replace_condition(incarnation: str, batch_id: int) -> str:
    """Delta ``replaceWhere`` predicate of a batch write."""
    if not incarnation.replace("-", "").replace("_", "").replace(".", "").isalnum():
        raise ValueError(f"invalid state incarnation {incarnation!r}")
    return f"incarnation = '{incarnation}' AND batch_id >= {int(batch_id)}"


Emitted = TypeVar("Emitted")


def apply_batch(
    state: StateMap,
    inputs: Any,
    transition: Callable[[StateMap, Any], tuple[StateMap, Emitted]],
) -> tuple[StateMap, Emitted]:
    """Run one micro-batch transition on a copy of ``state``: ``(new_state, emitted)``."""
    return transition({key: dict(value) for key, value in state.items()}, inputs)


# ---------------------------------------------------------------------------
# Store open/closed state
# ---------------------------------------------------------------------------

# A store's open/close events are replayed at most this far back, for example
# after a long stop or when history ended days before the stream started.
MAX_STORE_CATCHUP = timedelta(days=1)
_OPERATIONS = ("opened", "closed")


@dataclass(frozen=True)
class StoreOperation:
    store_id: int
    event_time: datetime
    operation_type: str


@dataclass(frozen=True)
class StoreBatch:
    stores: tuple[StoreHours, ...]
    clock: datetime
    previous_clock: datetime | None = None


def _iso(value: datetime) -> str:
    return value.astimezone(timezone.utc).isoformat()


def _parse(value: str) -> datetime:
    parsed = datetime.fromisoformat(value)
    return parsed if parsed.tzinfo else parsed.replace(tzinfo=timezone.utc)


def seed_store_state(
    stores: Iterable[StoreHours],
    last_operations: dict[int, tuple[datetime, str]],
    now: datetime,
) -> StateMap:
    """Initial store state of a new incarnation.

    A store's last historical operation (``fact_store_ops``) decides whether it
    is open; a store without one starts from its operating hours at ``now``.
    """
    seeded: StateMap = {}
    for store in stores:
        last = last_operations.get(store.store_id)
        operation = None if last is None else str(last[1]).strip().lower()
        if last is not None and operation in _OPERATIONS:
            event_time = last[0] if last[0].tzinfo else last[0].replace(tzinfo=timezone.utc)
            seeded[str(store.store_id)] = {
                "is_open": operation == "opened",
                "changed_at": _iso(event_time),
            }
        else:
            seeded[str(store.store_id)] = {
                "is_open": is_open_at(store, now),
                "changed_at": _iso(now),
            }
    return seeded


def apply_store_batch(state: StateMap, batch: StoreBatch) -> tuple[StateMap, list[StoreOperation]]:
    """Emit each store's open/close boundaries up to the batch clock.

    Boundaries come from the store's local operating hours. An operation that
    would not change the store's state is skipped, so every store alternates
    opened, closed, opened, ... and each open day yields one paired
    opened/closed. Only the window since the last batch (and at most
    :data:`MAX_STORE_CATCHUP`) is scanned.
    """
    new_state = dict(state)
    emitted: list[StoreOperation] = []
    floor = batch.clock - MAX_STORE_CATCHUP
    for store in sorted(batch.stores, key=lambda item: item.store_id):
        key = str(store.store_id)
        current = state.get(key)
        if current is None:
            current = {"is_open": is_open_at(store, floor), "changed_at": _iso(floor)}
            new_state[key] = current
        is_open = bool(current["is_open"])
        lower = max(_parse(current["changed_at"]), floor)
        if batch.previous_clock is not None:
            lower = max(lower, batch.previous_clock)
        changed = None
        for instant, operation in store_boundaries(store, lower, batch.clock):
            if (operation == "opened") == is_open:
                continue
            is_open = operation == "opened"
            changed = instant
            emitted.append(StoreOperation(store.store_id, instant, operation))
        if changed is not None:
            new_state[key] = {"is_open": is_open, "changed_at": _iso(changed)}
    return new_state, sorted(emitted, key=lambda op: (op.event_time, op.store_id))


# ---------------------------------------------------------------------------
# Customer visit state
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class VisitBatch:
    """Visits known to a batch: ``visits[key] = (state payload, visit end in ms)``."""

    clock_ms: int
    visits: dict[str, tuple[dict[str, Any], int]]


def apply_visit_batch(state: StateMap, batch: VisitBatch) -> tuple[StateMap, list[str]]:
    """Keep visits that end after the batch clock; return the keys that finished.

    A visit's events are emitted as their time passes, so a visit stays in
    state until its last event (its end) is at or before the batch clock.
    """
    new_state = {
        key: payload
        for key, (payload, end_ms) in sorted(batch.visits.items())
        if end_ms > batch.clock_ms
    }
    finished = sorted(key for key in batch.visits if key not in new_state)
    for key in state:
        if key not in batch.visits:
            new_state[key] = state[key]
    return dict(sorted(new_state.items())), finished


# ---------------------------------------------------------------------------
# Adapters
# ---------------------------------------------------------------------------


class StateStore(Protocol):
    incarnation: str

    def read(self, name: str, batch_id: int) -> StateMap: ...

    def read_keys(self, name: str, batch_id: int, keys: Iterable[str]) -> StateMap: ...

    def write(self, name: str, batch_id: int, changes: Changes) -> None: ...

    def has_rows(self, name: str) -> bool: ...

    def compact(self, name: str, batch_id: int) -> None: ...


class MemoryStateStore:
    """In-memory state store with the Delta store's semantics (tests, offline runs)."""

    def __init__(self, incarnation: str) -> None:
        self.incarnation = incarnation
        self.rows: dict[str, list[StateRow]] = {}

    def read(self, name: str, batch_id: int) -> StateMap:
        return state_as_of(self.rows.get(name, []), self.incarnation, batch_id)

    def read_keys(self, name: str, batch_id: int, keys: Iterable[str]) -> StateMap:
        wanted = set(keys)
        rows = [row for row in self.rows.get(name, []) if row.entity_key in wanted]
        return state_as_of(rows, self.incarnation, batch_id)

    def write(self, name: str, batch_id: int, changes: Changes) -> None:
        self.rows[name] = replace_batch(
            self.rows.get(name, []),
            self.incarnation,
            batch_id,
            version_rows(self.incarnation, batch_id, changes),
        )

    def has_rows(self, name: str) -> bool:
        return any(row.incarnation == self.incarnation for row in self.rows.get(name, []))

    def compact(self, name: str, batch_id: int) -> None:
        stale = set(superseded_rows(self.rows.get(name, []), self.incarnation, batch_id))
        self.rows[name] = [row for row in self.rows.get(name, []) if row not in stale]


class DeltaStateStore:
    """State tables ``<database>._stream_state_<name>`` in the Lakehouse (Delta)."""

    def __init__(self, spark: SparkSession, database: str, incarnation: str) -> None:
        replace_condition(incarnation, 0)
        self.spark = spark
        self.database = database
        self.incarnation = incarnation
        self._ready: set[str] = set()

    def table(self, name: str) -> str:
        return f"{self.database}.{STATE_TABLE_PREFIX}{name}"

    def _ensure(self, name: str) -> str:
        table = self.table(name)
        if name not in self._ready:
            self.spark.sql(
                f"CREATE TABLE IF NOT EXISTS {table} (incarnation STRING, batch_id BIGINT, "
                "entity_key STRING, payload STRING, deleted BOOLEAN) USING DELTA"
            )
            self._ready.add(name)
        return table

    def _versions(self, name: str, batch_id: int, keys: DataFrame | None = None) -> DataFrame:
        rows = self.spark.table(self._ensure(name)).where(
            (F.col("incarnation") == self.incarnation) & (F.col("batch_id") < int(batch_id))
        )
        if keys is not None:
            rows = rows.join(F.broadcast(keys), "entity_key")
        newest = Window.partitionBy("entity_key").orderBy(F.col("batch_id").desc())
        return rows.withColumn("_version", F.row_number().over(newest))

    def _latest(self, versions: DataFrame) -> StateMap:
        latest = versions.where((F.col("_version") == 1) & ~F.col("deleted"))
        return {
            row["entity_key"]: decode_payload(row["payload"])
            for row in latest.select("entity_key", "payload").collect()
        }

    def read(self, name: str, batch_id: int) -> StateMap:
        return self._latest(self._versions(name, batch_id))

    def read_keys(self, name: str, batch_id: int, keys: Iterable[str]) -> StateMap:
        """Latest versions of only ``keys`` (large tables such as store inventory)."""
        wanted = sorted(set(keys))
        if not wanted:
            return {}
        frame = self.spark.createDataFrame([(key,) for key in wanted], "entity_key string")
        return self._latest(self._versions(name, batch_id, frame))

    def write(self, name: str, batch_id: int, changes: Changes) -> None:
        rows = version_rows(self.incarnation, batch_id, changes)
        frame = self.spark.createDataFrame(
            [(r.incarnation, r.batch_id, r.entity_key, r.payload, r.deleted) for r in rows],
            STATE_SCHEMA,
        )
        (
            frame.write.format("delta")
            .mode("overwrite")
            .option("replaceWhere", replace_condition(self.incarnation, batch_id))
            .saveAsTable(self._ensure(name))
        )

    def has_rows(self, name: str) -> bool:
        table = self.spark.table(self._ensure(name))
        return not table.where(F.col("incarnation") == self.incarnation).limit(1).isEmpty()

    def compact(self, name: str, batch_id: int) -> None:
        if int(batch_id) <= 0 or int(batch_id) % COMPACT_EVERY_BATCHES:
            return
        stale = (
            self._versions(name, batch_id)
            .where((F.col("_version") > 1) | F.col("deleted"))
            .select("incarnation", "batch_id", "entity_key")
        )
        view = f"_stale_{STATE_TABLE_PREFIX}{name}"
        stale.createOrReplaceTempView(view)
        try:
            self.spark.sql(
                f"MERGE INTO {self.table(name)} AS t USING {view} AS s "
                "ON t.incarnation = s.incarnation AND t.batch_id = s.batch_id "
                "AND t.entity_key = s.entity_key WHEN MATCHED THEN DELETE"
            )
        finally:
            self.spark.catalog.dropTempView(view)


"""Live store inventory and truck logistics as one pure state machine.

Each micro-batch replays, in time order, everything that happens to store
stock and the DC fleets between the previous batch clock and this one:

- **Demand.** Every emitted in-store receipt line and every store-fulfilled
  online order line (ship-from-store, BOPIS) takes stock from its store. A line
  the store cannot fill becomes a lost sale instead, as in history: the stock
  never goes negative and the line is dropped from the receipt or order.
- **Stockouts and reorders.** A sale that takes the balance to zero reports a
  stockout (once per store, product and UTC day, like ``fact_stockouts``).
  After each sale or lost sale the shared replenishment policy
  (``inventory_policy``: EWMA demand, reorder point, order-up-to target, case
  pack, priority) checks the inventory position, on hand plus inbound.
- **Shipments.** A reorder joins its store's pending load at the store's DC.
  Each store takes one consolidated delivery a day: at the store's daily
  delivery slot the next free truck of the DC's fleet loads,
  drives the lane, arrives (planned arrival plus delay, zero until the
  incident model adds delays), unloads for a load- and site-driven dwell,
  departs and drives back. A truck carries one shipment at a time, so it is
  never in two places at once. Arrivals and departures are emitted only when
  their time has passed, and the unload adds the units to store stock.

State lives in two tables. ``inventory`` holds one entry per store and
product that the stream has touched (``inv:<store>:<product>``); entries the
stream has not touched yet are seeded from the Silver ledger as of the
incarnation start. ``logistics`` holds pending loads (``pending:<store>``),
shipments on the road (``shipment:<id>``) and truck positions
(``truck:<id>``); a truck with no entry is idle at its DC.

Everything here is pure Python: :func:`apply_supply_batch` maps a state and a
:class:`SupplyBatch` to the next state and the emitted records, so a replayed
batch emits exactly the same events. ``build_notebooks.py`` inlines this
module into the stream notebook.
"""

import heapq
from collections.abc import Callable
from copy import deepcopy
from dataclasses import dataclass, field
from datetime import date, datetime, timezone
from typing import Any


INVENTORY_STATE = "inventory"
LOGISTICS_STATE = "logistics"
STORE_SALE_SOURCE = "CUSTOMER_PURCHASE"
ONLINE_SALE_SOURCE = "ONLINE_ORDER"
_MINUTE_MS = 60_000
_DAY_MS = 86_400_000
# Same-instant events: arrivals, then unloads, then demand, then dispatching.
_ARRIVE, _UNLOAD, _DEMAND, _DISPATCH = range(4)

SupplyState = dict[str, dict[str, Any]]


def inventory_key(store_id: int, product_id: int) -> str:
    return f"inv:{int(store_id)}:{int(product_id)}"


def parse_inventory_key(key: str) -> tuple[int, int]:
    _, store, product = key.split(":")
    return int(store), int(product)


def inventory_entry(on_hand: int, rate: float) -> dict[str, Any]:
    """A store-product entry with no live history: balance and smoothed daily demand."""
    return {
        "on_hand": max(0, int(on_hand)),
        "inbound": 0,
        "rate": float(rate),
        "day": None,
        "attempted": 0,
        "stockout_day": None,
    }


def default_inventory_entry(
    seed: int, store_id: int, product_id: int, daily_prior: float
) -> dict[str, Any]:
    """Entry for a store-product the Silver ledger has never seen (history's opening stock)."""
    return inventory_entry(opening_stock(seed, store_id, product_id, daily_prior), daily_prior)


def split_supply_state(state: SupplyState) -> tuple[SupplyState, SupplyState]:
    """(inventory, logistics) entries of a combined supply state."""
    inventory = {key: value for key, value in state.items() if key.startswith("inv:")}
    logistics = {key: value for key, value in state.items() if not key.startswith("inv:")}
    return inventory, logistics


@dataclass(frozen=True)
class SupplyNetwork:
    """Static supply inputs: store routing, DC fleets, delivery slots and the demand prior.

    ``slots`` and ``zones`` hold each store's delivery slot (minute of the local
    day, from ``network.plan_routes`` as in history) and IANA time zone; a store
    without them ships at a stable UTC minute.
    """

    seed: int
    namespace: str
    store_dc: dict[int, int]
    fleets: dict[int, tuple[int, ...]]
    daily_prior: float
    slots: dict[int, int] = field(default_factory=dict)
    zones: dict[int, str] = field(default_factory=dict)

    def dc_of(self, store_id: int) -> int:
        return self.store_dc[int(store_id)]

    def transit_minutes(self, store_id: int) -> int:
        """One-way drive between a store and its DC (a stable draw per lane)."""
        return lane_transit_minutes(self.seed, self.dc_of(store_id), int(store_id))

    def delivery_slot_minute(self, store_id: int) -> int:
        """Minute of the store's day at which its pending load ships."""
        slot = self.slots.get(int(store_id))
        return delivery_slot_minute(self.seed, store_id) if slot is None else int(slot)

    def next_slot_ms(self, store_id: int, since_ms: int) -> int:
        """The store's first delivery slot at or after ``since_ms``."""
        return next_slot_ms(
            since_ms, self.delivery_slot_minute(store_id), self.zones.get(int(store_id))
        )

    def dock_factor(self, store_id: int) -> float:
        """How slowly a store's dock unloads relative to the calibrated rate."""
        return dock_factor(self.seed, store_id)

    def estimated_unload_minutes(self, store_id: int, units: int) -> int:
        """Planned dwell of a load: base time plus the units at the site's rate."""
        return estimated_unload_minutes(self.seed, store_id, units)

    def unload_minutes(self, store_id: int, units: int, shipment_id: str) -> int:
        """Actual dwell: the estimate plus a little noise per shipment."""
        return unload_minutes(self.seed, store_id, units, shipment_id)


@dataclass(frozen=True)
class StockDemand:
    """One demand line: a receipt line or a store-fulfilled online order line."""

    at_ms: int
    store_id: int
    product_id: int
    quantity: int
    local_day: date
    key: str
    source: str = STORE_SALE_SOURCE


@dataclass(frozen=True)
class SupplyBatch:
    """Inputs of one micro-batch.

    ``held_dcs`` is the hook for logistics incidents: a held DC dispatches no
    trucks, so its stores' loads wait (shipments already on the road continue).
    """

    network: SupplyNetwork
    clock_ms: int
    previous_ms: int | None
    demand: tuple[StockDemand, ...] = ()
    held_dcs: frozenset[int] = frozenset()


@dataclass(frozen=True)
class StockMove:
    at_ms: int
    store_id: int
    dc_id: int
    product_id: int
    quantity_delta: int
    reason: str
    source: str
    requested_quantity: int
    balance: int
    key: str


@dataclass(frozen=True)
class StockoutRecord:
    at_ms: int
    store_id: int
    dc_id: int
    product_id: int
    last_known_quantity: int
    key: str


@dataclass(frozen=True)
class ReorderRecord:
    at_ms: int
    store_id: int
    dc_id: int
    product_id: int
    current_quantity: int
    reorder_quantity: int
    reorder_point: int
    priority: str
    key: str


@dataclass(frozen=True)
class TruckArrival:
    at_ms: int
    truck_id: int
    dc_id: int
    store_id: int
    shipment_id: str
    planned_ms: int
    delay_minutes: int
    delay_reason: str | None
    estimated_unload_minutes: int


@dataclass(frozen=True)
class TruckDeparture:
    at_ms: int
    truck_id: int
    dc_id: int
    store_id: int
    shipment_id: str
    unload_minutes: int
    units: int


@dataclass
class SupplyOutput:
    moves: list[StockMove] = field(default_factory=list)
    stockouts: list[StockoutRecord] = field(default_factory=list)
    reorders: list[ReorderRecord] = field(default_factory=list)
    arrivals: list[TruckArrival] = field(default_factory=list)
    departures: list[TruckDeparture] = field(default_factory=list)
    lost: list[str] = field(default_factory=list)


def _utc_day(ms: int) -> str:
    return datetime.fromtimestamp(ms / 1000, timezone.utc).date().isoformat()


def _lines(payload: dict[str, Any]) -> list[tuple[int, int]]:
    return sorted((int(product), int(quantity)) for product, quantity in payload["lines"].items())


def _earliest_dispatch(
    network: SupplyNetwork, store_id: int, pending: dict[str, Any], floor_ms: int
) -> int:
    """The store's first delivery slot at or after its oldest pending reorder."""
    return max(network.next_slot_ms(store_id, int(pending["since_ms"])), floor_ms)


def batch_floor(batch: SupplyBatch) -> int:
    """Earliest instant this batch may act at: just after the previous batch clock."""
    if batch.previous_ms is not None:
        return int(batch.previous_ms) + 1
    return min([demand.at_ms for demand in batch.demand] + [batch.clock_ms])


def required_inventory_keys(logistics: SupplyState, batch: SupplyBatch) -> set[str]:
    """Inventory entries a batch can touch: its demand and every load that may unload by its clock."""
    keys = {inventory_key(d.store_id, d.product_id) for d in batch.demand}
    floor_ms = batch_floor(batch)
    network = batch.network
    for key, payload in logistics.items():
        if key.startswith("shipment:"):
            completes = int(payload["done_ms"])
        elif key.startswith("pending:"):
            store_id = int(key.split(":")[1])
            completes = (
                _earliest_dispatch(network, store_id, payload, floor_ms)
                + (TRUCK_LOAD_MINUTES + network.transit_minutes(store_id)) * _MINUTE_MS
                + TRUCK_NORMAL_DWELL_MINUTES_MIN * _MINUTE_MS
            )
        else:
            continue
        if completes <= batch.clock_ms:
            store_id = int(payload["store_id"]) if "store_id" in payload else int(key.split(":")[1])
            keys.update(inventory_key(store_id, product) for product, _ in _lines(payload))
    return keys


def apply_supply_batch(state: SupplyState, batch: SupplyBatch) -> tuple[SupplyState, SupplyOutput]:
    """Advance store stock and the fleets through one batch window, in time order."""
    st = deepcopy(state)
    out = SupplyOutput()
    network = batch.network
    floor_ms = batch_floor(batch)
    queue: list[tuple[int, int, str, Any]] = []

    def push(at_ms: int, kind: int, tie: str, ref: Any) -> None:
        heapq.heappush(queue, (int(at_ms), kind, tie, ref))

    def stock(store_id: int, product_id: int) -> dict[str, Any]:
        key = inventory_key(store_id, product_id)
        if key not in st:
            raise KeyError(f"inventory entry {key} was not loaded for this batch")
        return st[key]

    for key in sorted(st):
        payload = st[key]
        if key.startswith("shipment:"):
            if payload["arrived"]:
                push(payload["done_ms"], _UNLOAD, key, key)
            else:
                push(payload["arrival_ms"], _ARRIVE, key, key)
        elif key.startswith("pending:"):
            dc_id, store_id = int(payload["dc_id"]), int(key.split(":")[1])
            ready = _earliest_dispatch(network, store_id, payload, floor_ms)
            push(ready, _DISPATCH, f"dc:{dc_id:09d}", dc_id)
        elif key.startswith("truck:") and payload.get("free_ms") is not None:
            dc_id = int(payload["dc_id"])
            push(max(int(payload["free_ms"]), floor_ms), _DISPATCH, f"dc:{dc_id:09d}", dc_id)
    for dc_id in sorted(network.fleets):
        push(floor_ms, _DISPATCH, f"dc:{dc_id:09d}", dc_id)
    for demand in sorted(batch.demand, key=lambda d: (d.at_ms, d.key)):
        push(demand.at_ms, _DEMAND, demand.key, demand)

    def arrive(key: str, at_ms: int) -> None:
        shipment = st[key]
        shipment["arrived"] = True
        out.arrivals.append(
            TruckArrival(
                at_ms,
                int(shipment["truck_id"]),
                int(shipment["dc_id"]),
                int(shipment["store_id"]),
                key.split(":", 1)[1],
                int(shipment["planned_ms"]),
                int(shipment["delay_minutes"]),
                shipment["delay_reason"],
                int(shipment["estimated_unload"]),
            )
        )
        push(shipment["done_ms"], _UNLOAD, key, key)

    def unload(key: str, at_ms: int) -> None:
        shipment = st.pop(key)
        shipment_id = key.split(":", 1)[1]
        store_id, dc_id = int(shipment["store_id"]), int(shipment["dc_id"])
        for product_id, quantity in _lines(shipment):
            entry = stock(store_id, product_id)
            entry["on_hand"] += quantity
            entry["inbound"] = max(0, entry["inbound"] - quantity)
            out.moves.append(
                StockMove(
                    at_ms,
                    store_id,
                    dc_id,
                    product_id,
                    quantity,
                    "INBOUND_SHIPMENT",
                    shipment_id,
                    0,
                    entry["on_hand"],
                    f"{shipment_id}#{product_id}",
                )
            )
        out.departures.append(
            TruckDeparture(
                at_ms,
                int(shipment["truck_id"]),
                dc_id,
                store_id,
                shipment_id,
                int(shipment["unload_minutes"]),
                int(shipment["units"]),
            )
        )
        free_ms = at_ms + network.transit_minutes(store_id) * _MINUTE_MS
        st[f"truck:{int(shipment['truck_id'])}"] = {
            "dc_id": dc_id,
            "free_ms": free_ms,
            "shipment": None,
        }
        push(free_ms, _DISPATCH, f"dc:{dc_id:09d}", dc_id)

    def sell(demand: StockDemand) -> None:
        store_id, product_id = demand.store_id, demand.product_id
        dc_id = network.dc_of(store_id)
        entry = stock(store_id, product_id)
        day = demand.local_day.isoformat()
        if entry["day"] is None:
            entry["day"] = day
        elif entry["day"] != day:
            closed = date.fromisoformat(entry["day"])
            rate = smooth_demand(float(entry["rate"]), int(entry["attempted"]))
            entry["rate"] = decay_demand(rate, (demand.local_day - closed).days - 1)
            entry["attempted"] = 0
            entry["day"] = day
        entry["attempted"] += demand.quantity
        before = int(entry["on_hand"])
        if demand.quantity <= before:
            entry["on_hand"] = before - demand.quantity
            out.moves.append(
                StockMove(
                    demand.at_ms,
                    store_id,
                    dc_id,
                    product_id,
                    -demand.quantity,
                    "SALE",
                    demand.source,
                    demand.quantity,
                    entry["on_hand"],
                    demand.key,
                )
            )
            utc_day = _utc_day(demand.at_ms)
            crossed = before >= STOCKOUT_QUANTITY_THRESHOLD > entry["on_hand"]
            if crossed and entry["stockout_day"] != utc_day:
                entry["stockout_day"] = utc_day
                out.stockouts.append(
                    StockoutRecord(demand.at_ms, store_id, dc_id, product_id, before, demand.key)
                )
        else:
            out.lost.append(demand.key)
            out.moves.append(
                StockMove(
                    demand.at_ms,
                    store_id,
                    dc_id,
                    product_id,
                    0,
                    "LOST_SALE",
                    demand.source,
                    demand.quantity,
                    before,
                    demand.key,
                )
            )
        rate = float(entry["rate"])
        point, target = reorder_levels(rate)
        position = int(entry["on_hand"]) + int(entry["inbound"])
        if position > point:
            return
        quantity = reorder_quantity(target, position)
        entry["inbound"] += quantity
        out.reorders.append(
            ReorderRecord(
                demand.at_ms,
                store_id,
                dc_id,
                product_id,
                int(entry["on_hand"]),
                quantity,
                point,
                reorder_priority(int(entry["on_hand"]), rate),
                demand.key,
            )
        )
        pending_key = f"pending:{store_id}"
        pending = st.get(pending_key)
        if pending is None:
            pending = {"dc_id": dc_id, "since_ms": demand.at_ms, "lines": {}}
            st[pending_key] = pending
            ready = _earliest_dispatch(network, store_id, pending, floor_ms)
            push(ready, _DISPATCH, f"dc:{dc_id:09d}", dc_id)
        lines = pending["lines"]
        lines[str(product_id)] = int(lines.get(str(product_id), 0)) + quantity

    def dispatch(dc_id: int, at_ms: int) -> None:
        if dc_id in batch.held_dcs:
            return
        free = []
        for truck_id in network.fleets.get(dc_id, ()):
            truck = st.get(f"truck:{truck_id}")
            if truck is None:
                free.append((0, truck_id))
            elif truck["shipment"] is None and int(truck["free_ms"]) <= at_ms:
                free.append((int(truck["free_ms"]), truck_id))
        ready = sorted(
            (int(payload["since_ms"]), int(key.split(":")[1]))
            for key, payload in st.items()
            if key.startswith("pending:")
            and int(payload["dc_id"]) == dc_id
            and _earliest_dispatch(network, int(key.split(":")[1]), payload, floor_ms) <= at_ms
        )
        for (_, truck_id), (_, store_id) in zip(sorted(free), ready):
            pending = st.pop(f"pending:{store_id}")
            units = sum(quantity for _, quantity in _lines(pending))
            shipment_id = f"SHP-{network.namespace}-{store_id}-{at_ms}"
            depart_ms = at_ms + TRUCK_LOAD_MINUTES * _MINUTE_MS
            planned_ms = depart_ms + network.transit_minutes(store_id) * _MINUTE_MS
            delay_minutes, delay_reason = 0, None
            arrival_ms = planned_ms + delay_minutes * _MINUTE_MS
            unload = network.unload_minutes(store_id, units, shipment_id)
            st[f"shipment:{shipment_id}"] = {
                "truck_id": truck_id,
                "dc_id": dc_id,
                "store_id": store_id,
                "lines": pending["lines"],
                "units": units,
                "dispatch_ms": at_ms,
                "depart_ms": depart_ms,
                "planned_ms": planned_ms,
                "arrival_ms": arrival_ms,
                "delay_minutes": delay_minutes,
                "delay_reason": delay_reason,
                "estimated_unload": network.estimated_unload_minutes(store_id, units),
                "unload_minutes": unload,
                "done_ms": arrival_ms + unload * _MINUTE_MS,
                "arrived": False,
            }
            st[f"truck:{truck_id}"] = {"dc_id": dc_id, "free_ms": None, "shipment": shipment_id}
            push(arrival_ms, _ARRIVE, f"shipment:{shipment_id}", f"shipment:{shipment_id}")

    while queue and queue[0][0] <= batch.clock_ms:
        at_ms, kind, _tie, ref = heapq.heappop(queue)
        if kind == _ARRIVE:
            if ref in st and not st[ref]["arrived"]:
                arrive(ref, at_ms)
        elif kind == _UNLOAD:
            if ref in st and st[ref]["arrived"]:
                unload(ref, at_ms)
        elif kind == _DEMAND:
            sell(ref)
        else:
            dispatch(int(ref), at_ms)
    return st, out


def lost_lines(output: SupplyOutput) -> dict[str, list[int]]:
    """Lost-sale line numbers by receipt or order ID (demand keys are ``<id>#<line>``)."""
    lost: dict[str, list[int]] = {}
    for key in output.lost:
        owner, line = key.rsplit("#", 1)
        lost.setdefault(owner, []).append(int(line))
    return {owner: sorted(lines) for owner, lines in sorted(lost.items())}


def run_supply_batch(
    store: StateStore,
    batch_id: int,
    batch: SupplyBatch,
    seed_entries: Callable[[set[str]], SupplyState],
) -> tuple[SupplyOutput, dict[str, Changes]]:
    """Read the supply state a batch needs, run it, and return its output and state changes.

    Inventory entries the stream has not touched yet come from
    ``seed_entries(keys)``, which must return an entry for every key it is
    given (the Silver ledger as of the incarnation start, else history's
    opening stock). Reads are as of the previous batch, so a replay of
    ``batch_id`` returns the same output and changes.
    """
    logistics = store.read(LOGISTICS_STATE, batch_id)
    keys = required_inventory_keys(logistics, batch)
    inventory = store.read_keys(INVENTORY_STATE, batch_id, keys)
    missing = keys - inventory.keys()
    if missing:
        seeded = seed_entries(missing)
        absent = missing - seeded.keys()
        if absent:
            raise KeyError(f"no inventory seed for {sorted(absent)[:5]}")
        inventory.update({key: seeded[key] for key in sorted(missing)})
    new_state, output = apply_batch({**inventory, **logistics}, batch, apply_supply_batch)
    new_inventory, new_logistics = split_supply_state(new_state)
    return output, {
        INVENTORY_STATE: state_changes(inventory, new_inventory),
        LOGISTICS_STATE: state_changes(logistics, new_logistics),
    }


"""Live stream events, generated with the historical models.

One micro-batch of ``rate`` rows becomes the events of that minute:

- **Store visits.** A visit row picks a store weighted by its traffic
  multiplier and is kept with the store's local hour/day/month weight (no
  visits while closed). The shopper comes from the store's own customer pool
  (or, as in history, a network-wide pick), weighted by visit tickets. Each
  visit follows an ordered path through the store type's zones (entrance,
  department zones, checkout for buyers, exit) with per-zone dwell times; its
  events are emitted as their time passes, so a visit that spans micro-batches
  lives in the visit state.
  About 22% of visits buy, as in history.
- **Purchases.** Baskets, the basket-level promotion, tender mix, declines and
  marketing journeys reuse the batch helpers. Journey touches carry their
  touch time in ``impression_time`` and arrive with the purchase.
- **Store operations.** Opened/closed events at each store's local operating
  hour boundaries, from the store state.
- **Online orders.** Orders at the configured daily volume, following the
  monthly weights and the online hourly curve. Each order runs a staged
  lifecycle (created with one line event per line, paid, picked, shipped and
  delivered, or ready and collected for BOPIS) whose events are emitted as
  their time passes; open orders live in the order state. Cancellations,
  backorders, promotions, declines and destination tax follow history.
- **Inventory and logistics.** Receipt lines and store-fulfilled online lines
  take stock from a running per-store balance; stockouts, reorders, truck
  shipments and unloads follow from it (``stream_supply``).
- **Marketing.** Background impressions at the historical volume.

``build_notebooks.py`` inlines this module into the stream notebook; the
statistical tests run the same functions offline.
"""

from dataclasses import dataclass, field
from datetime import date, datetime, timezone
from math import floor
from typing import Any

from pyspark.sql import Column, DataFrame, SparkSession, Window
from pyspark.sql import functions as F


ISO_FORMAT = "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"
EVENT_COLUMNS = ["key", "value", "event_type"]
VISIT_SCHEMA = "v long, start_ms long, store_id long, customer_id long, rejected array<int>"
ORDER_SCHEMA = "v long, start_ms long, rejected array<int>"
STORE_OP_SCHEMA = "store_id long, event_ms long, operation_type string"
LOST_SCHEMA = "_owner string, _lost array<int>"
# Days of Silver ledger demand that seed a store-product's smoothed daily rate.
INVENTORY_SEED_LOOKBACK_DAYS = 7
# One row per supply record: inventory moves, stockouts, reorders and trucks.
SUPPLY_SCHEMA = (
    "event_type string, at_ms long, seed string, store_id long, dc_id long, product_id long, "
    "quantity_delta long, reason string, source string, requested_quantity long, "
    "last_known_quantity long, current_quantity long, reorder_quantity long, "
    "reorder_point long, priority string, truck_id long, shipment_id string, planned_ms long, "
    "delay_minutes long, delay_reason string, estimated_unload long, unload_minutes long"
)


@dataclass(frozen=True)
class StreamContext:
    """Static inputs of the live stream, built once per notebook run."""

    seed: int
    namespace: str
    source: str
    profile: StoreTypeProfile
    history_start: date
    lanes: LanePlan
    stores: DataFrame
    store_picker: DataFrame
    customer_pools: DataFrame
    lifecycle: CustomerLifecycle
    ids: dict[str, tuple[DataFrame, int]]
    baskets: BasketCatalog
    store_hours: tuple[StoreHours, ...]
    supply: SupplyNetwork
    customer_tax: DataFrame
    inventory_seed: DataFrame | None = None
    # Year-over-year volume trend, as history (pricing.volume_growth).
    anchor_date: date | None = None
    annual_volume_growth: float = 0.0

    def growth(self, day: Column) -> Column:
        """Volume trend multiplier on a date column (1 at the anchor date)."""
        anchor = self.anchor_date or self.history_start
        return volume_growth_column(self.annual_volume_growth, anchor, day)

    @property
    def draws(self) -> seeded_draws:
        return seeded_draws(self.seed)


@dataclass
class StreamBatch:
    """Events of one micro-batch and the state changes to commit after writing them."""

    events: DataFrame
    clock_ms: int | None
    changes: dict[str, Changes]
    cached: list[DataFrame] = field(default_factory=list)

    def release(self) -> None:
        for frame in self.cached:
            frame.unpersist()


# ---------------------------------------------------------------------------
# Context
# ---------------------------------------------------------------------------


def _single_row_array(spark: SparkSession, name: str, entries: list, element: str) -> DataFrame:
    return spark.createDataFrame([(entries,)], f"{name} array<struct<{element}>>")


def build_stream_context(
    spark: SparkSession,
    *,
    seed: int,
    namespace: str,
    source: str,
    profile: StoreTypeProfile,
    history_start: date,
    dim_stores: DataFrame,
    dim_customers: DataFrame,
    ids: dict[str, tuple[DataFrame, int]],
    baskets: BasketCatalog,
    lanes: LanePlan,
    supply: SupplyNetwork,
    lifecycle: CustomerLifecycle,
    inventory_seed: DataFrame | None = None,
    anchor_date: date | None = None,
    annual_volume_growth: float = 0.0,
) -> StreamContext:
    """Collect the bounded store metadata and build the customer pools.

    Each store's pool holds its home customers (``dim_customers.HomeStoreID``)
    weighted by their visit tickets; ``lifecycle`` (the history's
    :class:`CustomerLifecycle`) decides who is active on a live day.
    ``inventory_seed`` (``store_id, product_id, on_hand, rate``, see
    :func:`inventory_seed_frame`) seeds store stock the stream has not
    touched yet; without it, stock starts from history's opening stock.
    """
    stores = dim_stores.select(
        F.col("ID").cast("long").alias("store_id"),
        F.col("GeographyID").cast("long").alias("store_geo_id"),
        F.col("TimeZone").alias("tz"),
        "operating_hours",
        "store_format",
        F.col("daily_traffic_multiplier").cast("double").alias("multiplier"),
        F.round(F.col("tax_rate") * 10000).cast("long").alias("store_bps"),
        F.col("tax_rate").cast("double").alias("_tax_rate"),
    ).where(F.col("store_id").isNotNull())
    rows = stores.orderBy("store_id").collect()
    if not rows:
        raise ValueError("dim_stores has no stores; run setup first")
    for column, problem in (("tz", "TimeZone"), ("store_bps", "tax_rate")):
        missing = [row["store_id"] for row in rows if row[column] is None]
        if missing:
            raise ValueError(
                f"{len(missing)} dim_stores rows have no {problem}; re-run setup so live "
                "events follow local hours and tax"
            )

    patterns = sorted({row["operating_hours"] for row in rows})
    hours = spark.createDataFrame(
        [
            (
                pattern,
                hour_shares(profile.hourly_weights, pattern),
                *operating_window(pattern),
            )
            for pattern in patterns
        ],
        "operating_hours string, _hour_share array<double>, open_hour int, close_hour int",
    )

    customers = (
        dim_customers.select(
            F.col("ID").cast("long").alias("customer_id"),
            F.col("HomeStoreID").cast("long").alias("pool_store"),
        )
        .where(F.col("customer_id").isNotNull() & F.col("pool_store").isNotNull())
        .withColumn("_tickets", visit_tickets(lifecycle, F.col("customer_id")).cast("double"))
    )
    ordered = (
        Window.partitionBy("pool_store")
        .orderBy("customer_id")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    pools = (
        customers.withColumn(
            "_cdf",
            F.sum("_tickets").over(ordered)
            / F.sum("_tickets").over(Window.partitionBy("pool_store")),
        )
        .groupBy("pool_store")
        .agg(F.sort_array(F.collect_list(F.struct("_cdf", "customer_id"))).alias("_pool"))
        .persist()
    )
    if pools.limit(1).count() == 0:
        raise ValueError("dim_customers has no customers with a home store; re-run setup")

    total_multiplier = sum(row["multiplier"] for row in rows)
    cumulative, store_entries = 0.0, []
    for row in rows:
        cumulative += row["multiplier"] / total_multiplier
        store_entries.append((cumulative, row["store_id"]))

    stores = stores.join(F.broadcast(hours), "operating_hours").persist()
    # Destination tax for DC-shipped online orders, as in history: the mean
    # store rate of the customer's geography, else the network mean.
    mean_bps = int(round(sum(row["_tax_rate"] for row in rows) / len(rows) * 10000))
    geo_rates: dict[int, list[float]] = {}
    for row in rows:
        geo_rates.setdefault(int(row["store_geo_id"]), []).append(float(row["_tax_rate"]))
    geo_bps = spark.createDataFrame(
        [
            (geo, floor(sum(rates) / len(rates) * 10000 + 0.5))
            for geo, rates in sorted(geo_rates.items())
        ],
        "geo_id long, _geo_bps long",
    )
    customer_tax = (
        dim_customers.select(
            F.col("ID").cast("long").alias("customer_id"),
            F.col("GeographyID").cast("long").alias("geo_id"),
        )
        .join(F.broadcast(geo_bps), "geo_id", "left")
        .select("customer_id", F.coalesce("_geo_bps", F.lit(mean_bps)).alias("customer_bps"))
        .persist()
    )
    return StreamContext(
        seed=seed,
        namespace=namespace,
        source=source,
        profile=profile,
        history_start=history_start,
        lanes=lanes,
        stores=stores,
        store_picker=_single_row_array(
            spark, "_stores", store_entries, "_cdf:double,store_id:bigint"
        ),
        customer_pools=pools,
        lifecycle=lifecycle,
        ids=ids,
        baskets=baskets,
        store_hours=tuple(
            StoreHours(int(row["store_id"]), str(row["operating_hours"]), str(row["tz"]))
            for row in rows
        ),
        supply=supply,
        customer_tax=customer_tax,
        inventory_seed=inventory_seed,
        anchor_date=anchor_date,
        annual_volume_growth=annual_volume_growth,
    )


# ---------------------------------------------------------------------------
# Envelope
# ---------------------------------------------------------------------------


def iso(ts: Column) -> Column:
    return F.date_format(ts, ISO_FORMAT)


def _str(value: Column | None) -> Column:
    return value if value is not None else F.lit(None).cast("string")


def envelope(
    ctx: StreamContext,
    event_type: str | Column,
    payload: Column,
    ts: Column,
    pkey: Column,
    trace_seed: Column,
    *,
    session: Column | None = None,
    parent: Column | None = None,
    correlation: Column | None = None,
    arrival: Column | None = None,
) -> Column:
    """``struct(key, value, event_type)`` with the JSON ``EventEnvelope`` as value.

    ``ingest_timestamp`` is the event's arrival time: ``arrival`` when given
    (journey touches arrive with their purchase), else the event time. Both
    derive from the rate-source timestamp, so a replay is identical.
    ``correlation`` carries the attribution journey ID on linked events.
    """
    et = F.lit(event_type) if isinstance(event_type, str) else event_type
    trace = F.abs(F.xxhash64(F.lit(ctx.namespace), trace_seed, et)).cast("string")
    value = F.to_json(
        F.struct(
            et.alias("event_type"),
            payload.alias("payload"),
            F.concat(F.lit("TRC-"), trace).alias("trace_id"),
            iso(ts if arrival is None else arrival).alias("ingest_timestamp"),
            F.lit("1.0").alias("schema_version"),
            F.lit(ctx.source).alias("source"),
            _str(correlation).alias("correlation_id"),
            pkey.alias("partition_key"),
            _str(session).alias("session_id"),
            _str(parent).alias("parent_event_id"),
        )
    )
    return F.struct(pkey.alias("key"), value.alias("value"), et.alias("event_type"))


def slot(ctx: StreamContext, cond: Column, event_type: str | Column, *args, **kwargs) -> Column:
    """An envelope when ``cond`` holds, else NULL."""
    return F.when(cond, envelope(ctx, event_type, *args, **kwargs))


def collect_events(frame: DataFrame, events: Column) -> DataFrame:
    """Explode an array of nullable envelopes into ``key, value, event_type`` rows."""
    return (
        frame.select(F.explode(events).alias("e"))
        .where(F.col("e").isNotNull())
        .select("e.key", "e.value", "e.event_type")
    )


def _ms_ts(ms: Column) -> Column:
    return F.timestamp_millis(ms.cast("long"))


def _prefixed(ctx: StreamContext, prefix: str, column: Column) -> Column:
    return F.concat(F.lit(f"{prefix}-{ctx.namespace}-"), column.cast("string"))


def _store_key(store_id: Column) -> Column:
    return F.concat(F.lit("store_"), store_id.cast("string"))


def _empty_events(spark: SparkSession) -> DataFrame:
    return spark.createDataFrame([], "key string, value string, event_type string")


# ---------------------------------------------------------------------------
# Lanes
# ---------------------------------------------------------------------------


def assign_lanes(ctx: StreamContext, rows: DataFrame) -> DataFrame:
    """Add ``lane`` to rate rows (``ts``, ``v``) by the lane plan's shares."""
    draw = ctx.draws.u([F.col("v")], "lane")
    bound, branches = 0.0, []
    for lane in STREAM_LANES:
        bound += ctx.lanes.shares[lane]
        branches.append((draw < F.lit(bound), F.lit(lane)))
    return rows.withColumn("lane", case_when(branches).otherwise(F.lit("idle")))


def _keep(ctx: StreamContext, lane: str, time_factor: Column | None = None) -> Column:
    """Thinning: keep a lane row with ``min(1, factor * time_factor)``."""
    probability = F.lit(ctx.lanes.factors[lane])
    if time_factor is not None:
        probability = probability * time_factor
    return ctx.draws.u([F.col("v")], f"emit-{lane}") < F.least(F.lit(1.0), probability)


def _sample(ctx: StreamContext, frame: DataFrame, salt: str, name: str, output: str) -> DataFrame:
    lookup, count = ctx.ids[name]
    return attach_sampled_id(frame, F.col("v"), salt, ctx.seed, lookup, count, output)


# ---------------------------------------------------------------------------
# Store visits
# ---------------------------------------------------------------------------


def _local(ts: Column) -> Column:
    return F.from_utc_timestamp(ts, F.col("tz"))


def new_visits(ctx: StreamContext, lanes: DataFrame) -> DataFrame:
    """Visits that start in this batch: ``v, start_ms, store_id, customer_id``."""
    d = ctx.draws
    v = F.col("v")
    visits = (
        lanes.where(F.col("lane") == "visits")
        .crossJoin(F.broadcast(ctx.store_picker))
        .withColumn(
            "store_id", pick_retail_product(F.col("_stores"), d.u([v], "store"))["store_id"]
        )
        .drop("_stores")
        .join(F.broadcast(ctx.stores), "store_id")
    )
    local_day = F.to_date(_local(F.col("ts")))
    time_factor = F.when(closed_date_column(local_day), F.lit(0.0)).otherwise(
        ctx.growth(local_day)
        * daily_factor_column(ctx.profile, local_day)
        * monthly_factor_column(ctx.profile, local_day)
        * F.element_at(F.col("_hour_share"), F.hour(_local(F.col("ts"))) + 1)
    )
    visits = visits.where(_keep(ctx, "visits", time_factor))
    # As in history: most identified shoppers are the store's home customers,
    # the rest another store's; a shopper is kept with their lifecycle
    # activity that local day, else the visit is anonymous.
    home = d.u([v], "customer-home-visit") < F.lit(HOME_STORE_SHARE)
    away = pick_retail_product(F.col("_stores"), d.u([v], "customer-away-store"))["store_id"]
    visits = (
        visits.crossJoin(F.broadcast(ctx.store_picker))
        .withColumn("pool_store", F.when(home, F.col("store_id")).otherwise(away).cast("long"))
        .drop("_stores")
        .join(F.broadcast(ctx.customer_pools), "pool_store", "left")
    )
    visits = visits.withColumn(
        "_picked",
        F.when(
            F.col("_pool").isNotNull(),
            pick_retail_product(F.col("_pool"), d.u([v], "customer"))["customer_id"].cast("long"),
        ),
    )
    picked = F.col("_picked")
    kept = (d.u([v], "customer-identified") < F.lit(ctx.lifecycle.identify_probability)) & (
        d.u([v], "customer-active") < activity_column(ctx.lifecycle, picked, local_day)
    )
    return visits.select(
        v,
        F.unix_millis(F.col("ts")).alias("start_ms"),
        "store_id",
        F.when(kept, picked).cast("long").alias("customer_id"),
    )


def plan_visits(ctx: StreamContext, visits: DataFrame) -> DataFrame:
    """Lay out each visit's events in time.

    Adds the visit's IDs, conversion, BLE identity and detection, the ordered
    zone path with dwell times (``vocabulary.visit_zone_path``, as history),
    zone entry, ping and zone-change offsets, purchase and
    payment offsets (buyers), and ``end_ms``. All draws are keyed on the
    visit's rate row, so the plan is identical in every batch that sees it.
    Visits that would run past closing time are compressed to end before it.
    """
    d = ctx.draws
    v = F.col("v")
    frame = visits.join(F.broadcast(ctx.stores), "store_id")
    start = _ms_ts(F.col("start_ms"))
    local_start = _local(start)
    frame = (
        frame.withColumn("_local_day", F.to_date(local_start))
        .withColumn("_local_hour", F.hour(local_start))
        .withColumn("receipt_id", _prefixed(ctx, "RCP", v))
        .withColumn("session_id", _prefixed(ctx, "SES", v))
        .withColumn(
            "converts",
            d.u([v], "visit-conversion")
            < visit_conversion(d, F.col("store_id"), F.col("_local_day"), F.col("_local_hour")),
        )
        .withColumn(
            "ble_id",
            F.when(
                (d.u([v], "known") < F.lit(BLE_KNOWN_SHARE)) & F.col("customer_id").isNotNull(),
                ble_id_column(F.col("customer_id")),
            ).otherwise(anonymous_ble_id_column(F.col("store_id"), d.h64([v], "anon_id"))),
        )
        .withColumn("ble_detected", d.u([v], "ble-detected") < F.lit(BLE_DETECTED_SHARE))
    )

    # Ordered path shared with history: entrance, 1-3 department zones in
    # random order, checkout (buyers), exit.
    layout = ble_layout(ctx.profile)
    frame = frame.withColumn("zones", visit_zone_path(d, [v], F.col("converts"), layout))
    frame = frame.withColumn("dwell", visit_zone_dwell(d, [v], F.col("zones"), layout))
    frame = frame.withColumn("starts", visit_zone_starts(F.col("dwell"))).withColumn(
        "_duration_s", F.aggregate("dwell", F.lit(0).cast("long"), lambda acc, x: acc + x)
    )
    # Compress a visit that would run past a closing store's closing time.
    close_local = F.make_timestamp(
        F.year("_local_day"),
        F.month("_local_day"),
        F.dayofmonth("_local_day"),
        F.col("close_hour"),
        F.lit(0),
        F.lit(0),
    )
    close_ms = F.when(
        F.col("close_hour") < 24, F.unix_millis(F.to_utc_timestamp(close_local, F.col("tz")))
    )
    duration_ms = F.col("_duration_s") * 1000
    frame = frame.withColumn(
        "_scale",
        F.when(
            close_ms.isNotNull() & (F.col("start_ms") + duration_ms > close_ms),
            F.greatest(F.lit(0).cast("long"), close_ms - 1000 - F.col("start_ms")) / duration_ms,
        ).otherwise(F.lit(1.0)),
    )

    def offset(seconds: Column) -> Column:
        return F.floor(seconds * 1000 * F.col("_scale")).cast("long")

    frame = frame.withColumn("_ping_counts", visit_ping_counts(d, [v], F.col("zones")))
    frame = frame.withColumn(
        "pings",
        F.flatten(
            F.transform(
                "zones",
                lambda zone, i: F.transform(
                    F.sequence(F.lit(1), F.element_at("_ping_counts", i + 1)),
                    lambda k: F.struct(
                        (i + 1).alias("zone_index"),
                        k.alias("seq"),
                        zone.alias("zone"),
                        offset(
                            F.element_at("starts", i + 1)
                            + F.element_at("dwell", i + 1)
                            * k
                            / (F.element_at("_ping_counts", i + 1) + 1)
                        ).alias("offset_ms"),
                    ),
                ),
            )
        ),
    ).withColumn(
        "entries",
        F.transform(
            "zones",
            lambda zone, i: F.struct(
                (i + 1).alias("zone_index"),
                zone.alias("zone"),
                offset(F.element_at("starts", i + 1)).alias("offset_ms"),
                F.element_at("dwell", i + 1).alias("dwell"),
            ),
        ),
    ).withColumn(
        "changes",
        F.transform(
            F.sequence(F.lit(1), F.size("zones") - 1),
            lambda i: F.struct(
                i.alias("seq"),
                F.element_at("zones", i).alias("from_zone"),
                F.element_at("zones", i + 1).alias("to_zone"),
                offset(F.element_at("starts", i + 1)).alias("offset_ms"),
            ),
        ),
    )
    # Buyers pay in the checkout zone, the last zone before any exit.
    last = checkout_index(F.col("zones"), layout)
    frame = (
        frame.withColumn("tender", tender_column(d, [F.col("receipt_id")]))
        .withColumn(
            "processing_ms", processing_ms_column(d, [F.col("receipt_id")], F.col("tender"))
        )
        .withColumn(
            "purchase_ms",
            F.when(
                F.col("converts"),
                F.col("start_ms")
                + offset(F.element_at("starts", last) + F.element_at("dwell", last) / 2),
            ),
        )
        .withColumn("payment_ms", F.col("purchase_ms") + F.col("processing_ms"))
        .withColumn(
            "end_ms",
            F.greatest(
                F.col("start_ms") + offset(F.col("_duration_s")),
                F.coalesce(F.col("payment_ms"), F.col("start_ms")),
            ),
        )
    )
    return frame


def _due(ms: Column, previous_ms: int | None, clock_ms: int) -> Column:
    """Event time falls in this batch: after the last batch (open visits) and by the clock."""
    upto = ms <= F.lit(clock_ms)
    if previous_ms is None:
        return upto
    return upto & (F.col("_is_new") | (ms > F.lit(previous_ms)))


def visit_events(
    ctx: StreamContext, plan: DataFrame, previous_ms: int | None, clock_ms: int
) -> DataFrame:
    """Foot-traffic, BLE ping and zone-change events due in this batch."""
    v = F.col("v")
    store = _store_key(F.col("store_id"))
    session = F.col("session_id")
    # Foot-traffic sensors count every zone the visitor passes, as it enters it.
    def entered(entry: Column) -> Column:
        ms = F.col("start_ms") + entry["offset_ms"]
        zone = entry["zone"]
        return slot(
            ctx,
            _due(ms, previous_ms, clock_ms),
            "customer_entered",
            F.struct(
                F.col("store_id"),
                sensor_id_column(F.col("store_id"), zone).alias("sensor_id"),
                zone.alias("zone"),
                F.lit(1).cast("long").alias("customer_count"),
                entry["dwell"].alias("dwell_time"),
            ),
            _ms_ts(ms),
            store,
            F.concat(v, F.lit("-e"), entry["zone_index"]),
            session=session,
        )

    def ping(entry: Column) -> Column:
        ms = F.col("start_ms") + entry["offset_ms"]
        key = F.concat(v, F.lit("-p"), entry["zone_index"], F.lit("-"), entry["seq"])
        return slot(
            ctx,
            _due(ms, previous_ms, clock_ms),
            "ble_ping_detected",
            F.struct(
                F.col("store_id"),
                beacon_id_column(F.col("store_id"), entry["zone"]).alias("beacon_id"),
                F.col("ble_id").alias("customer_ble_id"),
                (F.lit(RSSI_FLOOR) + F.floor(ctx.draws.u([key], "rssi") * F.lit(RSSI_SPAN)))
                .cast("long")
                .alias("rssi"),
                entry["zone"].alias("zone"),
            ),
            _ms_ts(ms),
            store,
            key,
            session=session,
        )

    def change(entry: Column) -> Column:
        ms = F.col("start_ms") + entry["offset_ms"]
        return slot(
            ctx,
            _due(ms, previous_ms, clock_ms),
            "customer_zone_changed",
            F.struct(
                F.col("store_id"),
                F.col("ble_id").alias("customer_ble_id"),
                entry["from_zone"].alias("from_zone"),
                entry["to_zone"].alias("to_zone"),
                iso(_ms_ts(ms)).alias("timestamp"),
            ),
            _ms_ts(ms),
            store,
            F.concat(v, F.lit("-z"), entry["seq"]),
            session=session,
        )

    events = F.concat(
        F.transform("entries", entered),
        F.transform(F.filter("pings", lambda _entry: F.col("ble_detected")), ping),
        F.transform(F.filter("changes", lambda _entry: F.col("ble_detected")), change),
    )
    return collect_events(plan, events)


# ---------------------------------------------------------------------------
# Purchases
# ---------------------------------------------------------------------------


def _sum_lines(lines: str | Column, field_name: str) -> Column:
    return F.aggregate(lines, F.lit(0).cast("long"), lambda acc, line: acc + line[field_name])


def _cents_to_dollars(cents: Column) -> Column:
    return F.round(cents.cast("double") / F.lit(100.0), 2)


def _journey(ctx: StreamContext, frame: DataFrame, key: str, purchase_ms: str) -> DataFrame:
    """Select ~5% of approved purchases for a 2-touch journey, like history."""
    d = ctx.draws
    planned = with_journey_plan(
        frame.withColumn("match_key", F.col(key)).withColumn(
            "purchase_ts", _ms_ts(F.col(purchase_ms))
        ),
        d,
        ctx.history_start,
    )
    selected = (
        (F.col("status") == "APPROVED")
        & F.col("customer_id").isNotNull()
        & (d.u([F.col(key)], "attr_select") < F.lit(SELECTION_RATE))
    )
    return planned.withColumn("_attributed", selected).withColumn(
        "journey_id", F.when(F.col("_attributed"), F.col("attribution_journey_id"))
    )


def journey_touch_events(ctx: StreamContext, frame: DataFrame, due: Column) -> DataFrame:
    """The two ad-impression touches of each selected journey, arriving with the purchase."""
    touches = []
    for suffix, ts_col in (("older", "touch_ts_older"), ("newer", "touch_ts_newer")):
        impression_id = F.concat(F.lit("IMPJRN"), F.col("_jhash"), F.lit(suffix[0].upper()))
        device, cost = impression_device_cost(
            ctx.draws,
            [impression_id],
            F.col("channel"),
            f"attr_device_{suffix}",
            f"attr_cost_{suffix}",
        )
        touches.append(
            slot(
                ctx,
                due & F.col("_attributed"),
                "ad_impression",
                F.struct(
                    F.col("channel"),
                    F.col("campaign_id"),
                    F.concat(F.col("creative_id"), F.lit(f"-{suffix.upper()}")).alias(
                        "creative_id"
                    ),
                    ad_id_column(F.col("customer_id")).alias("customer_ad_id"),
                    impression_id.alias("impression_id"),
                    F.round(cost, 2).alias("cost"),
                    device.alias("device_type"),
                    F.col("customer_id"),
                    iso(F.col(ts_col)).alias("impression_time"),
                ),
                F.col(ts_col),
                F.concat(F.lit("camp_"), F.col("campaign_id")),
                F.concat(F.col("match_key"), F.lit(f"-attr{suffix}")),
                session=F.col("_journey_session"),
                correlation=F.col("journey_id"),
                arrival=F.col("purchase_ts"),
            )
        )
    return collect_events(frame, F.array(*touches))


def purchase_baskets(
    ctx: StreamContext, plan: DataFrame, previous_ms: int | None, clock_ms: int
) -> DataFrame:
    """Buying visits with a purchase or payment due in this batch, with priced basket lines.

    ``_purchase_due`` marks purchases that happen in this batch (their lines
    are this batch's store demand); ``rejected`` holds the line numbers an
    earlier batch turned into lost sales.
    """
    d = ctx.draws
    purchase_due = _due(F.col("purchase_ms"), previous_ms, clock_ms)
    payment_due = _due(F.col("payment_ms"), previous_ms, clock_ms)
    local_purchase = _local(_ms_ts(F.col("purchase_ms")))
    buyers = plan.where(F.col("converts") & (purchase_due | payment_due)).select(
        "v",
        "store_id",
        "customer_id",
        "store_bps",
        "receipt_id",
        "session_id",
        "tender",
        "processing_ms",
        "purchase_ms",
        "payment_ms",
        "rejected",
        purchase_due.alias("_purchase_due"),
        payment_due.alias("_payment_due"),
        F.month(F.to_date(local_purchase)).alias("_month"),
        F.date_format(local_purchase, "yyyy-MM-dd").alias("_local_day"),
        "tz",
    )
    buyers = attach_baskets(buyers, ctx.baskets, ctx.profile, ctx.seed, COMPLEMENT_RATE)
    # As in history: complementary products join the basket and repeated SKUs
    # merge into one line. Prices and the promotion are set when the purchase
    # is emitted, over the lines the store could fill (purchase_events).
    return (
        buyers.withColumn(
            "priced_lines", with_complements(d, F.col("receipt_id"), F.col("basket_lines"))
        )
        .withColumn("priced_lines", merge_lines(F.col("priced_lines")))
        .withColumn(
            "priced_lines",
            F.transform(
                "priced_lines",
                lambda line: line.withField("gross_cents", line["unit_cents"] * line["quantity"]),
            ),
        )
    )


def _demand_rows(frame: DataFrame, owner: str, at_ms: str, store: str) -> list[Any]:
    return (
        frame.select(
            F.col(owner).alias("owner"),
            F.col(at_ms).alias("at_ms"),
            F.col(store).alias("store_id"),
            "_local_day",
            F.explode("priced_lines").alias("line"),
        )
        .select(
            "owner",
            "at_ms",
            "store_id",
            "_local_day",
            F.col("line.line_num").alias("line_num"),
            F.col("line.product_id").alias("product_id"),
            F.col("line.quantity").alias("quantity"),
        )
        .collect()
    )


def _stock_demand(rows: list[Any], source: str) -> list[StockDemand]:
    return [
        StockDemand(
            int(row["at_ms"]),
            int(row["store_id"]),
            int(row["product_id"]),
            int(row["quantity"]),
            date.fromisoformat(row["_local_day"]),
            f"{row['owner']}#{int(row['line_num'])}",
            source,
        )
        for row in rows
    ]


def purchase_demand(buyers: DataFrame) -> list[StockDemand]:
    """Store demand of this batch's purchases: one line per basket line."""
    due = buyers.where(F.col("_purchase_due"))
    return _stock_demand(
        _demand_rows(due, "receipt_id", "purchase_ms", "store_id"), STORE_SALE_SOURCE
    )


def _kept_lines(lines: str) -> Column:
    """Basket lines that were not lost sales."""
    return F.filter(lines, lambda line: ~F.array_contains(F.col("rejected"), line["line_num"]))


def purchase_events(ctx: StreamContext, buyers: DataFrame) -> DataFrame:
    """Receipt, lines, promotions, payment and journey touches of buying visits.

    Lines in ``rejected`` were lost sales (the store had no stock): they are
    dropped, as history drops them, and a purchase with no line left emits
    nothing.
    """
    d = ctx.draws
    purchase_due = F.col("_purchase_due")
    payment_due = F.col("_payment_due")
    buyers = buyers.withColumn("priced_lines", _kept_lines("priced_lines")).where(
        F.size("priced_lines") > 0
    )
    # One basket-level promotion over the filled lines, then integer tax, as
    # history prices its fulfilled baskets.
    buyers = buyers.withColumn(
        "priced_lines",
        price_basket(
            d,
            F.col("receipt_id"),
            F.col("priced_lines"),
            F.col("_month"),
            F.col("_offer"),
            F.col("store_bps"),
            ctx.profile,
        ),
    )
    gross = _sum_lines("priced_lines", "gross_cents")
    discount = _sum_lines("priced_lines", "discount_cents")
    tax = _sum_lines("priced_lines", "tax_cents")
    buyers = (
        buyers.withColumn("gross_cents", gross)
        .withColumn("discount_cents", discount)
        .withColumn("subtotal_cents", gross - discount)
        .withColumn("tax_cents", tax)
        .withColumn("total_cents", gross - discount + tax)
        .withColumn("status", payment_status_column(d, [F.col("receipt_id")], F.col("tender")))
        .withColumn("_journey_session", F.col("session_id"))
    )
    buyers = _journey(ctx, buyers, "receipt_id", "purchase_ms")
    purchase_ts = _ms_ts(F.col("purchase_ms"))
    store = _store_key(F.col("store_id"))
    session = F.col("session_id")
    journey = F.col("journey_id")
    newer_impression = F.concat(F.lit("IMPJRN"), F.col("_jhash"), F.lit("N"))
    total = _cents_to_dollars(F.col("total_cents"))

    receipt = slot(
        ctx,
        purchase_due,
        "receipt_created",
        F.struct(
            F.col("store_id"),
            F.col("customer_id"),
            F.col("receipt_id"),
            _cents_to_dollars(F.col("subtotal_cents")).alias("subtotal"),
            _cents_to_dollars(F.col("tax_cents")).alias("tax"),
            total.alias("total"),
            F.col("tender").alias("tender_type"),
            F.size("priced_lines").cast("long").alias("item_count"),
            F.when(F.col("_attributed"), F.col("campaign_id")).alias("campaign_id"),
            F.when(F.col("_attributed"), newer_impression).alias("impression_id"),
            F.col("gross_cents").alias("gross_subtotal_cents"),
            F.col("discount_cents").alias("discount_cents"),
            F.col("subtotal_cents").alias("subtotal_cents"),
            F.col("tax_cents").alias("tax_cents"),
            F.col("total_cents").alias("total_cents"),
        ),
        purchase_ts,
        store,
        F.col("v"),
        session=session,
        correlation=journey,
    )
    payment_ts = _ms_ts(F.col("payment_ms"))
    payment = slot(
        ctx,
        payment_due,
        "payment_processed",
        F.struct(
            F.col("receipt_id"),
            F.lit(None).cast("string").alias("order_id"),
            F.col("tender").alias("payment_method"),
            total.alias("amount"),
            F.col("total_cents").alias("amount_cents"),
            _prefixed(ctx, "TXN", F.col("v")).alias("transaction_id"),
            iso(payment_ts).alias("processing_time"),
            F.col("processing_ms").cast("int").alias("processing_time_ms"),
            F.col("status"),
            decline_reason_column(d, [F.col("receipt_id")], F.col("status")).alias(
                "decline_reason"
            ),
            F.col("store_id"),
            F.col("customer_id"),
        ),
        payment_ts,
        store,
        F.col("v"),
        session=session,
        parent=F.col("receipt_id"),
        correlation=journey,
    )

    def line_event(line: Column) -> Column:
        return slot(
            ctx,
            purchase_due,
            "receipt_line_added",
            F.struct(
                F.col("receipt_id"),
                line["line_num"].cast("long").alias("line_number"),
                line["product_id"].alias("product_id"),
                line["quantity"].cast("long").alias("quantity"),
                (line["unit_cents"] / 100.0).alias("unit_price"),
                (line["ext_cents"] / 100.0).alias("extended_price"),
                F.when(line["discount_cents"] > 0, line["promo_code"]).alias("promo_code"),
            ),
            purchase_ts,
            store,
            F.concat(F.col("v"), F.lit("-"), line["line_num"]),
            session=session,
            parent=F.col("receipt_id"),
        )

    discounted = F.filter("priced_lines", lambda line: line["discount_cents"] > 0)

    def promotion_event(code: Column) -> Column:
        lines = F.filter(discounted, lambda line: line["promo_code"] == code)
        products = F.array_sort(
            F.array_distinct(F.transform(lines, lambda line: line["product_id"]))
        )
        cents = _sum_lines(lines, "discount_cents")
        return slot(
            ctx,
            purchase_due,
            "promotion_applied",
            F.struct(
                F.col("receipt_id"),
                code.alias("promo_code"),
                _cents_to_dollars(cents).alias("discount_amount"),
                cents.alias("discount_cents"),
                F.when(code.startswith("BOGO"), F.lit("BOGO"))
                .otherwise(F.lit("PERCENTAGE"))
                .alias("discount_type"),
                F.size(products).cast("long").alias("product_count"),
                products.alias("product_ids"),
                F.col("store_id"),
                F.col("customer_id"),
            ),
            purchase_ts,
            store,
            F.concat(F.col("v"), F.lit("-promo-"), code),
            session=session,
            correlation=journey,
        )

    codes = F.array_sort(F.array_distinct(F.transform(discounted, lambda line: line["promo_code"])))
    events = F.concat(
        F.array(receipt, payment),
        F.transform("priced_lines", line_event),
        F.transform(codes, promotion_event),
    )
    return collect_events(buyers, events).unionByName(
        journey_touch_events(ctx, buyers, purchase_due)
    )


# ---------------------------------------------------------------------------
# Online orders
# ---------------------------------------------------------------------------


def new_orders(ctx: StreamContext, lanes: DataFrame) -> DataFrame:
    """Orders placed in this batch, at the configured daily volume.

    Volume follows the profile's monthly weights and the online hourly curve
    (UTC); ``rejected`` starts empty.
    """
    rows = lanes.where(F.col("lane") == "online")
    day = F.to_date(F.col("ts"))
    factor = (
        ctx.growth(day)
        * monthly_factor_column(ctx.profile, day)
        * (online_hour_factor_column(F.col("ts")))
    )
    return rows.where(_keep(ctx, "online", factor)).select(
        F.col("v"),
        F.unix_millis(F.col("ts")).alias("start_ms"),
        F.array().cast("array<int>").alias("rejected"),
    )


def _seconds(ctx: StreamContext, bounds: tuple[int, int], salt: str) -> Column:
    draw = ctx.draws.u([F.col("order_id")], salt)
    return (F.lit(bounds[0]) + draw * F.lit(float(bounds[1] - bounds[0]))).cast("long")


def _online_line(ctx: StreamContext, line: Column) -> Column:
    """Online promo (history's per-order promo share and codes), then destination tax."""
    d = ctx.draws
    keys = [F.col("order_id"), line["line_num"]]
    has_promo = d.u(keys, "online-promo") < F.col("_promo_share")
    index = (d.h64(keys, "online-promo-code") % len(ONLINE_PROMOS)).cast("int") + 1
    code = F.when(has_promo, F.element_at(F.array(*[F.lit(c) for c, _ in ONLINE_PROMOS]), index))
    percent = F.when(
        has_promo, F.element_at(F.array(*[F.lit(p).cast("long") for _, p in ONLINE_PROMOS]), index)
    ).otherwise(F.lit(0).cast("long"))
    discount = F.floor((line["gross_cents"] * percent + F.lit(50)) / F.lit(100)).cast("long")
    net = line["gross_cents"] - discount
    multiplier = (
        F.when(line["taxability"] == "TAXABLE", F.lit(100))
        .when(line["taxability"] == "REDUCED_RATE", F.lit(50))
        .otherwise(F.lit(0))
        .cast("long")
    )
    tax = F.floor((net * F.col("tax_bps") * multiplier + F.lit(500_000)) / F.lit(1_000_000)).cast(
        "long"
    )
    return (
        line.withField("promo_code", code)
        .withField("discount_cents", discount)
        .withField("net_cents", net)
        .withField("tax_cents", tax)
    )


def _online_customers(ctx: StreamContext, orders: DataFrame) -> DataFrame:
    """Add each order's customer: the first of a few tries active that (UTC) day.

    As in history, a try picks a store by traffic and a ticket in its pool;
    an order whose tries all find inactive customers is never placed.
    """
    d = ctx.draws
    tries = orders.select(
        "v",
        F.to_date(_ms_ts(F.col("start_ms"))).alias("_day"),
        F.explode(F.sequence(F.lit(1), F.lit(ONLINE_CUSTOMER_ATTEMPTS))).alias("_try"),
    )
    k = [F.col("v"), F.col("_try")]
    store = pick_retail_product(F.col("_stores"), d.u(k, "online-customer-store"))["store_id"]
    tries = (
        tries.crossJoin(F.broadcast(ctx.store_picker))
        .withColumn("pool_store", store.cast("long"))
        .drop("_stores")
        .join(F.broadcast(ctx.customer_pools), "pool_store")
        .withColumn(
            "_picked",
            pick_retail_product(F.col("_pool"), d.u(k, "online-customer"))["customer_id"].cast(
                "long"
            ),
        )
    )
    active = d.u(k, "online-customer-active") < activity_column(
        ctx.lifecycle, F.col("_picked"), F.col("_day")
    )
    chosen = (
        tries.where(active)
        .groupBy("v")
        .agg(F.min_by("_picked", "_try").cast("long").alias("customer_id"))
    )
    return orders.join(chosen, "v")


def plan_orders(ctx: StreamContext, orders: DataFrame) -> DataFrame:
    """Lay out each order's lifecycle (``v, start_ms, rejected, _is_new`` rows).

    The customer, store, DC, tender, fulfillment mode, basket, promotions,
    cancellation, backorder and every stage time are drawn from the order's
    rate row, so the plan is identical in every batch that sees the order.
    One fulfillment mode and node serve the whole order, picked by the
    historical mix. Store-fulfilled orders (ship-from-store, BOPIS) are taxed
    at the store's rate, DC-shipped orders at the customer's destination rate.
    """
    d = ctx.draws
    v = F.col("v")
    for salt, name, output in (
        ("store", "stores", "store_id"),
        ("dc", "dcs", "dc_id"),
    ):
        orders = _sample(ctx, orders, salt, name, output)
    orders = _online_customers(ctx, orders)
    key = [F.col("order_id")]
    mode = F.col("fulfillment_mode")
    bopis = mode == "BOPIS"
    orders = (
        orders.join(F.broadcast(ctx.stores.select("store_id", "store_bps", "tz")), "store_id")
        .join(ctx.customer_tax, "customer_id", "left")
        .withColumn("order_id", _prefixed(ctx, "ONL", v))
        .withColumn("receipt_id", F.col("order_id"))
        .withColumn("tender", tender_column(d, key, ONLINE_TENDERS, "online-tender"))
        .withColumn(
            "status",
            payment_status_column(d, key, F.col("tender"), ONLINE_TENDERS, "online-decline"),
        )
        .withColumn(
            "processing_ms",
            processing_ms_column(d, key, F.col("tender"), ONLINE_TENDERS, "online-proc"),
        )
        .withColumn(
            "fulfillment_mode", d.pick_by_weights(key, "online-mode", ONLINE_FULFILLMENT_MIX)
        )
        .withColumn(
            "node_type", F.when(mode == "SHIP_FROM_DC", F.lit("DC")).otherwise(F.lit("STORE"))
        )
        .withColumn(
            "node_id",
            F.when(F.col("node_type") == "DC", F.col("dc_id")).otherwise(F.col("store_id")),
        )
        .withColumn(
            "tax_bps",
            F.when(F.col("node_type") == "STORE", F.col("store_bps")).otherwise(
                F.coalesce(F.col("customer_bps"), F.col("store_bps"))
            ),
        )
        .withColumn("cancelled", d.u(key, "online-cancel") < F.lit(ONLINE_CANCEL_RATE))
        .withColumn(
            "backordered",
            ~bopis
            & ~F.col("cancelled")
            & (d.u(key, "online-backorder") < F.lit(ONLINE_BACKORDER_RATE)),
        )
        .withColumn(
            "_promo_share",
            F.lit(ONLINE_PROMO_LINE_SHARE[0])
            + d.u(key, "online-promo-share")
            * F.lit(ONLINE_PROMO_LINE_SHARE[1] - ONLINE_PROMO_LINE_SHARE[0]),
        )
        .withColumn(
            "_local_day",
            F.date_format(_local(_ms_ts(F.col("start_ms"))), "yyyy-MM-dd"),
        )
    )
    orders = attach_baskets(orders, ctx.baskets, ctx.profile, ctx.seed)
    orders = orders.withColumn(
        "priced_lines", F.transform("basket_lines", lambda line: _online_line(ctx, line))
    )
    live = ~F.col("cancelled")
    pick_s = F.when(bopis, _seconds(ctx, ONLINE_PICK_SECONDS["BOPIS"], "online-pick")).otherwise(
        _seconds(ctx, ONLINE_PICK_SECONDS["SHIP"], "online-pick")
    )
    orders = (
        orders.withColumn("payment_ms", F.when(live, F.col("start_ms") + F.col("processing_ms")))
        .withColumn("picked_ms", F.when(live, F.col("start_ms") + pick_s * 1000))
        .withColumn(
            "shipped_ms",
            F.when(
                live & ~bopis,
                F.col("picked_ms") + _seconds(ctx, ONLINE_SHIP_SECONDS, "online-ship") * 1000,
            ),
        )
        .withColumn(
            "delivered_ms",
            F.when(
                live & bopis,
                F.col("picked_ms") + _seconds(ctx, ONLINE_PICKUP_SECONDS, "online-pickup") * 1000,
            ).when(
                live & ~F.col("backordered"),
                F.col("shipped_ms")
                + _seconds(ctx, ONLINE_DELIVER_SECONDS, "online-deliver") * 1000,
            ),
        )
        .withColumn(
            "end_ms",
            F.greatest(
                F.col("start_ms"),
                F.col("payment_ms"),
                F.col("picked_ms"),
                F.col("shipped_ms"),
                F.col("delivered_ms"),
            ),
        )
    )
    return orders


def order_demand(orders: DataFrame) -> list[StockDemand]:
    """Store demand of orders placed in this batch that a store fulfills."""
    placed = orders.where(F.col("_is_new") & (F.col("node_type") == "STORE") & ~F.col("cancelled"))
    return _stock_demand(
        _demand_rows(placed, "order_id", "start_ms", "node_id"), ONLINE_SALE_SOURCE
    )


def online_events(
    ctx: StreamContext, orders: DataFrame, previous_ms: int | None, clock_ms: int
) -> DataFrame:
    """Lifecycle events of planned orders that fall due in this batch.

    An order is created with one ``online_order_line_added`` per line (lines
    a fulfilling store could not supply were dropped as lost sales), paid
    after the tender's processing time, picked, shipped and delivered; a BOPIS
    order is picked (ready for pickup) and delivered when the customer
    collects it, and is never shipped. Cancelled orders have zero totals, no
    payment and no fulfillment; backordered orders ship but are not delivered.
    """
    v = F.col("v")
    orders = orders.withColumn("priced_lines", _kept_lines("priced_lines")).where(
        F.size("priced_lines") > 0
    )
    live = ~F.col("cancelled")
    subtotal = F.when(live, _sum_lines("priced_lines", "net_cents")).otherwise(F.lit(0))
    discount = F.when(live, _sum_lines("priced_lines", "discount_cents")).otherwise(F.lit(0))
    tax = F.when(live, _sum_lines("priced_lines", "tax_cents")).otherwise(F.lit(0))
    orders = (
        orders.withColumn("subtotal_cents", subtotal.cast("long"))
        .withColumn("discount_cents", discount.cast("long"))
        .withColumn("gross_cents", (subtotal + discount).cast("long"))
        .withColumn("tax_cents", tax.cast("long"))
        .withColumn("total_cents", (subtotal + tax).cast("long"))
        .withColumn("_journey_session", F.col("order_id"))
    )
    # Cancelled orders are never paid, so they are never attributed.
    orders = _journey(
        ctx, orders.withColumn("status", F.when(live, F.col("status"))), "order_id", "start_ms"
    )
    created_ts = _ms_ts(F.col("start_ms"))
    created_due = _due(F.col("start_ms"), previous_ms, clock_ms)
    pkey = F.concat(F.lit("order_"), F.col("order_id"))
    session = F.col("order_id")
    journey = F.col("journey_id")
    total = _cents_to_dollars(F.col("total_cents"))

    def stage(event_type: str, ms_col: str, time_field: str) -> Column:
        ms = F.col(ms_col)
        return slot(
            ctx,
            ms.isNotNull() & _due(ms, previous_ms, clock_ms),
            event_type,
            F.struct(
                F.col("order_id"),
                F.col("node_type"),
                F.col("node_id"),
                F.col("fulfillment_mode"),
                iso(_ms_ts(ms)).alias(time_field),
            ),
            _ms_ts(ms),
            pkey,
            v,
            session=session,
            parent=F.col("order_id"),
            correlation=journey,
        )

    def line_event(line: Column) -> Column:
        return slot(
            ctx,
            created_due,
            "online_order_line_added",
            F.struct(
                F.col("order_id"),
                line["line_num"].cast("long").alias("line_number"),
                line["product_id"].alias("product_id"),
                line["quantity"].cast("long").alias("quantity"),
                (line["unit_cents"] / 100.0).alias("unit_price"),
                (line["net_cents"] / 100.0).alias("extended_price"),
                line["promo_code"].alias("promo_code"),
                F.col("fulfillment_mode"),
                F.col("node_type"),
                F.col("node_id"),
            ),
            created_ts,
            pkey,
            F.concat(v, F.lit("-"), line["line_num"]),
            session=session,
            parent=F.col("order_id"),
        )

    payment_ms = F.col("payment_ms")
    payment_ts = _ms_ts(payment_ms)
    events = F.concat(
        F.array(
            slot(
                ctx,
                created_due,
                "online_order_created",
                F.struct(
                    F.col("order_id"),
                    F.col("customer_id"),
                    F.col("fulfillment_mode"),
                    F.col("node_type"),
                    F.col("node_id"),
                    F.size("priced_lines").cast("long").alias("item_count"),
                    _cents_to_dollars(F.col("subtotal_cents")).alias("subtotal"),
                    _cents_to_dollars(F.col("tax_cents")).alias("tax"),
                    total.alias("total"),
                    F.col("tender").alias("tender_type"),
                    F.when(F.col("_attributed"), F.col("campaign_id")).alias("campaign_id"),
                    F.when(
                        F.col("_attributed"),
                        F.concat(F.lit("IMPJRN"), F.col("_jhash"), F.lit("N")),
                    ).alias("impression_id"),
                    F.col("gross_cents").alias("gross_subtotal_cents"),
                    F.col("discount_cents"),
                    F.col("subtotal_cents"),
                    F.col("tax_cents"),
                    F.col("total_cents"),
                ),
                created_ts,
                pkey,
                v,
                session=session,
                correlation=journey,
            ),
            slot(
                ctx,
                payment_ms.isNotNull() & _due(payment_ms, previous_ms, clock_ms),
                "payment_processed",
                F.struct(
                    F.lit(None).cast("string").alias("receipt_id"),
                    F.col("order_id"),
                    F.col("tender").alias("payment_method"),
                    total.alias("amount"),
                    F.col("total_cents").alias("amount_cents"),
                    _prefixed(ctx, "TXN-ONL", v).alias("transaction_id"),
                    iso(payment_ts).alias("processing_time"),
                    F.col("processing_ms").cast("int").alias("processing_time_ms"),
                    F.col("status"),
                    decline_reason_column(
                        ctx.draws, [F.col("order_id")], F.col("status"), "online-reason"
                    ).alias("decline_reason"),
                    # Online payments carry no store, as in history.
                    F.lit(None).cast("long").alias("store_id"),
                    F.col("customer_id"),
                ),
                payment_ts,
                pkey,
                v,
                session=session,
                parent=F.col("order_id"),
                correlation=journey,
            ),
            stage("online_order_picked", "picked_ms", "picked_time"),
            stage("online_order_shipped", "shipped_ms", "shipped_time"),
            stage("online_order_delivered", "delivered_ms", "delivered_time"),
        ),
        F.transform("priced_lines", line_event),
    )
    orders = orders.withColumn("purchase_ts", created_ts)
    return collect_events(orders, events).unionByName(
        journey_touch_events(ctx, orders, created_due)
    )


# ---------------------------------------------------------------------------
# Marketing
# ---------------------------------------------------------------------------


def marketing_events(ctx: StreamContext, lanes: DataFrame) -> DataFrame:
    """Background ad impressions with the historical campaigns, channels and costs."""
    d = ctx.draws
    v = F.col("v")
    archetype = d.pick_by_weights(
        [v],
        "mk_archetype",
        [
            (str(index + 1), float(base))
            for index, (_, _, base, _) in enumerate(CAMPAIGN_ARCHETYPES)
        ],
    ).cast("int")
    day = F.to_date(F.col("ts"))
    flash = next(i + 1 for i, entry in enumerate(CAMPAIGN_ARCHETYPES) if entry[0] == "flash_sale")
    impressions = (
        lanes.where(F.col("lane") == "marketing")
        .withColumn("archetype", archetype)
        .withColumn("day", day)
        # flash_sale runs about one day in seven (the same per-day gate as history).
        .withColumn(
            "_active",
            F.when(
                F.col("archetype") == flash,
                (d.u([F.col("day")], "mk_flash_gate") < F.lit(1.0 / 7.0)).cast("double"),
            ).otherwise(F.lit(1.0)),
        )
    )
    impressions = impressions.where(_keep(ctx, "marketing", F.col("_active")))
    impressions = _sample(ctx, impressions, "mkcust", "customers", "customer_pick")
    channels = case_when(
        (F.col("archetype") == index + 1, F.array(*[F.lit(c) for c in entry[1]]))
        for index, entry in enumerate(CAMPAIGN_ARCHETYPES)
    )
    duration = case_when(
        (F.col("archetype") == index + 1, F.lit(entry[3]))
        for index, entry in enumerate(CAMPAIGN_ARCHETYPES)
    )
    impressions = impressions.withColumn(
        "impression_id",
        F.concat(
            F.lit("IMP"),
            F.date_format("day", "yyyyMMdd"),
            F.lpad(F.col("archetype").cast("string"), 2, "0"),
            F.lit(f"-{ctx.namespace}-"),
            v.cast("string"),
        ),
    )
    key = [F.col("impression_id")]
    impressions = impressions.withColumn("_channels", channels).withColumn(
        "channel",
        F.element_at(
            F.col("_channels"),
            (F.floor(d.u(key, "mk_channel") * F.size("_channels")) + 1).cast("int"),
        ),
    )
    device, cost = impression_device_cost(d, key, F.col("channel"), "mk_device", "mk_cost")
    campaign = campaign_id_column(
        campaign_window_start(F.col("day"), ctx.history_start, duration), F.col("archetype")
    )
    impressions = impressions.withColumn("campaign_id", campaign)
    payload = F.struct(
        F.col("channel"),
        F.col("campaign_id"),
        F.concat(F.lit("CREAT"), F.substring(F.col("impression_id"), 4, 200)).alias("creative_id"),
        ad_id_column(F.col("customer_pick")).alias("customer_ad_id"),
        F.col("impression_id"),
        F.round(cost, 2).alias("cost"),
        device.alias("device_type"),
        # A CRM match needs a customer who is signed up and not churned, as in history.
        F.when(
            (d.u(key, "mk_crm") < F.lit(CRM_MATCH_RATE))
            & (activity_column(ctx.lifecycle, F.col("customer_pick"), F.col("day")) > 0),
            F.col("customer_pick"),
        )
        .cast("long")
        .alias("customer_id"),
        iso(F.col("ts")).alias("impression_time"),
    )
    event = envelope(
        ctx,
        "ad_impression",
        payload,
        F.col("ts"),
        F.concat(F.lit("camp_"), F.col("campaign_id")),
        v,
    )
    return collect_events(impressions, F.array(event))


# ---------------------------------------------------------------------------
# Inventory and logistics
# ---------------------------------------------------------------------------


def inventory_seed_frame(
    ledger: DataFrame, as_of: datetime, lookback_days: int = INVENTORY_SEED_LOOKBACK_DAYS
) -> DataFrame:
    """Store stock as of ``as_of`` from the Silver ledger (``fact_store_inventory_txn``).

    One row per store and product: the latest balance at or before ``as_of``
    (never below zero) and the smoothed daily demand ``rate``, the mean daily
    requested quantity (sales and lost sales) over the last ``lookback_days``.
    The stream pins ``as_of`` to its incarnation start, so the seed of an
    entry it has not touched yet never changes while the incarnation runs.
    """
    cutoff = F.lit(as_of.astimezone(timezone.utc).replace(tzinfo=None)).cast("timestamp")
    rows = ledger.where(F.col("event_ts") <= cutoff)
    sold = F.when(F.col("txn_type") == "SALE", -F.col("quantity")).otherwise(F.lit(0))
    requested = (
        F.coalesce(F.col("requested_quantity"), sold)
        if "requested_quantity" in ledger.columns
        else sold
    )
    keys = ["store_id", "product_id"]
    latest = (
        rows.where(F.col("balance").isNotNull())
        .groupBy(*keys)
        .agg(F.max_by("balance", F.struct("event_ts", "trace_id")).alias("on_hand"))
    )
    since = cutoff - F.expr(f"INTERVAL {int(lookback_days)} DAYS")
    demand = (
        rows.where((F.col("event_ts") > since) & F.col("txn_type").isin("SALE", "LOST_SALE"))
        .groupBy(*keys)
        .agg((F.sum(requested) / F.lit(float(lookback_days))).alias("rate"))
    )
    return latest.join(demand, keys, "left").select(
        F.col("store_id").cast("long").alias("store_id"),
        F.col("product_id").cast("long").alias("product_id"),
        F.greatest(F.lit(0), F.col("on_hand")).cast("long").alias("on_hand"),
        F.coalesce(F.col("rate"), F.lit(0.0)).cast("double").alias("rate"),
    )


def seed_inventory_entries(ctx: StreamContext, spark: SparkSession, keys: set[str]) -> SupplyState:
    """Inventory entries for ``keys`` the stream has not touched yet.

    From the incarnation's Silver seed when it has the store and product,
    else history's opening stock with the demand prior.
    """
    pairs = sorted(parse_inventory_key(key) for key in keys)
    seeded: SupplyState = {}
    if ctx.inventory_seed is not None and pairs:
        wanted = spark.createDataFrame(pairs, "store_id long, product_id long")
        for row in ctx.inventory_seed.join(
            F.broadcast(wanted), ["store_id", "product_id"]
        ).collect():
            seeded[inventory_key(row["store_id"], row["product_id"])] = inventory_entry(
                row["on_hand"], row["rate"]
            )
    for store_id, product_id in pairs:
        key = inventory_key(store_id, product_id)
        if key not in seeded:
            seeded[key] = default_inventory_entry(
                ctx.seed, store_id, product_id, ctx.supply.daily_prior
            )
    return seeded


def _supply_rows(output: SupplyOutput) -> list[tuple]:
    def row(event_type: str, at_ms: int, seed: str, **values: Any) -> tuple:
        names = SUPPLY_SCHEMA.split(", ")[3:]
        return (event_type, int(at_ms), seed) + tuple(values.get(name.split()[0]) for name in names)

    rows = []
    for move in output.moves:
        rows.append(
            row(
                "inventory_updated",
                move.at_ms,
                f"inv-{move.key}",
                store_id=move.store_id,
                dc_id=move.dc_id,
                product_id=move.product_id,
                quantity_delta=move.quantity_delta,
                reason=move.reason,
                source=move.source,
                requested_quantity=move.requested_quantity,
            )
        )
    for stockout in output.stockouts:
        rows.append(
            row(
                "stockout_detected",
                stockout.at_ms,
                f"so-{stockout.key}",
                store_id=stockout.store_id,
                dc_id=stockout.dc_id,
                product_id=stockout.product_id,
                last_known_quantity=stockout.last_known_quantity,
            )
        )
    for reorder in output.reorders:
        rows.append(
            row(
                "reorder_triggered",
                reorder.at_ms,
                f"ro-{reorder.key}",
                store_id=reorder.store_id,
                dc_id=reorder.dc_id,
                product_id=reorder.product_id,
                current_quantity=reorder.current_quantity,
                reorder_quantity=reorder.reorder_quantity,
                reorder_point=reorder.reorder_point,
                priority=reorder.priority,
            )
        )
    for arrival in output.arrivals:
        rows.append(
            row(
                "truck_arrived",
                arrival.at_ms,
                f"arr-{arrival.shipment_id}",
                store_id=arrival.store_id,
                dc_id=arrival.dc_id,
                truck_id=arrival.truck_id,
                shipment_id=arrival.shipment_id,
                planned_ms=arrival.planned_ms,
                delay_minutes=arrival.delay_minutes,
                delay_reason=arrival.delay_reason,
                estimated_unload=arrival.estimated_unload_minutes,
            )
        )
    for departure in output.departures:
        rows.append(
            row(
                "truck_departed",
                departure.at_ms,
                f"dep-{departure.shipment_id}",
                store_id=departure.store_id,
                dc_id=departure.dc_id,
                truck_id=departure.truck_id,
                shipment_id=departure.shipment_id,
                unload_minutes=departure.unload_minutes,
            )
        )
    return rows


def supply_events(ctx: StreamContext, spark: SparkSession, output: SupplyOutput) -> DataFrame:
    """Inventory, stockout, reorder and truck events of a batch's supply output."""
    rows = _supply_rows(output)
    if not rows:
        return _empty_events(spark)
    frame = spark.createDataFrame(rows, SUPPLY_SCHEMA)
    kind = F.col("event_type")
    ts = _ms_ts(F.col("at_ms"))
    seed = F.col("seed")
    store = _store_key(F.col("store_id"))
    dc = F.concat(F.lit("dc_"), F.col("dc_id").cast("string"))
    truck_id = F.concat(F.lit("TRK"), F.lpad(F.col("truck_id").cast("string"), 4, "0"))
    shipment = F.col("shipment_id")
    events = F.array(
        slot(
            ctx,
            kind == "inventory_updated",
            "inventory_updated",
            F.struct(
                F.col("store_id"),
                F.col("dc_id"),
                F.col("product_id"),
                F.col("quantity_delta"),
                F.col("reason"),
                F.col("source"),
                F.col("requested_quantity"),
            ),
            ts,
            store,
            seed,
        ),
        slot(
            ctx,
            kind == "stockout_detected",
            "stockout_detected",
            F.struct(
                F.col("store_id"),
                F.col("dc_id"),
                F.col("product_id"),
                F.col("last_known_quantity"),
                iso(ts).alias("detection_time"),
            ),
            ts,
            store,
            seed,
        ),
        slot(
            ctx,
            kind == "reorder_triggered",
            "reorder_triggered",
            F.struct(
                F.col("store_id"),
                F.col("dc_id"),
                F.col("product_id"),
                F.col("current_quantity"),
                F.col("reorder_quantity"),
                F.col("reorder_point"),
                F.col("priority"),
            ),
            ts,
            store,
            seed,
        ),
        slot(
            ctx,
            kind == "truck_arrived",
            "truck_arrived",
            F.struct(
                truck_id.alias("truck_id"),
                F.col("dc_id"),
                F.col("store_id"),
                shipment.alias("shipment_id"),
                iso(ts).alias("arrival_time"),
                F.col("estimated_unload").alias("estimated_unload_duration"),
                iso(_ms_ts(F.col("planned_ms"))).alias("planned_arrival_time"),
                F.col("delay_minutes"),
                F.col("delay_reason"),
            ),
            ts,
            dc,
            seed,
            session=shipment,
        ),
        slot(
            ctx,
            kind == "truck_departed",
            "truck_departed",
            F.struct(
                truck_id.alias("truck_id"),
                F.col("dc_id"),
                F.col("store_id"),
                shipment.alias("shipment_id"),
                iso(ts).alias("departure_time"),
                F.col("unload_minutes").alias("actual_unload_duration"),
            ),
            ts,
            dc,
            seed,
            session=shipment,
        ),
    )
    return collect_events(frame, events)


# ---------------------------------------------------------------------------
# Store operations
# ---------------------------------------------------------------------------


def store_operation_events(ctx: StreamContext, spark: SparkSession, operations: list) -> DataFrame:
    """``store_opened`` / ``store_closed`` events at operating-hour boundaries."""
    if not operations:
        return _empty_events(spark)
    frame = spark.createDataFrame(
        [
            (
                op.store_id,
                int(op.event_time.astimezone(timezone.utc).timestamp() * 1000),
                op.operation_type,
            )
            for op in operations
        ],
        STORE_OP_SCHEMA,
    )
    ts = _ms_ts(F.col("event_ms"))
    event = envelope(
        ctx,
        F.concat(F.lit("store_"), F.col("operation_type")),
        F.struct(
            F.col("store_id"),
            iso(ts).alias("operation_time"),
            F.col("operation_type"),
        ),
        ts,
        _store_key(F.col("store_id")),
        F.concat(F.lit("ops-"), F.col("store_id"), F.lit("-"), F.col("event_ms")),
    )
    return collect_events(frame, F.array(event))


# ---------------------------------------------------------------------------
# Micro-batch
# ---------------------------------------------------------------------------


def _lost_frame(spark: SparkSession, lost: dict[str, list[int]]) -> DataFrame:
    return spark.createDataFrame(sorted(lost.items()), LOST_SCHEMA)


def _with_lost(frame: DataFrame, lost: DataFrame, owner: str) -> DataFrame:
    """Merge this batch's lost-sale lines into ``rejected`` (keyed by receipt or order)."""
    return (
        frame.join(F.broadcast(lost.withColumnRenamed("_owner", owner)), owner, "left")
        .withColumn("rejected", F.coalesce(F.col("_lost"), F.col("rejected")))
        .drop("_lost")
    )


def _open_frame(
    spark: SparkSession, state: dict[str, dict[str, Any]], fields: tuple[str, ...], schema: str
) -> DataFrame:
    return spark.createDataFrame(
        [
            tuple(
                payload.get(name, []) if name == "rejected" else payload.get(name)
                for name in fields
            )
            for payload in state.values()
        ],
        schema,
    ).withColumn("_is_new", F.lit(False))


def _open_entities(
    frame: DataFrame,
    fields: tuple[str, ...],
    owner: str,
    lost: dict[str, list[int]],
    drop_when_all_lost: bool = False,
) -> dict[str, tuple[dict[str, Any], int]]:
    """State payloads (with their final lost-sale lines) and end times of planned entities.

    With ``drop_when_all_lost``, an entity whose every basket line was a lost
    sale is not kept: nothing more happens to it (an online order with no line
    left is never placed).
    """
    known = {}
    columns = [*fields, owner, "end_ms"] + (["basket_n"] if drop_when_all_lost else [])
    for row in frame.select(*columns).collect():
        payload = {
            name: (list(row[name] or []) if name == "rejected" else row[name]) for name in fields
        }
        payload["rejected"] = lost.get(row[owner], payload["rejected"])
        if drop_when_all_lost and len(payload["rejected"]) >= int(row["basket_n"]):
            continue
        known[str(row["v"])] = (payload, int(row["end_ms"]))
    return known


VISIT_FIELDS = ("v", "start_ms", "store_id", "customer_id", "rejected")
ORDER_FIELDS = ("v", "start_ms", "rejected")


def generate_stream_batch(
    ctx: StreamContext, rows: DataFrame, batch_id: int, state: StateStore
) -> StreamBatch:
    """Events of one micro-batch of rate rows (``ts``, ``v``) and its state changes.

    Reads every state as of the previous batch, so a replay of ``batch_id``
    regenerates identical events. Purchases and online orders are planned
    first; their store demand then runs through the supply state machine,
    whose lost sales are dropped from the receipts and orders before their
    events are built. Write the events first, then commit the returned
    changes with :func:`commit_stream_batch`.
    """
    spark = rows.sparkSession
    meta = state.read("meta", batch_id).get("clock")
    previous_ms = None if meta is None else int(meta["clock_ms"])
    newest = rows.agg(F.max(F.unix_millis(F.col("ts")))).first()
    latest = None if newest is None else newest[0]
    if latest is None:
        return StreamBatch(_empty_events(spark), previous_ms, {})
    clock_ms = max(int(latest), previous_ms or 0)
    clock = datetime.fromtimestamp(clock_ms / 1000, timezone.utc)
    previous = (
        None if previous_ms is None else datetime.fromtimestamp(previous_ms / 1000, timezone.utc)
    )

    store_state = state.read("stores", batch_id)
    new_store_state, operations = apply_batch(
        store_state, StoreBatch(ctx.store_hours, clock, previous), apply_store_batch
    )

    lanes = assign_lanes(ctx, rows).persist()
    visit_state = state.read("visits", batch_id)
    started = (
        new_visits(ctx, lanes)
        .withColumn("rejected", F.array().cast("array<int>"))
        .withColumn("_is_new", F.lit(True))
    )
    plan = plan_visits(
        ctx, _open_frame(spark, visit_state, VISIT_FIELDS, VISIT_SCHEMA).unionByName(started)
    ).persist()
    buyers = purchase_baskets(ctx, plan, previous_ms, clock_ms).persist()

    order_state = state.read("orders", batch_id)
    orders = plan_orders(
        ctx,
        _open_frame(spark, order_state, ORDER_FIELDS, ORDER_SCHEMA).unionByName(
            new_orders(ctx, lanes).withColumn("_is_new", F.lit(True))
        ),
    ).persist()

    demand = tuple(purchase_demand(buyers) + order_demand(orders))
    supply, supply_changes = run_supply_batch(
        state,
        batch_id,
        SupplyBatch(ctx.supply, clock_ms, previous_ms, demand),
        lambda keys: seed_inventory_entries(ctx, spark, keys),
    )
    lost = lost_lines(supply)
    lost_frame = _lost_frame(spark, lost)
    buyers = _with_lost(buyers, lost_frame, "receipt_id")
    orders = _with_lost(orders, lost_frame, "order_id")

    new_visit_state, _finished = apply_batch(
        visit_state,
        VisitBatch(clock_ms, _open_entities(plan, VISIT_FIELDS, "receipt_id", lost)),
        apply_visit_batch,
    )
    new_order_state, _delivered = apply_batch(
        order_state,
        VisitBatch(clock_ms, _open_entities(orders, ORDER_FIELDS, "order_id", lost, True)),
        apply_visit_batch,
    )

    events = (
        store_operation_events(ctx, spark, operations)
        .unionByName(visit_events(ctx, plan, previous_ms, clock_ms))
        .unionByName(purchase_events(ctx, buyers))
        .unionByName(online_events(ctx, orders, previous_ms, clock_ms))
        .unionByName(marketing_events(ctx, lanes))
        .unionByName(supply_events(ctx, spark, supply))
    )
    changes: dict[str, Changes] = {
        "stores": state_changes(store_state, new_store_state),
        "visits": state_changes(visit_state, new_visit_state),
        "orders": state_changes(order_state, new_order_state),
        INVENTORY_STATE: supply_changes[INVENTORY_STATE],
        LOGISTICS_STATE: supply_changes[LOGISTICS_STATE],
        "meta": {"clock": {"clock_ms": clock_ms}},
    }
    return StreamBatch(
        events.select(*EVENT_COLUMNS), clock_ms, changes, [lanes, plan, buyers, orders]
    )


def commit_stream_batch(state: StateStore, batch_id: int, batch: StreamBatch) -> None:
    """Persist a batch's state changes after its events were written."""
    for name, changes in batch.changes.items():
        state.write(name, batch_id, changes)
        state.compact(name, batch_id)


In [ ]:
# PARAMETERS — rendered by `retail-setup render`; defaults work unrendered.
def _param(value: str, default: str) -> str:
    return default if len(value) > 1 and value[0] == value[1] == "{" else value

LAKEHOUSE_NAME = _param("{{LAKEHOUSE_NAME}}", "retail_lakehouse")
SILVER_DB = _param("{{SILVER_DB}}", "ag")
STORE_TYPE = _param("{{STORE_TYPE}}", "supercenter")
SEED = int(_param("{{SEED}}", "42"))
# The historical volume the live stream continues, and the first history day
# (campaign windows are anchored on it).
TRANSACTIONS_PER_STORE_DAY = int(_param("{{TRANSACTIONS_PER_STORE_DAY}}", "0"))
ONLINE_ORDERS_PER_DAY = int(_param("{{ONLINE_ORDERS_PER_DAY}}", "0"))
HISTORY_START = _param("{{START_DATE}}", "")
HISTORY_END = _param("{{END_DATE}}", "")
# Prices and the volume trend are anchored on the configuration's anchor date.
ANCHOR_DATE = _param("{{ANCHOR_DATE}}", "")
ANNUAL_VOLUME_GROWTH = float(_param(
    "{{ANNUAL_VOLUME_GROWTH}}", str(GenerationConfig.model_fields["annual_volume_growth"].default)))
# Customer lifecycle of the history (who is signed up, churned or anonymous).
ANNUAL_CHURN_RATE = float(_param(
    "{{ANNUAL_CHURN_RATE}}", str(GenerationConfig.model_fields["annual_churn_rate"].default)))
ANONYMOUS_SHARE = float(_param(
    "{{ANONYMOUS_SHARE}}", str(GenerationConfig.model_fields["anonymous_share"].default)))

spark.conf.set("spark.sql.session.timeZone", "UTC")  # ingest_timestamp depends on it

if not delta_landing_table:
    delta_landing_table = f"{LAKEHOUSE_NAME}.cusn_landing.events"

# A logical stream keeps one stable identity across notebook and Spark retries.
# Deleting the checkpoint root also deletes this metadata and starts a fresh ID
# namespace, preventing business-key collisions between intentional resets.
import re
import uuid

from notebookutils import mssparkutils

_STREAM_ID_PATTERN = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]{0,63}$")


def _validate_stream_id(value):
    value = str(value).strip()
    if not _STREAM_ID_PATTERN.fullmatch(value):
        raise ValueError(
            "stream_id must be 1-64 characters using letters, numbers, '.', '_', or '-'"
        )
    return value


def _resolve_stream_id(root, requested):
    metadata_dir = f"{root.rstrip('/')}/_metadata"
    identity_path = f"{metadata_dir}/stream_id"
    requested = str(requested).strip()

    if mssparkutils.fs.exists(identity_path):
        persisted = _validate_stream_id(mssparkutils.fs.head(identity_path, 128).strip())
        if requested and requested != persisted:
            raise ValueError(
                f"stream_id {requested!r} does not match checkpoint identity "
                f"{persisted!r}; use the persisted value or reset {root!r}"
            )
        return persisted

    candidate = _validate_stream_id(requested or uuid.uuid4().hex)
    mssparkutils.fs.mkdirs(metadata_dir)
    try:
        created = mssparkutils.fs.put(identity_path, candidate, False)
    except Exception:
        if not mssparkutils.fs.exists(identity_path):
            raise
        created = False
    if not created:
        persisted = _validate_stream_id(mssparkutils.fs.head(identity_path, 128).strip())
        if requested and persisted != requested:
            raise ValueError(
                f"stream_id {requested!r} lost a checkpoint identity race to "
                f"{persisted!r}; restart with the persisted value"
            )
        return persisted
    return candidate


STREAM_ID = _resolve_stream_id(checkpoint_path, stream_id)
print(f"stream identity: {STREAM_ID}")

# Each sink has its own Spark checkpoint, and bounded (smoke) runs never share
# offsets with the live run. Every checkpoint has an incarnation record beside
# stream_id: its nonce goes into ingest tags and business IDs, so deleting only
# the Spark checkpoint can never reuse tags Kusto already holds (the connector
# would silently skip those writes). The record also pins the catalog date, so
# a replayed batch regenerates the same baskets. Inconsistent folders stop here.
from datetime import datetime as _datetime, timezone as _timezone

CHECKPOINT_NAME = stream_checkpoint_name(sink, int(run_seconds))
CHECKPOINT_LOCATION = f"{checkpoint_path.rstrip('/')}/{CHECKPOINT_NAME}"
INCARNATION_PLAN = plan_incarnation(
    mssparkutils.fs, checkpoint_path, CHECKPOINT_NAME, STREAM_ID,
    _datetime.now(_timezone.utc).date(),
)
INCARNATION = INCARNATION_PLAN.incarnation
CATALOG_DATE = INCARNATION_PLAN.catalog_date
STREAM_NAMESPACE = INCARNATION_PLAN.namespace
print(f"checkpoint {CHECKPOINT_LOCATION}: incarnation {INCARNATION} "
      f"({'resumed' if INCARNATION_PLAN.resumed else 'new'}), catalog date {CATALOG_DATE}"
      + (f", replaying batch {INCARNATION_PLAN.pending_batch}"
         if INCARNATION_PLAN.pending_batch is not None else ""))

In [ ]:
# Dimension IDs — read from the Silver dims that setup-02 wrote, so events carry
# valid foreign keys. Each lookup numbers the IDs that actually exist (gaps from
# a custom layout or a filtered publish are fine), and inventory events only use
# products launched by the catalog date. Missing/empty dimensions stop here.
from pyspark.sql import functions as _F


def _ids(table: str, where=None, what: str = "rows"):
    frame = spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table}")
    if where is not None:
        frame = frame.where(where)
    lookup, count = dense_id_lookup(frame)
    if count < 1:
        raise ValueError(f"Required streaming dimension {table} has no {what}; run setup first")
    return lookup, count


STORE_IDS, STORE_COUNT = _ids("dim_stores")
CUSTOMER_IDS, CUSTOMER_COUNT = _ids("dim_customers")
LAUNCHED_PRODUCT_IDS, PRODUCT_COUNT = _ids(
    "dim_products", _F.to_date("LaunchDate") <= _F.lit(CATALOG_DATE),
    f"products launched by {CATALOG_DATE}",
)
DC_IDS, DC_COUNT = _ids("dim_distribution_centers")
TRUCK_IDS, TRUCK_COUNT = _ids("dim_trucks")
print(f"ids: stores={STORE_COUNT} customers={CUSTOMER_COUNT} launched products={PRODUCT_COUNT} "
      f"dcs={DC_COUNT} trucks={TRUCK_COUNT}")

# Store, customer and product attributes for the calibrated live models: local
# time zones and hours, traffic multipliers, customer pools by geography, and
# real product prices and store tax rates.
import json
from datetime import date as _date

STREAM_PROFILE = StoreTypeProfile.model_validate(json.loads(
    mssparkutils.fs.head(f"Files/setup/dictionaries/{STORE_TYPE}/profile.json", 65536)
))
STREAM_PRODUCTS = basket_products(spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.dim_products"), SEED)
DIM_STORES = spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.dim_stores")
DIM_CUSTOMERS = spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.dim_customers")

# History-to-stream handoff: history should end within a day before now. A new
# stream refuses a gap or overlap unless allow_history_gap is set; a resumed
# checkpoint only warns (its own downtime is an expected gap).
_history = (spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_receipts")
            .agg(_F.max("event_ts").cast("long").alias("end"),
                 _F.min("event_date").alias("start"),
                 _F.max("event_date").alias("last")).first())
_history_end = _history["end"]
enforce_history_handoff(
    check_history_handoff(
        None if _history_end is None
        else _datetime.fromtimestamp(_history_end, _timezone.utc),
        _datetime.now(_timezone.utc),
    ),
    resumed=INCARNATION_PLAN.resumed,
    allow_gap=bool(allow_history_gap),
)
if not HISTORY_START:
    HISTORY_START = (_history["start"] or CATALOG_DATE).isoformat()
HISTORY_START_DATE = _date.fromisoformat(HISTORY_START)
if not HISTORY_END:
    HISTORY_END = (_history["last"] or CATALOG_DATE).isoformat()
ANCHOR = _date.fromisoformat(ANCHOR_DATE) if ANCHOR_DATE else HISTORY_START_DATE
STREAM_LIFECYCLE = CustomerLifecycle(
    seed=SEED,
    history_start=HISTORY_START_DATE,
    history_end=_date.fromisoformat(HISTORY_END),
    annual_churn_rate=ANNUAL_CHURN_RATE,
    anonymous_share=ANONYMOUS_SHARE,
    visits_per_year=STREAM_PROFILE.resolved_customer_visits_per_year(),
)
# Unrendered runs take the store type's volumes, as configuration derives them.
if TRANSACTIONS_PER_STORE_DAY <= 0:
    TRANSACTIONS_PER_STORE_DAY = STREAM_PROFILE.resolved_transactions_per_store_day()
if ONLINE_ORDERS_PER_DAY <= 0:
    ONLINE_ORDERS_PER_DAY = default_online_orders_per_day(
        STREAM_PROFILE.online_order_share, STORE_COUNT, TRANSACTIONS_PER_STORE_DAY
    )

_store_rows = DIM_STORES.select("daily_traffic_multiplier", "operating_hours").collect()
# The rate source takes whole rows per second; 0 derives the rate from history.
_requested_rate = float(source_rows_per_second)
LANE_PLAN = plan_lanes(
    STREAM_PROFILE,
    store_count=STORE_COUNT,
    multiplier_total=sum(float(r["daily_traffic_multiplier"]) for r in _store_rows),
    operating_hours=[r["operating_hours"] for r in _store_rows],
    transactions_per_store_day=TRANSACTIONS_PER_STORE_DAY,
    online_orders_per_day=ONLINE_ORDERS_PER_DAY,
    marketing_impressions_per_day=sum(entry[2] for entry in CAMPAIGN_ARCHETYPES),
    volume_scale=float(volume_scale),
    rows_per_second=max(1, round(_requested_rate)) if _requested_rate > 0 else None,
    growth=volume_growth(ANNUAL_VOLUME_GROWTH, ANCHOR, CATALOG_DATE),
)
RATE_ROWS_PER_SECOND = int(LANE_PLAN.rows_per_second)
if LANE_PLAN.clipped:
    print(f"WARNING source_rows_per_second={source_rows_per_second} is below the peak "
          f"{sum(LANE_PLAN.peaks.values()):.2f} rows/s; peak hours will be under-produced")

# Stateful features keep their state in Delta tables, one incarnation each.
STREAM_STATE = DeltaStateStore(spark, f"{LAKEHOUSE_NAME}.{SILVER_DB}", STREAM_NAMESPACE)

# Store inventory is seeded from the Silver ledger as of the incarnation start,
# which is pinned in the meta state so restarts and replays seed the same way.
_seed_meta = STREAM_STATE.read("meta", 2**62).get("seed")
if _seed_meta is not None:
    INVENTORY_SEED_AS_OF = _datetime.fromtimestamp(_seed_meta["as_of_ms"] / 1000, _timezone.utc)
else:
    INVENTORY_SEED_AS_OF = _datetime.now(_timezone.utc)
    if STREAM_STATE.has_rows("meta"):
        print("WARNING this incarnation predates the inventory seed; seeding stock as of now")
    else:
        STREAM_STATE.write("meta", SEED_BATCH_ID, {
            "seed": {"as_of_ms": int(INVENTORY_SEED_AS_OF.timestamp() * 1000)}})
try:
    _ledger = spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_store_inventory_txn")
except Exception:  # noqa: BLE001 - no ledger yet; stock starts from opening stock
    _ledger = None
INVENTORY_SEED = (None if _ledger is None
                  else inventory_seed_frame(_ledger, INVENTORY_SEED_AS_OF).persist())

# Stores route to their DC, ship at their route's daily delivery slot (store
# local time) and trucks serve their DC's fleet, exactly as in the historical
# inventory chain; the demand prior is history's.
_DIM_DCS = spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.dim_distribution_centers")
_routes = store_routes(
    SEED, DIM_STORES, spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.dim_geographies"), _DIM_DCS,
)
_store_dc = {store: route.dc_id for store, route in _routes.items()}
_fleets = dc_fleets(
    ((r["ID"], r["DCID"]) for r in spark.table(
        f"{LAKEHOUSE_NAME}.{SILVER_DB}.dim_trucks").select("ID", "DCID").collect()),
    [int(r["ID"]) for r in _DIM_DCS.select("ID").collect()],
)
STREAM_SUPPLY = SupplyNetwork(
    seed=SEED,
    namespace=STREAM_NAMESPACE,
    store_dc=_store_dc,
    fleets=_fleets,
    daily_prior=TRANSACTIONS_PER_STORE_DAY * 10.0
    / max(1, spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.dim_products").count()),
    slots={store: route.slot_minute for store, route in _routes.items()},
    zones={int(r["ID"]): r["TimeZone"] for r in DIM_STORES.select("ID", "TimeZone").collect()},
)
print(f"supply: {len(_store_dc)} stores on {len(_fleets)} DC fleets "
      f"({sum(len(f) for f in _fleets.values())} trucks); stock seeded as of "
      f"{INVENTORY_SEED_AS_OF.isoformat()}")

STREAM_CONTEXT = build_stream_context(
    spark,
    seed=SEED,
    namespace=STREAM_NAMESPACE,
    source=event_source,
    profile=STREAM_PROFILE,
    history_start=HISTORY_START_DATE,
    dim_stores=DIM_STORES,
    dim_customers=DIM_CUSTOMERS,
    ids={
        "stores": (STORE_IDS, STORE_COUNT),
        "customers": (CUSTOMER_IDS, CUSTOMER_COUNT),
        "products": (LAUNCHED_PRODUCT_IDS, PRODUCT_COUNT),
        "dcs": (DC_IDS, DC_COUNT),
        "trucks": (TRUCK_IDS, TRUCK_COUNT),
    },
    baskets=prepare_basket_catalog(
        spark, STREAM_PRODUCTS, STREAM_PROFILE, SEED, CATALOG_DATE, ANCHOR
    ),
    lanes=LANE_PLAN,
    supply=STREAM_SUPPLY,
    lifecycle=STREAM_LIFECYCLE,
    inventory_seed=INVENTORY_SEED,
    anchor_date=ANCHOR,
    annual_volume_growth=ANNUAL_VOLUME_GROWTH,
)
print(f"live volume: {RATE_ROWS_PER_SECOND} source rows/s; lane shares "
      + ", ".join(f"{lane}={share:.3f}" for lane, share in LANE_PLAN.shares.items()))

# A new incarnation seeds store open/closed from the last historical operation.


def _last_store_operations():
    """{store_id: (UTC time, operation)} of each store's last Silver operation."""
    try:
        ops = spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_store_ops")
    except Exception:  # noqa: BLE001 - no history yet; seed from operating hours
        return {}
    rows = ops.groupBy("store_id").agg(
        _F.max(_F.unix_timestamp("event_ts")).alias("at"),
        _F.max_by("operation_type", "event_ts").alias("operation"),
    ).collect()
    return {
        int(r["store_id"]): (_datetime.fromtimestamp(r["at"], _timezone.utc), r["operation"])
        for r in rows if r["at"] is not None
    }


if not STREAM_STATE.has_rows("stores"):
    STREAM_STATE.write("stores", SEED_BATCH_ID, seed_store_state(
        STREAM_CONTEXT.store_hours, _last_store_operations(), _datetime.now(_timezone.utc)))
    print(f"seeded live store state for incarnation {STREAM_NAMESPACE}")

In [ ]:
# ruff: noqa: F821, E402  (Fabric-injected globals; imports live in notebook cells)
# Every micro-batch is generated by `generate_stream_batch` (stream_events in the
# behavior cell above): calibrated lanes of the rate source, store visits with
# their state, purchases, store operations, staged online orders and marketing;
# store inventory, reorders and trucks follow from the purchases (stream_supply).
from pyspark.sql import functions as F

In [ ]:
# Eventhouse (Kusto) routing. Each micro-batch is split by `event_type` and each
# subset is written to its own KQL table with the Fabric Spark connector. The
# per-table column mapping mirrors the KQL `EventMapping` ingestion mappings
# (`fabric/kql_database/02-create-ingestion-mappings.kql`) exactly: the envelope
# fields are shared, and each event type contributes its `$.payload.*` fields. The
# one rename is `inventory_updated.payload_source` <- `$.payload.source`.
import json  # noqa: E402
from pyspark.sql.types import (  # noqa: E402
    ArrayType, IntegerType, LongType, DoubleType, StringType, StructField, StructType)

_ISO_FMT = "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"
_SPARK_TYPE = {
    "long": LongType(), "int": IntegerType(), "real": DoubleType(),
    "string": StringType(),
    "datetime": StringType(),     # parsed from the ISO string, then cast below
    "dynamic": ArrayType(LongType()),  # only product_ids (array<long>)
}

# Envelope fields ($.<field>) — identical for every event table.
ENVELOPE = [
    ("event_type", "string"), ("trace_id", "string"),
    ("ingest_timestamp", "datetime"), ("schema_version", "string"),
    ("source", "string"), ("correlation_id", "string"),
    ("partition_key", "string"), ("session_id", "string"),
    ("parent_event_id", "string"),
]

# Per event type: (kusto_column, json_payload_field, datatype). Generated from the
# KQL ingestion mappings; keep in sync if those change.
EVENT_PAYLOADS = {
    "receipt_created": [("store_id", "store_id", "long"), ("customer_id", "customer_id", "long"), ("receipt_id", "receipt_id", "string"), ("subtotal", "subtotal", "real"), ("tax", "tax", "real"), ("total", "total", "real"), ("tender_type", "tender_type", "string"), ("item_count", "item_count", "long"), ("campaign_id", "campaign_id", "string"), ("impression_id", "impression_id", "string"), ("gross_subtotal_cents", "gross_subtotal_cents", "long"), ("discount_cents", "discount_cents", "long"), ("subtotal_cents", "subtotal_cents", "long"), ("tax_cents", "tax_cents", "long"), ("total_cents", "total_cents", "long")],
    "receipt_line_added": [("receipt_id", "receipt_id", "string"), ("line_number", "line_number", "long"), ("product_id", "product_id", "long"), ("quantity", "quantity", "long"), ("unit_price", "unit_price", "real"), ("extended_price", "extended_price", "real"), ("promo_code", "promo_code", "string")],
    "payment_processed": [("receipt_id", "receipt_id", "string"), ("order_id", "order_id", "string"), ("payment_method", "payment_method", "string"), ("amount", "amount", "real"), ("amount_cents", "amount_cents", "long"), ("transaction_id", "transaction_id", "string"), ("processing_time", "processing_time", "datetime"), ("processing_time_ms", "processing_time_ms", "int"), ("status", "status", "string"), ("decline_reason", "decline_reason", "string"), ("store_id", "store_id", "long"), ("customer_id", "customer_id", "long")],
    "inventory_updated": [("store_id", "store_id", "long"), ("dc_id", "dc_id", "long"), ("product_id", "product_id", "long"), ("quantity_delta", "quantity_delta", "long"), ("reason", "reason", "string"), ("payload_source", "source", "string"), ("requested_quantity", "requested_quantity", "long")],
    "stockout_detected": [("store_id", "store_id", "long"), ("dc_id", "dc_id", "long"), ("product_id", "product_id", "long"), ("last_known_quantity", "last_known_quantity", "long"), ("detection_time", "detection_time", "datetime")],
    "reorder_triggered": [("store_id", "store_id", "long"), ("dc_id", "dc_id", "long"), ("product_id", "product_id", "long"), ("current_quantity", "current_quantity", "long"), ("reorder_quantity", "reorder_quantity", "long"), ("reorder_point", "reorder_point", "long"), ("priority", "priority", "string")],
    "customer_entered": [("store_id", "store_id", "long"), ("sensor_id", "sensor_id", "string"), ("zone", "zone", "string"), ("customer_count", "customer_count", "long"), ("dwell_time", "dwell_time", "long")],
    "customer_zone_changed": [("store_id", "store_id", "long"), ("customer_ble_id", "customer_ble_id", "string"), ("from_zone", "from_zone", "string"), ("to_zone", "to_zone", "string"), ("timestamp", "timestamp", "datetime")],
    "ble_ping_detected": [("store_id", "store_id", "long"), ("beacon_id", "beacon_id", "string"), ("customer_ble_id", "customer_ble_id", "string"), ("rssi", "rssi", "long"), ("zone", "zone", "string")],
    "truck_arrived": [("truck_id", "truck_id", "string"), ("dc_id", "dc_id", "long"), ("store_id", "store_id", "long"), ("shipment_id", "shipment_id", "string"), ("arrival_time", "arrival_time", "datetime"), ("estimated_unload_duration", "estimated_unload_duration", "long"), ("planned_arrival_time", "planned_arrival_time", "datetime"), ("delay_minutes", "delay_minutes", "long"), ("delay_reason", "delay_reason", "string")],
    "truck_departed": [("truck_id", "truck_id", "string"), ("dc_id", "dc_id", "long"), ("store_id", "store_id", "long"), ("shipment_id", "shipment_id", "string"), ("departure_time", "departure_time", "datetime"), ("actual_unload_duration", "actual_unload_duration", "long")],
    "store_opened": [("store_id", "store_id", "long"), ("operation_time", "operation_time", "datetime"), ("operation_type", "operation_type", "string")],
    "store_closed": [("store_id", "store_id", "long"), ("operation_time", "operation_time", "datetime"), ("operation_type", "operation_type", "string")],
    "ad_impression": [("channel", "channel", "string"), ("campaign_id", "campaign_id", "string"), ("creative_id", "creative_id", "string"), ("customer_ad_id", "customer_ad_id", "string"), ("impression_id", "impression_id", "string"), ("cost", "cost", "real"), ("device_type", "device_type", "string"), ("customer_id", "customer_id", "long"), ("impression_time", "impression_time", "datetime")],
    "promotion_applied": [("receipt_id", "receipt_id", "string"), ("promo_code", "promo_code", "string"), ("discount_amount", "discount_amount", "real"), ("discount_cents", "discount_cents", "long"), ("discount_type", "discount_type", "string"), ("product_count", "product_count", "long"), ("product_ids", "product_ids", "dynamic"), ("store_id", "store_id", "long"), ("customer_id", "customer_id", "long")],
    "online_order_created": [("order_id", "order_id", "string"), ("customer_id", "customer_id", "long"), ("fulfillment_mode", "fulfillment_mode", "string"), ("node_type", "node_type", "string"), ("node_id", "node_id", "long"), ("item_count", "item_count", "long"), ("subtotal", "subtotal", "real"), ("tax", "tax", "real"), ("total", "total", "real"), ("tender_type", "tender_type", "string"), ("campaign_id", "campaign_id", "string"), ("impression_id", "impression_id", "string"), ("gross_subtotal_cents", "gross_subtotal_cents", "long"), ("discount_cents", "discount_cents", "long"), ("subtotal_cents", "subtotal_cents", "long"), ("tax_cents", "tax_cents", "long"), ("total_cents", "total_cents", "long")],
    "online_order_picked": [("order_id", "order_id", "string"), ("node_type", "node_type", "string"), ("node_id", "node_id", "long"), ("fulfillment_mode", "fulfillment_mode", "string"), ("picked_time", "picked_time", "datetime")],
    "online_order_shipped": [("order_id", "order_id", "string"), ("node_type", "node_type", "string"), ("node_id", "node_id", "long"), ("fulfillment_mode", "fulfillment_mode", "string"), ("shipped_time", "shipped_time", "datetime")],
    # Declared for the logistics incident model; the stream does not emit
    # these yet.
    "logistics_incident_started": [("incident_id", "incident_id", "string"), ("incident_type", "incident_type", "string"), ("dc_id", "dc_id", "long"), ("store_id", "store_id", "long"), ("started_at", "started_at", "datetime"), ("severity", "severity", "string"), ("affected_store_count", "affected_store_count", "long"), ("description", "description", "string")],
    "logistics_incident_resolved": [("incident_id", "incident_id", "string"), ("incident_type", "incident_type", "string"), ("dc_id", "dc_id", "long"), ("store_id", "store_id", "long"), ("resolved_at", "resolved_at", "datetime")],
    "online_order_line_added": [("order_id", "order_id", "string"), ("line_number", "line_number", "long"), ("product_id", "product_id", "long"), ("quantity", "quantity", "long"), ("unit_price", "unit_price", "real"), ("extended_price", "extended_price", "real"), ("promo_code", "promo_code", "string"), ("fulfillment_mode", "fulfillment_mode", "string"), ("node_type", "node_type", "string"), ("node_id", "node_id", "long")],
    "online_order_delivered": [("order_id", "order_id", "string"), ("node_type", "node_type", "string"), ("node_id", "node_id", "long"), ("fulfillment_mode", "fulfillment_mode", "string"), ("delivered_time", "delivered_time", "datetime")],
}

# Columns present in the KQL table but NOT produced by the stream. They sit
# between the payload block and the envelope block in the table schema (see
# 01-create-tables.kql), so they must be written as explicit nulls in that
# position to keep the positional ingestion aligned.
TABLE_EXTRA_NULLS = {}

KUSTO_FORMAT = "com.microsoft.kusto.spark.synapse.datasource"
# flushImmediately tells the Kusto data-management service to ingest each write
# right away instead of waiting for queued-ingestion batching, so events are
# queryable seconds after their micro-batch. To keep extents and ingest-by tags
# few, every table is written once per batch from a single coalesced partition,
# and live runs trigger once a minute (S-17). The event tables carry an
# extent_tags_retention policy (01-create-tables.kql) that drops the stream's
# ingest-by tags after seven days.

# The per-event_type writes each target a different table, so run them concurrently
# to overlap the blob-stage + ingest round-trips; sequential writes serialize ~18
# blocking calls per micro-batch and fall behind the trigger.
_WRITE_PARALLELISM = 8


def _from_json_schema(event_type):
    """from_json schema for the full envelope: typed top-level fields + payload struct."""
    payload = StructType([
        StructField(jf, _SPARK_TYPE[dt], True) for _col, jf, dt in EVENT_PAYLOADS[event_type]
    ])
    fields = [StructField(name, _SPARK_TYPE[dt], True) for name, dt in ENVELOPE]
    fields.append(StructField("payload", payload, True))
    return StructType(fields)


def _kusto_columns(event_type):
    """Project a parsed-envelope frame to the target KQL table's exact column
    ORDER.

    The Spark Kusto connector ingests a DataFrame into an existing table by
    column POSITION, not by name. The KQL tables (fabric/kql_database/
    01-create-tables.kql) are laid out as: payload columns, then any
    table-only columns, then the shared envelope columns. The DataFrame we
    write here MUST mirror that exact order or every value lands one block
    over (e.g. store_id ends up null and partition_key lands in tender_type)."""
    cols = []
    for col, jf, dt in EVENT_PAYLOADS[event_type]:
        c = F.col("payload").getField(jf)
        if dt == "datetime":
            c = F.to_timestamp(c, _ISO_FMT)
        cols.append(c.alias(col))
    for name, dt in TABLE_EXTRA_NULLS.get(event_type, []):
        cols.append(F.lit(None).cast(_SPARK_TYPE[dt]).alias(name))
    for name, dt in ENVELOPE:
        c = F.col(name)
        if dt == "datetime":
            c = F.to_timestamp(c, _ISO_FMT)
        cols.append(c.alias(name))
    return cols


def _kusto_write_metadata(event_type, batch_id):
    """Return deterministic connector metadata for an idempotent table batch."""
    identity = ingest_tag(STREAM_ID, INCARNATION, event_type, batch_id)
    ingestion_properties = json.dumps({
        "flushImmediately": True,
        "ingestByTags": [identity],
        "ingestIfNotExists": [identity],
    })
    request_id = str(uuid.uuid5(uuid.NAMESPACE_URL, identity))
    return ingestion_properties, request_id


def _write_event_table(batch_df, event_type, batch_id, token):
    """Map one event_type subset to its KQL columns and append it to its table."""
    ingestion_properties, request_id = _kusto_write_metadata(event_type, batch_id)
    mapped = (batch_df.where(F.col("event_type") == event_type)
              .select(F.from_json("value", _from_json_schema(event_type)).alias("e"))
              .select("e.*")
              .select(*_kusto_columns(event_type))
              # One partition -> one blob, one extent and one tag per table per batch.
              .coalesce(1))
    (mapped.write.format(KUSTO_FORMAT)
        .option("kustoCluster", kusto_uri)
        .option("kustoDatabase", kql_database)
        .option("kustoTable", event_type)
        .option("accessToken", token)
        .option("tableCreateOptions", "FailIfNotExist")
        .option("writeMode", "Transactional")
        .option("requestId", request_id)
        .option("ensureNoDuplicatedBlobs", "true")
        .option("sparkIngestionPropertiesJson", ingestion_properties)
        .mode("Append").save())


def write_to_eventhouse(batch_df, batch_id):
    """foreachBatch sink: split by event_type and write each to its KQL table.

    Each event_type targets a different table, so the writes run concurrently in a
    bounded thread pool — this overlaps the blob-stage + ingest round-trips so a
    micro-batch finishes inside the trigger window instead of serializing ~18
    blocking calls (the old behaviour fell behind and looked "exceptionally slow").
    Each table write retries transient errors (throttling, 5xx, timeouts, an
    expired token) with exponential backoff and a fresh token. Per-table errors
    are collected so every worker can finish, then raised to prevent Spark from
    committing the checkpoint. Deterministic Kusto ingestion tags make
    already-completed sibling writes idempotent when the batch replays.
    """
    import concurrent.futures as cf

    batch_df = batch_df.persist()
    try:
        seen = [r["event_type"] for r in batch_df.select("event_type").distinct().collect()]
        present = [et for et in seen if et in EVENT_PAYLOADS]
        unmapped = [et for et in seen if et not in EVENT_PAYLOADS]
        if unmapped:
            raise ValueError(f"unmapped event_types in batch {batch_id}: {unmapped}")
        if not present:
            return
        token = notebookutils.credentials.getToken(kusto_uri)  # noqa: F821

        def _task(event_type):
            def _attempt(attempt):
                # A retry fetches a fresh token in case the last one expired.
                credential = token if attempt == 1 else (
                    notebookutils.credentials.getToken(kusto_uri)  # noqa: F821
                )
                _write_event_table(batch_df, event_type, batch_id, credential)

            try:
                call_with_retry(_attempt, label=f"batch {batch_id} {event_type}")
                return (event_type, None)
            except Exception as exc:  # noqa: BLE001 - collect, then fail the batch below
                return (event_type, exc)

        with cf.ThreadPoolExecutor(max_workers=min(_WRITE_PARALLELISM, len(present))) as pool:
            results = list(pool.map(_task, present))

        failed = [(et, err) for et, err in results if err is not None]
        succeeded = len(results) - len(failed)
        print(f"batch {batch_id}: wrote {succeeded}/{len(results)} event tables")
        if failed:
            detail = ", ".join(
                f"{event_type} ({type(error).__name__}: {error})"
                for event_type, error in failed
            )
            raise RuntimeError(
                f"batch {batch_id} failed required Eventhouse writes: {detail}"
            )
    finally:
        batch_df.unpersist()


def write_to_delta(batch_df, batch_id):
    """Append a micro-batch to the debug landing table, once per batch id."""
    (batch_df.write.format("delta").mode("append")
        .option("txnAppId", f"retail-stream-{STREAM_NAMESPACE}")
        .option("txnVersion", int(batch_id))
        .saveAsTable(delta_landing_table))


def process_batch(source_df, batch_id):
    """foreachBatch: generate the micro-batch's events, write them, then commit state.

    State is read as of the previous batch, so if this batch fails after its
    writes and Spark replays it, the replay regenerates the same events (the
    sinks ignore the duplicate writes) and replaces its own state versions.
    """
    rows = source_df.select(F.col("timestamp").alias("ts"), F.col("value").alias("v")).persist()
    try:
        batch = generate_stream_batch(STREAM_CONTEXT, rows, batch_id, STREAM_STATE)
        try:
            if sink == "eventhouse":
                write_to_eventhouse(batch.events, batch_id)
            else:
                write_to_delta(batch.events, batch_id)
            commit_stream_batch(STREAM_STATE, batch_id, batch)
        finally:
            batch.release()
    finally:
        rows.unpersist()


def _resolve_kusto_uri(database_name):
    """Resolve the KQL database Query URI from the current Fabric workspace.

    The Eventhouse Query URI is assigned by Fabric when the database is created,
    so it is not known ahead of time and is left blank in the parameters. Resolve
    it at runtime by matching the KQL database display name in this workspace and
    returning its ``queryServiceUri`` (the same property the deploy's KQL step
    uses). This lets the notebook run without anyone pasting the URI by hand.
    """
    import sempy.fabric as fabric

    client = fabric.FabricRestClient()
    workspace_id = fabric.get_notebook_workspace_id()
    resp = client.get(f"v1/workspaces/{workspace_id}/kqlDatabases")
    resp.raise_for_status()
    databases = resp.json().get("value", [])
    for item in databases:
        if item.get("displayName") == database_name:
            uri = (item.get("properties") or {}).get("queryServiceUri")
            if uri:
                return uri
    found = ", ".join(sorted(d.get("displayName", "?") for d in databases)) or "<none>"
    raise ValueError(
        f"Could not find a KQL database named {database_name!r} in this workspace "
        f"(found: {found}). Set kql_database to your Eventhouse database name, or "
        "paste its Query URI into kusto_uri."
    )

In [ ]:
# Write the stream to the chosen sink from its own checkpoint (resolved above).
TRIGGER_SECONDS = trigger_interval_seconds(int(run_seconds), int(trigger_seconds))

if sink == "eventhouse":
    # NOTE: the Kusto Spark connector (com.microsoft.kusto.spark) ships with the
    # Fabric Spark runtime. The notebook identity needs ingestor rights on the DB.
    if not kusto_uri:
        # Auto-resolve the Query URI from this workspace so the notebook works
        # without manual configuration. Set kusto_uri explicitly to override.
        kusto_uri = _resolve_kusto_uri(kql_database)
        print(f"Using Eventhouse '{kql_database}' Query URI: {kusto_uri}")
elif sink == "delta":
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {delta_landing_table.rsplit('.', 1)[0]}")
else:
    raise ValueError(f"unknown sink: {sink!r} (expected 'eventhouse' or 'delta')")


def start_query():
    """Start (or restart) the streaming query from the sink checkpoint."""
    rate = (spark.readStream.format("rate")
            .option("rowsPerSecond", RATE_ROWS_PER_SECOND).load())
    # Cap catch-up after a stop: keep only the newest `max_lag_seconds` of rows.
    rate = cap_backlog(rate, int(max_lag_seconds))
    return (rate.writeStream
            .option("checkpointLocation", CHECKPOINT_LOCATION)
            .trigger(processingTime=f"{TRIGGER_SECONDS} seconds")
            .foreachBatch(process_batch)
            .start())


print(f"streaming {RATE_ROWS_PER_SECOND} source rows/s to {sink} every {TRIGGER_SECONDS}s "
      f"(checkpoint {CHECKPOINT_LOCATION}, max lag {int(max_lag_seconds)}s)")
# Record the incarnation only now, after every startup check has passed.
INCARNATION_PLAN = persist_incarnation(mssparkutils.fs, INCARNATION_PLAN)
restarts = run_supervised(start_query, run_seconds=int(run_seconds),
                          max_restarts=int(max_restarts))
if int(run_seconds) > 0:
    print(f"stopped after {run_seconds}s")
else:
    print(f"stream stopped after {restarts} supervised restart(s)")